# Benchmark Calculation Overview

This notebook builds and verifies the official benchmark calculation for the Pismo freeway corridor using PeMS detector data.

The goal of the benchmark is simple:

> Simulate the corridor under the observed traffic condition, calculate the total delay and penalties, and use that result as the baseline for comparison with ADMM / ADMM-MPC control results.

This benchmark is a **state-based CTM benchmark**, meaning the delay is calculated from the simulated number of vehicles inside each freeway cell and ramp queue, not only from detector speeds.

---

# 1. Data Used

The notebook uses two PeMS files:

| File                                   | Purpose                                                                                  |
| -------------------------------------- | ---------------------------------------------------------------------------------------- |
| `d05_text_meta_2026_04_28.txt`         | Gives station metadata: station ID, type, postmile, lanes, station name, etc.            |
| `d05_text_station_5min_2026_05_27.txt` | Gives 5-minute traffic measurements: flow, occupancy, speed, station ID, timestamp, etc. |

The benchmark uses one single observed day:

$$
\text{Benchmark date} = 2026\text{-}05\text{-}27
$$

The official benchmark window is:

$$
16{:}00 \text{ to } 18{:}00
$$

This is a 2-hour window.

---

# 2. CTM Time Setup

The model uses a 15-second CTM step.

$$
\Delta t = 0.25 \text{ minutes}
$$

Since 15 seconds is 0.25 minutes:

$$
\text{steps per hour} = \frac{60}{0.25} = 240
$$

The benchmark is 2 hours long:

$$
\text{num_steps} = 2 \times 240 = 480
$$

So the benchmark simulation runs for:

$$
480 \text{ CTM steps}
$$

Each PeMS row is 5 minutes. Since 5 minutes contains twenty 15-second steps:

$$
\text{steps per 5 minutes} = \frac{5}{0.25} = 20
$$

Each 5-minute flow count is divided evenly across 20 CTM steps:

$$
q_{15s} = \frac{q_{5min}}{20}
$$

---

# 3. Corridor Geometry

The model uses:

* 10 mainline detectors
* 4 on-ramp detectors
* 1 off-ramp detector

The 10 mainline detectors create 9 freeway cells:

$$
10 \text{ mainline stations} \Rightarrow 9 \text{ CTM cells}
$$

So the official model is:

$$
9 \text{ cells}, 4 \text{ on-ramps}, 1 \text{ detected off-ramp}
$$

The notebook checks that:

* all selected station IDs exist in the metadata file
* mainline stations have type `ML`
* on-ramp stations are real on-ramp detectors
* off-ramp stations are real off-ramp detectors
* every segment has positive length

Each cell length is calculated from PeMS absolute postmile:

$$
L_i = PM_{downstream} - PM_{upstream}
$$

This is justified because each CTM cell is built directly between two consecutive mainline detectors.

---

# 4. Selected Stations and Ramp Mapping

The mainline detectors define the 9 cells from 4TH ST to SAN LUIS BAY DR.

The on-ramps are mapped into cells as follows:

| On-ramp    | CTM cell                           |
| ---------- | ---------------------------------- |
| `u_4th`    | Cell 2                             |
| `u_price`  | Cell 2                             |
| `u_mattie` | Cell 6                             |
| `u_avila`  | Cell 9, handled as a special merge |

The detected off-ramp is:

| Off-ramp       | CTM cell |
| -------------- | -------- |
| Bello off-ramp | Cell 5   |

The Bello off-ramp is placed in Cell 5 because its detector location belongs geometrically between the Cell 5 mainline detector pair. Older code/comments that place it in Cell 4 are outdated.

---

# 5. Special Avila Merge Rule

The Avila Beach on-ramp is not treated like a normal on-ramp.

The normal on-ramps enter through `u_in_series`:

* `u_4th`
* `u_price`
* `u_mattie`

But Avila is handled separately as a merge into Cell 9.

This means:

$$
u_{\text{avila}}(t) \notin \text{normal } u_in_series
$$

Instead, Avila has its own merge acceptance calculation at the Cell 8 to Cell 9 interface.

This is justified because Avila enters near the downstream bottleneck area, so treating it as a normal lateral inflow would over-simplify the merge competition between:

1. mainline traffic coming from Cell 8
2. Avila ramp traffic entering Cell 9

The model gives Avila a merge priority parameter:

$$
\text{MERGE_PRIORITY} = 0.3
$$

The Avila accepted flow is limited by both Avila demand and Cell 9 receiving space.

In simple words:

> Avila can only enter Cell 9 if Cell 9 has room, and it must share that room with mainline traffic coming from Cell 8.

This is a reasonable modeling rule because the ramp is near a bottleneck. However, it is still a modeling assumption, not a directly observed PeMS value.

---

# 6. Free-Flow Speed Calculation

Free-flow speed is estimated from the low-traffic period:

$$
01{:}00 \text{ to } 05{:}00
$$

For each mainline station, the notebook calculates:

$$
v^{ff}_{station} = \text{median speed during } 01{:}00\text{--}05{:}00
$$

The median is used instead of the mean because it is more stable against unusual detector readings.

For each cell, the upstream and downstream free-flow speeds are combined using the harmonic mean:

$$
v^{ff}*i =
\frac{2v^{ff}*{up}v^{ff}*{down}}
{v^{ff}*{up}+v^{ff}_{down}}
$$

The harmonic mean is used because speed is a rate. It is better for travel-time-related calculations than a simple average.

Then free-flow travel time for each cell is:

$$
TT^{ff}_i =
\frac{L_i}{v^{ff}_i} \times 60
$$

where:

* $L_i$ = cell length in miles
* $v^{ff}_i$ = free-flow speed in mph
* $TT^{ff}_i$ = free-flow travel time in minutes

---

# 7. Initial Mainline State

The initial mainline state is the number of vehicles inside each CTM cell at the beginning of the benchmark window.

The benchmark starts at:

$$
16{:}00
$$

For each mainline detector, density is estimated from flow and speed:

$$
k = \frac{q}{v}
$$

where:

* $k$ = density in vehicles per mile
* $q$ = flow in vehicles per hour
* $v$ = speed in miles per hour

The notebook then interpolates density from detector locations to the midpoint of each CTM cell.

The initial number of vehicles in each cell is:

$$
x_i(0) = k_i \times L_i
$$

where:

* $x_i(0)$ = initial vehicles in Cell $i$
* $k_i$ = interpolated density at the cell midpoint
* $L_i$ = cell length in miles

This is justified because CTM needs a vehicle count in each cell, but PeMS gives detector measurements at points. Interpolating density to each cell midpoint is a reasonable way to convert detector data into cell states.

---

# 8. Doorway Capacity

Each cell has a doorway capacity, meaning the maximum number of vehicles that can enter or leave a cell during one 15-second step.

For most cells, the base capacity is calculated from free-flow speed and lane count.

If free-flow speed is at most 70 mph:

$$
C_{\text{lane}} = 1700 + 10v^{ff}
$$

If free-flow speed is greater than 70 mph:

$$
C_{\text{lane}} = 2400
$$

Then total cell capacity in vehicles per hour is:

$$
C_i^{vph} = C_{\text{lane}} \times \text{lanes}_i
$$

The 15-second capacity is:

$$
C_i^{15s} = \frac{C_i^{vph}}{240}
$$

because there are 240 fifteen-second steps in one hour.

---

# 9. Bottleneck Capacity Rule

Cell 9 is treated as the downstream bottleneck area.

The notebook uses Spyglass, Avila Beach, and San Luis Bay detector data to identify active bottleneck periods.

The bottleneck is considered active when:

$$
\text{Spyglass speed} < 45 \text{ mph}
$$

and

$$
\text{San Luis Bay speed} \ge 50 \text{ mph}
$$

This means upstream traffic is congested while downstream traffic is still discharging.

For the official run, Cell 9 inflow and outflow capacity are set to:

$$
2400 \text{ vehicles/hour}
$$

Converted to 15-second CTM steps:

$$
C_9 = \frac{2400}{240} = 10 \text{ vehicles per 15 seconds}
$$

This is a modeling assumption, but it is data-motivated because it is based on the observed downstream bottleneck discharge behavior.

---

# 10. Physical Storage Capacity

Each CTM cell has a physical storage capacity.

This is the maximum number of vehicles that can physically fit inside the cell.

The formula is:

$$
N_i^{max} = k_{jam} \times L_i \times n_i
$$

where:

* $N_i^{max}$ = physical capacity of Cell $i$
* $k_{jam}$ = jam density
* $L_i$ = cell length in miles
* $n_i$ = number of lanes

The notebook uses:

$$
k_{jam} = 190 \text{ vehicles/mile/lane}
$$

The safe threshold is set to 70% of physical capacity:

$$
N_i^{safe} = 0.7N_i^{max}
$$

The physical capacity is a hard storage reference. The safe threshold is a soft warning threshold.

---

# 11. Ramp Arrival and Release Assumption

The observed PeMS on-ramp flow is used as the observed ramp release.

For each on-ramp:

$$
u^{obs}_r(t) = \text{observed PeMS ramp flow}
$$

The benchmark assumes that true ramp arrival demand is larger than observed release:

$$
a_r(t) = 1.2u^{obs}_r(t)
$$

So:

$$
\text{arrival_multiplier} = 1.2
$$

In simple terms:

> If the detector saw 10 vehicles enter the freeway, the model assumes 12 vehicles wanted to enter. The extra 2 vehicles may become queue.

This is important because observed ramp flow only tells us how many vehicles entered the freeway. It does not directly tell us how many vehicles were waiting on the ramp.

This assumption is useful for creating ramp queues in the benchmark, but it must be clearly stated because it is not directly observed from PeMS.

---

# 12. Ramp Queue Capacity and Spillback

Each ramp has a maximum physical queue capacity.

The formula is:

$$
R^{max}*r =
\frac{L_r \times n_r}{l*{veh}}
$$

where:

* $R^{max}_r$ = maximum ramp queue in vehicles
* $L_r$ = ramp length in feet
* $n_r$ = number of ramp lanes
* $l_{veh}$ = assumed vehicle length

The notebook uses:

$$
l_{veh} = 25 \text{ ft/vehicle}
$$

The ramp queue update uses:

$$
\text{available}_r(t)
=====================

R_r(t) + B_r(t) + a_r(t)
$$

where:

* $R_r(t)$ = physical ramp queue
* $B_r(t)$ = external spillback queue
* $a_r(t)$ = ramp arrival demand

Actual ramp release cannot exceed available demand:

$$
u^{actual}_r(t)
===============

\min(u^{cmd}_r(t), \text{available}_r(t))
$$

Then waiting vehicles after release are:

$$
W_r(t+1)
========

\max(0, \text{available}_r(t) - u^{actual}_r(t))
$$

The physical ramp queue is capped:

$$
R_r(t+1)
========

\min(W_r(t+1), R^{max}_r)
$$

Anything above physical ramp storage becomes external spillback:

$$
B_r(t+1)
========

\max(0, W_r(t+1) - R^{max}_r)
$$

So the model separates ramp waiting into two parts:

| Queue | Meaning                                                           |
| ----- | ----------------------------------------------------------------- |
| $R_r$ | vehicles physically stored on the ramp                            |
| $B_r$ | vehicles that exceed ramp storage and spill back outside the ramp |

This is justified because real ramps cannot store infinite vehicles.

---

# 13. CTM Sending and Receiving

For each cell, the CTM calculates how many vehicles want to leave and how many vehicles the downstream cell can receive.

The sending flow is:

$$
S_i(t)
======

\min(m_i x_i(t), C_i^{out})
$$

where:

* $S_i(t)$ = sending flow from Cell $i$
* $m_i$ = movement factor
* $x_i(t)$ = vehicles in Cell $i$
* $C_i^{out}$ = outflow capacity

The movement factor is:

$$
m_i = \min\left(1, \frac{\Delta t}{TT^{ff}_i}\right)
$$

The receiving flow is:

$$
R_i(t)
======

\min(C_i^{in}, w_i(N_i^{max} - x_i(t)))
$$

where:

* $R_i(t)$ = receiving space in Cell $i$
* $C_i^{in}$ = inflow capacity
* $w_i$ = wave-speed ratio
* $N_i^{max}$ = physical storage capacity
* $x_i(t)$ = current vehicles in the cell

The model uses a backward wave speed of:

$$
15 \text{ mph}
$$

---

# 14. Mainline State Update

For each cell, the next state is calculated by vehicle conservation:

$$
x_i(t+1)
========

x_i(t)

* q^{in}_i(t)
* u_i(t)

- q^{out}_i(t)
- f_i(t)
  $$

where:

* $x_i(t)$ = current vehicles in Cell $i$
* $q^{in}_i(t)$ = mainline inflow into Cell $i$
* $u_i(t)$ = on-ramp / undetected inflow into Cell $i$
* $q^{out}_i(t)$ = mainline outflow from Cell $i$
* $f_i(t)$ = off-ramp or exit flow leaving Cell $i$

For Cell 9, Avila accepted merge flow is added separately:

$$
x_9(t+1)
========

x_9(t)

* q^{in}_9(t)
* u_9(t)
* u^{accepted}_{avila}(t)

- q^{out}_9(t)
- f_9(t)
  $$

The model also has an upstream boundary queue. If Cell 1 cannot accept all boundary inflow, the leftover demand waits upstream.

---

# 15. Mainline Delay Calculation

Mainline delay is calculated from vehicle inventory inside the CTM cells.

First, total vehicle-time in a cell during one step is approximated using the trapezoid rule:

$$
TTT_i(t)
========

\frac{x_i(t)+x_i(t+1)}{2}
\Delta t
$$

This gives vehicle-minutes spent inside the cell.

The free-flow vehicle-time is:

$$
FF_i(t)
=======

(q^{out}_i(t) + f^{actual}_i(t))TT^{ff}_i
$$

The raw delay is:

$$
D^{raw}_{M,i}(t)
================

TTT_i(t) - FF_i(t)
$$

Delay cannot be negative, so the official benchmark clips it at zero:

$$
D_{M,i}(t)
==========

\max(D^{raw}_{M,i}(t), 0)
$$

Total mainline delay for one step is:

$$
D_M(t)
======

\sum_i D_{M,i}(t)
$$

The benchmark also adds upstream boundary delay if boundary demand cannot enter Cell 1.

---

# 16. Local Ramp Delay Calculation

Ramp delay is calculated from average ramp waiting vehicles.

For each ramp:

$$
D_{R,r}(t)
==========

\frac{
(R_r(t)+B_r(t)) + (R_r(t+1)+B_r(t+1))
}{2}
\Delta t
$$

where:

* $R_r$ = physical ramp queue
* $B_r$ = external spillback queue
* $\Delta t$ = 0.25 minutes

Total ramp delay is:

$$
D_R(t)
======

\sum_r D_{R,r}(t)
$$

This means both physical ramp queue and spillback queue are counted as waiting delay.

---

# 17. Fairness Penalty

The fairness penalty measures how uneven the ramp queue stress is across ramps.

First, each ramp gets a stress value:

$$
\phi_r(t)
=========

\min\left(
\max\left(\frac{R_r(t)}{R^{max}_r}, 0\right),
1
\right)
$$

So:

* 0 means the ramp is empty
* 1 means the ramp is full
* values above 1 are capped at 1

The fairness penalty is:

$$
F(t)
====

\gamma
\sum_{r<s}
(\phi_r(t)-\phi_s(t))^2
$$

The notebook uses:

$$
\gamma = 1
$$

This penalty increases when one ramp is much more congested than another ramp.

This is justified if the goal is not only to reduce total delay, but also to avoid unfairly overloading one ramp while keeping others much emptier.

---

# 18. Capacity Penalty

The benchmark calculates four capacity-related penalties.

## Doorway penalty

This penalizes demand trying to enter a cell above its inflow capacity:

$$
P_{door}(t)
===========

\lambda_1
\sum_i
\max(q^{in}_i(t)+u_i(t)-C_i^{in},0)^2
$$

## Safe-threshold penalty

This penalizes states above the safe threshold:

$$
P_{safe}(t)
===========

\lambda_2
\sum_i
\max(x_i(t+1)-N_i^{safe},0)^2
$$

## Physical-capacity penalty

This penalizes states above physical storage capacity:

$$
P_{physical}(t)
===============

\lambda_3
\sum_i
\max(x_i(t+1)-N_i^{max},0)^2
$$

## Spillback penalty

This penalizes external spillback queues:

$$
P_{spillback}(t)
================

\lambda_4
\sum_r
\max(B_r(t+1),0)^2
$$

The weights are:

$$
\lambda_1 = 1.0
$$

$$
\lambda_2 = 0.5
$$

$$
\lambda_3 = 1.0
$$

$$
\lambda_4 = 0.5
$$

The total capacity penalty is:

$$
P_{capacity}(t)
===============

P_{door}(t)
+
P_{safe}(t)
+
P_{physical}(t)
+
P_{spillback}(t)
$$

---

# 19. Total Benchmark Objective

For each CTM step, the benchmark objective is:

$$
J(t)
====

D_M(t)
+
D_R(t)
+
F(t)
+
P_{capacity}(t)
$$

where:

* $D_M(t)$ = mainline delay
* $D_R(t)$ = local ramp delay
* $F(t)$ = fairness penalty
* $P_{capacity}(t)$ = capacity penalty

The full benchmark objective over all 480 steps is:

$$
J_{total}
=========

\sum_{t=1}^{480}
J(t)
$$

Expanded:

$$
J_{total}
=========

\sum_{t=1}^{480}
[
D_M(t)
+
D_R(t)
+
F(t)
+
P_{capacity}(t)
]
$$

This final value is the official benchmark baseline.

---

# 20. Verification Checks

The notebook performs several checks to make sure the benchmark calculation is internally correct.

It checks that:

* all selected detector IDs exist in metadata
* all input series have exactly 480 steps
* the benchmark window has full station coverage
* cell lengths are positive
* capacities are positive
* CTM first-step mass balance holds
* ramp mass is conserved
* manual mainline delay equals stored mainline delay
* all history arrays have length 480

The most important conservation check is ramp mass:

$$
R(0) + B(0) + \text{arrivals}
-----------------------------

## \text{actual releases}

## R(T)

# B(T)

0
$$

In the verified run, the ramp mass residual is essentially zero, so the ramp queue accounting is correct.

The manual mainline delay recomputation also exactly matches the stored mainline delay, so the mainline delay calculation is correct.

---

# 21. Verified Output From Current Notebook Run

Using the current uploaded data and notebook logic, the official benchmark totals are:

| Metric                    |         Value |
| ------------------------- | ------------: |
| Mainline delay            |    25,799.028 |
| Local ramp delay          |    44,657.000 |
| Fairness penalty          |       143.624 |
| Doorway penalty           |         0.104 |
| Safe-threshold penalty    |         0.000 |
| Physical-capacity penalty |         0.000 |
| Spillback penalty         | 7,849,384.087 |
| Total capacity penalty    | 7,849,384.191 |
| Raw total objective       | 7,919,983.843 |

Ramp service / conservation results:

| Metric                         |           Value |
| ------------------------------ | --------------: |
| Total ramp arrivals            |       4,387.200 |
| Total actual release           |       3,656.000 |
| Final physical ramp queue      |         146.257 |
| Final external spillback queue |         584.943 |
| Served fraction                |        0.833333 |
| Ramp mass residual             | approximately 0 |

The large spillback penalty means the benchmark produces significant ramp overflow under the assumed arrival multiplier of 1.2. This is not a coding error; it follows from the ramp demand assumption and physical ramp storage limits.

---

# 22. Important Notes About Justification

The calculation is internally consistent and mathematically correct under the notebook assumptions.

The strongest parts of the benchmark are:

* detector-based geometry
* PeMS-based free-flow speed
* PeMS-based initial state
* PeMS-based observed ramp releases
* mass-conserving CTM update
* explicit ramp queue and spillback accounting
* manual verification of mainline delay

The assumptions that must be clearly stated are:

1. Ramp arrival demand is assumed to be 1.2 times observed ramp release.
2. Avila is handled as a special Cell 9 merge instead of a normal on-ramp.
3. Undetected entries and exits are estimated from detector-balance residuals.
4. Cell 9 bottleneck capacity is set to 2400 vehicles/hour.
5. Spillback is penalized heavily because external queue overflow is treated as severe.

These assumptions do not make the benchmark wrong. They just mean the benchmark is a controlled, reproducible modeling baseline rather than a perfect reconstruction of every real vehicle movement.


In [48]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import datetime as dt

import os
%matplotlib inline


In [49]:
# 1. LOAD METADATA AND SELECT OFFICIAL PISMO CORRIDOR STATIONS

import pandas as pd
from pathlib import Path


# 1. OFFICIAL SELECTED DETECTOR IDS

# 10 mainline + 4 on-ramp + 1 off-ramp = 15 stations
mainline_ids = [
    501015153,  # 4TH ST ML
    501016013,  # 4TH ST ON AREA ML
    501016023,  # PRICE ST ML
    501016031,  # HINDS AVE ML
    501016043,  # BELLO ST ML
    501016053,  # SHELL BEACH RD ML
    501016062,  # MATTIE RD ML
    501016071,  # SPYGLASS DR ML
    501016082,  # AVILA BEACH DR ML
    501016091,  # SAN LUIS BAY DR ML
]

onramp_ids = [
    501016014,  # 4TH ST ON
    501016024,  # PRICE ST ON
    501016063,  # MATTIE RD ON
    501016083,  # AVILA BEACH ON
]

offramp_ids = [
    501016044,  # BELLO ST OFF
]

ids_to_keep = (
    mainline_ids
    + onramp_ids
    + offramp_ids
)

ids_to_keep_str = {
    str(station_id).strip()
    for station_id in ids_to_keep
}

assert len(mainline_ids) == 10
assert len(onramp_ids) == 4
assert len(offramp_ids) == 1
assert len(ids_to_keep) == 15
assert len(ids_to_keep_str) == 15


# 2. OFFICIAL CTM MODEL IDS
cell_ids = [
    f"Cell {i}"
    for i in range(1, 10)
]

segment_ids = [
    f"S{i}"
    for i in range(1, 10)
]

ramp_ids = [
    "u_4th",
    "u_price",
    "u_mattie",
    "u_avila",
]

ramp_name_map = {
    "u_4th": "4TH ST ON",
    "u_price": "PRICE ST ON",
    "u_mattie": "MATTIE RD ON",
    "u_avila": "AVILA BEACH ON",
}



# 3. OFFICIAL CORRECTED RAMP-TO-CELL GEOMETRY.
generic_ramp_cell_map = {
    "u_4th": "Cell 2",
    "u_price": "Cell 2",
    "u_mattie": "Cell 6",
}

merge_ramp_id = "u_avila"
merge_ramp_cell = "Cell 9"

# Full ramp map is still used for metadata, export, queue accounting, and labels.
ramp_cell_map = {
    "u_4th": "Cell 2",
    "u_price": "Cell 2",
    "u_mattie": "Cell 6",
    "u_avila": "Cell 9",
}

ramp_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

external_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

assert len(cell_ids) == 9
assert len(segment_ids) == 9
assert len(ramp_ids) == 4
assert set(ramp_cell_map.keys()) == set(ramp_ids)
assert set(ramp_name_map.keys()) == set(ramp_ids)
assert set(generic_ramp_cell_map.keys()) == {
    "u_4th",
    "u_price",
    "u_mattie",
}
assert merge_ramp_id == "u_avila"
assert ramp_cell_map[merge_ramp_id] == merge_ramp_cell
assert all(cell in cell_ids for cell in ramp_cell_map.values())



# 4. LOAD PEMS METADATA
metadata_file_path = (
    Path("District_5_5min_station_data")
    / "d05_text_meta_2026_04_28.txt"
)

if not metadata_file_path.exists():
    raise FileNotFoundError(
        f"Metadata file not found: {metadata_file_path}"
    )

print("Loading metadata file:", metadata_file_path)

metadata_raw = pd.read_csv(
    metadata_file_path,
    sep=None,
    engine="python",
    header=None,
    dtype=str,
    skip_blank_lines=True
)

metadata_raw = metadata_raw.apply(
    lambda col: col.astype(str).str.strip()
)

if metadata_raw.shape[1] < 14:
    raise ValueError(
        f"Metadata file has {metadata_raw.shape[1]} columns, expected at least 14."
    )

selected_metadata_clean = pd.DataFrame({
    "station_id": metadata_raw[0],
    "station_name": metadata_raw[13],
    "station_type": metadata_raw[11],
    "freeway": metadata_raw[1],
    "direction": metadata_raw[2],
    "absolute_postmile": metadata_raw[7],
    "length": metadata_raw[10],
    "lanes": metadata_raw[12],
    "latitude": metadata_raw[8],
    "longitude": metadata_raw[9],
})

selected_metadata_clean["station_id"] = (
    selected_metadata_clean["station_id"]
    .astype(str)
    .str.strip()
)

selected_metadata_clean = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(ids_to_keep_str)
].copy()

numeric_cols = [
    "station_id",
    "freeway",
    "absolute_postmile",
    "length",
    "lanes",
    "latitude",
    "longitude",
]

for col in numeric_cols:
    selected_metadata_clean[col] = pd.to_numeric(
        selected_metadata_clean[col],
        errors="coerce"
    )

selected_metadata_clean["station_type"] = (
    selected_metadata_clean["station_type"]
    .astype(str)
    .str.strip()
)

selected_metadata_clean["station_name"] = (
    selected_metadata_clean["station_name"]
    .astype(str)
    .str.strip()
)

selected_metadata_clean = selected_metadata_clean.sort_values(
    [
        "absolute_postmile",
        "station_type",
    ]
).reset_index(drop=True)


# 5. METADATA SAFETY CHECKS
found_station_ids = set(
    selected_metadata_clean["station_id"]
    .dropna()
    .astype(int)
)

expected_station_ids = set(ids_to_keep)

missing_metadata_ids = sorted(
    expected_station_ids - found_station_ids
)

extra_metadata_ids = sorted(
    found_station_ids - expected_station_ids
)

if len(missing_metadata_ids) > 0:
    print("Missing metadata IDs:", missing_metadata_ids)

if len(extra_metadata_ids) > 0:
    print("Extra metadata IDs:", extra_metadata_ids)

if len(selected_metadata_clean) != len(ids_to_keep):
    display(selected_metadata_clean)

    raise ValueError(
        f"Expected {len(ids_to_keep)} selected metadata rows, "
        f"but found {len(selected_metadata_clean)}."
    )

required_metadata_cols = [
    "station_id",
    "station_name",
    "station_type",
    "absolute_postmile",
    "lanes",
]

for col in required_metadata_cols:
    if selected_metadata_clean[col].isna().any():
        bad_rows = selected_metadata_clean[
            selected_metadata_clean[col].isna()
        ].copy()

        print(f"Bad rows for missing column {col}:")
        display(bad_rows)

        raise ValueError(
            f"Missing values found in metadata column: {col}"
        )


# 6. BUILD METADATA DICTIONARIES
station_name_by_id = {
    int(row["station_id"]): str(row["station_name"])
    for _, row in selected_metadata_clean.iterrows()
}

station_type_by_id = {
    int(row["station_id"]): str(row["station_type"])
    for _, row in selected_metadata_clean.iterrows()
}

station_pm = {
    int(row["station_id"]): float(row["absolute_postmile"])
    for _, row in selected_metadata_clean.iterrows()
}

station_lane_count = {
    int(row["station_id"]): int(float(row["lanes"]))
    for _, row in selected_metadata_clean.iterrows()
}

for station_id in ids_to_keep:
    assert station_id in station_name_by_id
    assert station_id in station_type_by_id
    assert station_id in station_pm
    assert station_id in station_lane_count


# 7. STATION TYPE / NAME CHECKS

def normalize_detector_name(station_id):
    return (
        station_name_by_id[station_id]
        .upper()
        .replace(" ", "")
        .replace("-", "")
        .replace("_", "")
    )


def detector_name_indicates_onramp(station_id):
    name = normalize_detector_name(station_id)

    return (
        "ON" in name
        or "ONSB" in name
        or "ONNB" in name
    )


def detector_name_indicates_offramp(station_id):
    name = normalize_detector_name(station_id)

    return (
        "OFF" in name
        or "OFSB" in name
        or "OFNB" in name
    )


for station_id in mainline_ids:
    station_type_upper = station_type_by_id[station_id].upper()

    if station_type_upper != "ML":
        raise ValueError(
            f"Expected mainline station {station_id} to have station_type ML, "
            f"but got station_type={station_type_by_id[station_id]} "
            f"and station_name={station_name_by_id[station_id]}"
        )

for station_id in onramp_ids:
    if not detector_name_indicates_onramp(station_id):
        raise ValueError(
            f"Expected on-ramp station {station_id} name to indicate ON, "
            f"but got station_name={station_name_by_id[station_id]} "
            f"and station_type={station_type_by_id[station_id]}"
        )

for station_id in offramp_ids:
    if not detector_name_indicates_offramp(station_id):
        raise ValueError(
            f"Expected off-ramp station {station_id} name to indicate OFF/OFSB/OFNB, "
            f"but got station_name={station_name_by_id[station_id]} "
            f"and station_type={station_type_by_id[station_id]}"
        )


# 8. OFFICIAL 9-CELL MAINLINE SEGMENT GEOMETRY
mainline_segments = []

for i in range(len(mainline_ids) - 1):
    from_id = mainline_ids[i]
    to_id = mainline_ids[i + 1]

    segment = {
        "segment": segment_ids[i],
        "cell": cell_ids[i],
        "from_id": from_id,
        "to_id": to_id,
        "from_station": station_name_by_id[from_id],
        "to_station": station_name_by_id[to_id],
        "from_postmile": station_pm[from_id],
        "to_postmile": station_pm[to_id],
        "length_miles": station_pm[to_id] - station_pm[from_id],
    }

    if segment["length_miles"] <= 0:
        raise ValueError(
            f"{segment['cell']} has nonpositive length: "
            f"{segment['length_miles']}"
        )

    mainline_segments.append(segment)

assert len(mainline_segments) == 9

mainline_segments_df = pd.DataFrame(mainline_segments)



# 9. DISPLAY OFFICIAL METADATA
print("Official model IDs loaded.")
print("Cells:", cell_ids)
print("Segments:", segment_ids)
print("Ramps:", ramp_ids)
print("Ramp names:", ramp_name_map)

print("Official ramp mapping")
print("generic_ramp_cell_map:", generic_ramp_cell_map)
print("merge_ramp_id:", merge_ramp_id)
print("merge_ramp_cell:", merge_ramp_cell)
print("ramp_cell_map:", ramp_cell_map)

print("Selected Station Metadata")
print("Number of selected stations:", len(selected_metadata_clean))
display(selected_metadata_clean)

print("Official mainline segments")
display(mainline_segments_df)

print("station_pm:")
print(station_pm)

print("station_lane_count:")
print(station_lane_count)

print("PASS: Cell 1 metadata / official geometry loaded cleanly.")

Loading metadata file: District_5_5min_station_data\d05_text_meta_2026_04_28.txt
Official model IDs loaded.
Cells: ['Cell 1', 'Cell 2', 'Cell 3', 'Cell 4', 'Cell 5', 'Cell 6', 'Cell 7', 'Cell 8', 'Cell 9']
Segments: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Ramps: ['u_4th', 'u_price', 'u_mattie', 'u_avila']
Ramp names: {'u_4th': '4TH ST ON', 'u_price': 'PRICE ST ON', 'u_mattie': 'MATTIE RD ON', 'u_avila': 'AVILA BEACH ON'}
Official ramp mapping
generic_ramp_cell_map: {'u_4th': 'Cell 2', 'u_price': 'Cell 2', 'u_mattie': 'Cell 6'}
merge_ramp_id: u_avila
merge_ramp_cell: Cell 9
ramp_cell_map: {'u_4th': 'Cell 2', 'u_price': 'Cell 2', 'u_mattie': 'Cell 6', 'u_avila': 'Cell 9'}
Selected Station Metadata
Number of selected stations: 15


,station_id,station_name,station_type,freeway,direction,absolute_postmile,length,lanes,latitude,longitude
0,501015153,4TH ST 101 NB EXIT VDS MLSB SB,ML,101,S,188.738,0.506,2,35.133413,-120.616280
1,501016013,4TH ST 101 NB ON RAMP VDS MLSB S,ML,101,S,189.253,0.481,2,35.136794,-120.624363
2,501016014,4TH ST 101 NB ON RAMP VDS ONSB S,OR,101,S,189.254,NaN,1,35.136798,-120.624380
3,501016023,PRICE ST EXIT SIGN 101 NB VDS ML,ML,101,S,189.702,0.475,2,35.138331,-120.632049
4,501016024,PRICE ST EXIT SIGN 101 NB VDS ON,OR,101,S,189.703,NaN,1,35.138334,-120.632066
5,501016031,HINDS AVE 101 SB VDS MLSB SB,ML,101,S,190.204,0.506,2,35.141950,-120.639431
6,501016043,BELLO ST 101 NB VDS MLSB SB,ML,101,S,190.714,0.623,2,35.147275,-120.645623
7,501016044,BELLO ST 101 NB VDS OFSB SB,FR,101,S,190.715,NaN,1,35.147282,-120.645638
8,501016053,SHELL BEACH RD 101 NB VDS MLSB S,ML,101,S,191.451,0.540,2,35.151759,-120.657382
9,501016062,MATTIE RD 101 NB VDS MLSB SB,ML,101,S,191.796,0.935,2,35.154187,-120.662680


Official mainline segments


,segment,cell,from_id,to_id,from_station,to_station,from_postmile,to_postmile,length_miles
0,S1,Cell 1,501015153,501016013,4TH ST 101 NB EXIT VDS MLSB SB,4TH ST 101 NB ON RAMP VDS MLSB S,188.738,189.253,0.515
1,S2,Cell 2,501016013,501016023,4TH ST 101 NB ON RAMP VDS MLSB S,PRICE ST EXIT SIGN 101 NB VDS ML,189.253,189.702,0.449
2,S3,Cell 3,501016023,501016031,PRICE ST EXIT SIGN 101 NB VDS ML,HINDS AVE 101 SB VDS MLSB SB,189.702,190.204,0.502
3,S4,Cell 4,501016031,501016043,HINDS AVE 101 SB VDS MLSB SB,BELLO ST 101 NB VDS MLSB SB,190.204,190.714,0.510
4,S5,Cell 5,501016043,501016053,BELLO ST 101 NB VDS MLSB SB,SHELL BEACH RD 101 NB VDS MLSB S,190.714,191.451,0.737
5,S6,Cell 6,501016053,501016062,SHELL BEACH RD 101 NB VDS MLSB S,MATTIE RD 101 NB VDS MLSB SB,191.451,191.796,0.345
6,S7,Cell 7,501016062,501016071,MATTIE RD 101 NB VDS MLSB SB,SPYGLASS DR 101 SB VDS MLSB SB,191.796,193.322,1.526
7,S8,Cell 8,501016071,501016082,SPYGLASS DR 101 SB VDS MLSB SB,AVILA BEACH DR 101 NB VDS MLSB S,193.322,194.463,1.141
8,S9,Cell 9,501016082,501016091,AVILA BEACH DR 101 NB VDS MLSB S,SAN LUIS BAY DR 101 SB VDS MLSB,194.463,195.520,1.057


station_pm:
{501015153: 188.738, 501016013: 189.253, 501016014: 189.254, 501016023: 189.702, 501016024: 189.703, 501016031: 190.204, 501016043: 190.714, 501016044: 190.715, 501016053: 191.451, 501016062: 191.796, 501016063: 191.797, 501016071: 193.322, 501016082: 194.463, 501016083: 194.464, 501016091: 195.52}
station_lane_count:
{501015153: 2, 501016013: 2, 501016014: 1, 501016023: 2, 501016024: 1, 501016031: 2, 501016043: 2, 501016044: 1, 501016053: 2, 501016062: 2, 501016063: 1, 501016071: 2, 501016082: 3, 501016083: 1, 501016091: 2}
PASS: Cell 1 metadata / official geometry loaded cleanly.


In [50]:
# 2. LOAD SINGLE-DAY PEMS 5-MINUTE STATION DATA
data_folder = Path("District_5_5min_station_data")
# Official single-day benchmark selection
benchmark_date = pd.to_datetime("2026-05-27").date()
benchmark_source_file = "d05_text_station_5min_2026_05_27.txt"

station_files = [
    benchmark_source_file
]

all_days = []

for file_name in station_files:
    file_path = data_folder / file_name

    if not file_path.exists():
        print("MISSING:", file_path)
        continue

    print("Loading:", file_path)

    one_day = pd.read_csv(
        file_path,
        header=None
    )

    one_day["source_file"] = file_name

    all_days.append(one_day)

if len(all_days) == 0:
    raise FileNotFoundError(
        "No PeMS station files were loaded."
    )

station_data = pd.concat(
    all_days,
    ignore_index=True
)

station_data[0] = pd.to_datetime(
    station_data[0]
)

station_data[1] = pd.to_numeric(
    station_data[1],
    errors="coerce"
).astype("Int64")

filtered_data = station_data[
    station_data[1].isin(ids_to_keep)
].copy()

selected_data = filtered_data[
    [
        0,
        1,
        5,
        9,
        10,
        11,
        "source_file",
    ]
].copy()

selected_data.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow_5min",
    "avg_occupancy",
    "avg_speed",
    "source_file",
]

selected_data["station_id"] = pd.to_numeric(
    selected_data["station_id"],
    errors="coerce"
).astype("Int64")

for col in [
    "total_flow_5min",
    "avg_occupancy",
    "avg_speed",
]:
    selected_data[col] = pd.to_numeric(
        selected_data[col],
        errors="coerce"
    )

selected_data["flow_vph"] = (
    selected_data["total_flow_5min"]
    * 12.0
)

selected_data["date"] = selected_data["timestamp"].dt.date
selected_data["time_of_day"] = selected_data["timestamp"].dt.time
selected_data["hour"] = selected_data["timestamp"].dt.hour

loaded_dates = sorted(
    selected_data["date"].dropna().unique()
)

if loaded_dates != [benchmark_date]:
    raise ValueError(
        f"Expected only benchmark date {benchmark_date}, "
        f"but loaded dates are {loaded_dates}."
    )

print("Selected single-day corridor data")
print("Benchmark source file:", benchmark_source_file)
print("Benchmark date:", benchmark_date)
print("Rows:", len(selected_data))
print("Unique dates:", selected_data["date"].nunique())
print("Unique timestamps:", selected_data["timestamp"].nunique())
print("Unique stations:", selected_data["station_id"].nunique())

display(selected_data.head(20))

Loading: District_5_5min_station_data\d05_text_station_5min_2026_05_27.txt
Selected single-day corridor data
Benchmark source file: d05_text_station_5min_2026_05_27.txt
Benchmark date: 2026-05-27
Rows: 4320
Unique dates: 1
Unique timestamps: 288
Unique stations: 15


,timestamp,station_id,station_type,total_flow_5min,avg_occupancy,avg_speed,source_file,flow_vph,date,time_of_day,hour
339,2026-05-27 00:00:00,501015153,ML,16.0,0.0063,68.1,d05_text_station_5min_2026_05_27.txt,192.0,2026-05-27,00:00:00,0
343,2026-05-27 00:00:00,501016013,ML,16.0,0.0063,67.8,d05_text_station_5min_2026_05_27.txt,192.0,2026-05-27,00:00:00,0
344,2026-05-27 00:00:00,501016014,OR,2.0,0.0017,NaN,d05_text_station_5min_2026_05_27.txt,24.0,2026-05-27,00:00:00,0
347,2026-05-27 00:00:00,501016023,ML,18.0,0.0075,67.8,d05_text_station_5min_2026_05_27.txt,216.0,2026-05-27,00:00:00,0
348,2026-05-27 00:00:00,501016024,OR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,00:00:00,0
349,2026-05-27 00:00:00,501016031,ML,19.0,0.0075,68.0,d05_text_station_5min_2026_05_27.txt,228.0,2026-05-27,00:00:00,0
353,2026-05-27 00:00:00,501016043,ML,19.0,0.0074,67.3,d05_text_station_5min_2026_05_27.txt,228.0,2026-05-27,00:00:00,0
354,2026-05-27 00:00:00,501016044,FR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,00:00:00,0
357,2026-05-27 00:00:00,501016053,ML,22.0,0.0093,67.6,d05_text_station_5min_2026_05_27.txt,264.0,2026-05-27,00:00:00,0
359,2026-05-27 00:00:00,501016062,ML,18.0,0.0075,67.1,d05_text_station_5min_2026_05_27.txt,216.0,2026-05-27,00:00:00,0


In [51]:
# 3. FREE-FLOW WINDOW AND FREE-FLOW SPEEDS
selected_data_midnight = selected_data[
    (selected_data["timestamp"].dt.hour >= 1) &
    (selected_data["timestamp"].dt.hour < 5)
].copy()

print("\nMidnight / Free-Flow Window Data")
print("Time window: 01:00–05:00 across all loaded days")
print("Rows:", len(selected_data_midnight))
print("Unique dates:", selected_data_midnight["date"].nunique())
print("Unique timestamps:", selected_data_midnight["timestamp"].nunique())
print("Unique stations:", selected_data_midnight["station_id"].nunique())

display(selected_data_midnight.head(20))

freeflow_mainline = selected_data_midnight[
    selected_data_midnight["station_id"].isin(mainline_ids)
].copy()

median_speed_by_station = (
    freeflow_mainline
    .groupby("station_id")["avg_speed"]
    .median()
    .reset_index()
)

median_speed_by_station.columns = [
    "station_id",
    "median_speed"
]

print("Station-Level Free-Flow Speeds")
display(median_speed_by_station.round(3))


Midnight / Free-Flow Window Data
Time window: 01:00–05:00 across all loaded days
Rows: 720
Unique dates: 1
Unique timestamps: 48
Unique stations: 15


,timestamp,station_id,station_type,total_flow_5min,avg_occupancy,avg_speed,source_file,flow_vph,date,time_of_day,hour
7383,2026-05-27 01:00:00,501015153,ML,9.0,0.0039,67.9,d05_text_station_5min_2026_05_27.txt,108.0,2026-05-27,01:00:00,1
7387,2026-05-27 01:00:00,501016013,ML,7.0,0.0029,66.7,d05_text_station_5min_2026_05_27.txt,84.0,2026-05-27,01:00:00,1
7388,2026-05-27 01:00:00,501016014,OR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,01:00:00,1
7391,2026-05-27 01:00:00,501016023,ML,6.0,0.0025,67.1,d05_text_station_5min_2026_05_27.txt,72.0,2026-05-27,01:00:00,1
7392,2026-05-27 01:00:00,501016024,OR,1.0,0.0008,NaN,d05_text_station_5min_2026_05_27.txt,12.0,2026-05-27,01:00:00,1
7393,2026-05-27 01:00:00,501016031,ML,7.0,0.0027,66.0,d05_text_station_5min_2026_05_27.txt,84.0,2026-05-27,01:00:00,1
7397,2026-05-27 01:00:00,501016043,ML,11.0,0.0044,65.9,d05_text_station_5min_2026_05_27.txt,132.0,2026-05-27,01:00:00,1
7398,2026-05-27 01:00:00,501016044,FR,0.0,0.0000,NaN,d05_text_station_5min_2026_05_27.txt,0.0,2026-05-27,01:00:00,1
7401,2026-05-27 01:00:00,501016053,ML,11.0,0.0048,66.7,d05_text_station_5min_2026_05_27.txt,132.0,2026-05-27,01:00:00,1
7403,2026-05-27 01:00:00,501016062,ML,11.0,0.0046,67.3,d05_text_station_5min_2026_05_27.txt,132.0,2026-05-27,01:00:00,1


Station-Level Free-Flow Speeds


,station_id,median_speed
0,501015153,66.90
1,501016013,66.65
2,501016023,66.80
3,501016031,66.85
4,501016043,66.95
5,501016053,67.05
6,501016062,67.00
7,501016071,66.90
8,501016082,69.55
9,501016091,66.80


In [52]:
# 4. OFFICIAL SINGLE-DAY 2-HOUR BENCHMARK WINDOW
print("Finding congestion pattern for selected single benchmark day")

single_day_mainline_data = selected_data[
    selected_data["station_id"].isin(mainline_ids)
].copy()

if single_day_mainline_data.empty:
    raise ValueError(
        "No mainline data found for selected single-day benchmark."
    )

single_day_hour_summary = (
    single_day_mainline_data
    .groupby("hour")
    .agg(
        median_speed=("avg_speed", "median"),
        mean_speed=("avg_speed", "mean"),
        min_speed=("avg_speed", "min"),
        median_flow_vph=("flow_vph", "median"),
        mean_flow_vph=("flow_vph", "mean"),
        pct_speed_below_60=(
            "avg_speed",
            lambda s: (s < 60.0).mean()
        ),
        pct_speed_below_45=(
            "avg_speed",
            lambda s: (s < 45.0).mean()
        ),
        num_rows=("avg_speed", "size"),
        num_time_slots=("time_of_day", "nunique"),
        num_mainline_stations=("station_id", "nunique"),
    )
    .reset_index()
    .sort_values(
        [
            "median_speed",
            "mean_speed",
            "pct_speed_below_45",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
)

print("Single-day hourly congestion summary")
display(single_day_hour_summary.round(3))


# Official benchmark window:
# May 27, 2026, 16:00–18:00
benchmark_profile_type = "single_observed_day_2hour"

benchmark_start_hour = 16
benchmark_end_hour = 18

benchmark_start_time = pd.to_datetime(
    f"{benchmark_start_hour:02d}:00:00"
).time()

benchmark_end_time = pd.to_datetime(
    f"{benchmark_end_hour:02d}:00:00"
).time()

benchmark_window_label = (
    f"{benchmark_start_hour:02d}:00–{benchmark_end_hour:02d}:00"
)

# CTM timing
delta_t = 0.25
steps_per_5min = 20
steps_per_hour = 240

num_steps = int(
    (
        benchmark_end_hour
        - benchmark_start_hour
    )
    * steps_per_hour
)

print("Selected official single-day benchmark")
print("Benchmark date:", benchmark_date)
print("Benchmark window:", benchmark_window_label)
print("Benchmark profile type:", benchmark_profile_type)
print("num_steps:", num_steps)

assert num_steps == 480

Finding congestion pattern for selected single benchmark day
Single-day hourly congestion summary


,hour,median_speed,mean_speed,min_speed,median_flow_vph,mean_flow_vph,pct_speed_below_60,pct_speed_below_45,num_rows,num_time_slots,num_mainline_stations
17,17,43.80,42.478,10.3,2844.0,2786.4,0.792,0.525,120,12,10
16,16,61.90,46.814,7.0,2382.0,2300.1,0.450,0.375,120,12,10
15,15,63.60,59.966,10.3,2616.0,2419.3,0.158,0.100,120,12,10
14,14,64.10,64.222,52.9,2604.0,2654.0,0.025,0.000,120,12,10
13,13,64.60,64.927,62.9,2160.0,2176.5,0.000,0.000,120,12,10
12,12,65.10,65.328,61.9,2112.0,2114.7,0.000,0.000,120,12,10
18,18,66.00,62.192,23.2,2100.0,2122.5,0.208,0.067,120,12,10
11,11,66.40,66.458,63.6,1908.0,1911.0,0.000,0.000,120,12,10
10,10,66.50,66.748,64.6,1716.0,1726.5,0.000,0.000,120,12,10
9,9,66.65,66.818,63.9,1506.0,1525.0,0.000,0.000,120,12,10


Selected official single-day benchmark
Benchmark date: 2026-05-27
Benchmark window: 16:00–18:00
Benchmark profile type: single_observed_day_2hour
num_steps: 480


In [53]:
# 5. LOAD SINGLE-DAY 2-HOUR BENCHMARK WINDOW DATA
selected_data_afternoon = selected_data[
    (
        selected_data["date"] == benchmark_date
    )
    & (
        selected_data["timestamp"].dt.hour >= benchmark_start_hour
    )
    & (
        selected_data["timestamp"].dt.hour < benchmark_end_hour
    )
].copy()

if selected_data_afternoon.empty:
    raise ValueError(
        f"No rows found for {benchmark_date} {benchmark_window_label}."
    )

print("Single-Day 2-Hour Benchmark Window Data")
print("Benchmark date:", benchmark_date)
print("Benchmark source file:", benchmark_source_file)
print("Time window:", benchmark_window_label)
print("Rows:", len(selected_data_afternoon))
print("Unique dates:", selected_data_afternoon["date"].nunique())
print("Unique timestamps:", selected_data_afternoon["timestamp"].nunique())
print("Unique stations:", selected_data_afternoon["station_id"].nunique())

display(selected_data_afternoon.head(30))


# Since this is a single observed day, each station/time slot should have one row.
single_day_duplicates = (
    selected_data_afternoon
    .groupby(
        [
            "station_id",
            "station_type",
            "time_of_day",
        ]
    )
    .size()
    .reset_index(name="row_count")
)

duplicate_rows = single_day_duplicates[
    single_day_duplicates["row_count"] > 1
].copy()

if len(duplicate_rows) > 0:
    print("Duplicate station/time rows found:")
    display(duplicate_rows.head(50))

    raise ValueError(
        "Single-day benchmark has duplicate station/time rows. "
        "Inspect selected_data_afternoon."
    )


single_day_benchmark_profile = (
    selected_data_afternoon
    .groupby(
        [
            "station_id",
            "station_type",
            "time_of_day",
        ]
    )
    .agg(
        actual_flow_5min=("total_flow_5min", "first"),
        actual_flow_vph=("flow_vph", "first"),
        actual_occupancy=("avg_occupancy", "first"),
        actual_speed=("avg_speed", "first"),
        source_file=("source_file", "first"),
        date=("date", "first"),
    )
    .reset_index()
)

# Backward-compatible aliases.
# Downstream cells expect these names.
# For a single-day benchmark, these are not medians;
# they equal the actual observed values.
typical_pm_profile = single_day_benchmark_profile.copy()

typical_pm_profile["median_flow_5min"] = (
    typical_pm_profile["actual_flow_5min"]
)

typical_pm_profile["median_flow_vph"] = (
    typical_pm_profile["actual_flow_vph"]
)

typical_pm_profile["mean_flow_vph"] = (
    typical_pm_profile["actual_flow_vph"]
)

typical_pm_profile["median_occupancy"] = (
    typical_pm_profile["actual_occupancy"]
)

typical_pm_profile["median_speed"] = (
    typical_pm_profile["actual_speed"]
)

print("Single-day 2-hour benchmark profile")
print("Benchmark date:", benchmark_date)
print("Time window:", benchmark_window_label)
print("Rows:", len(typical_pm_profile))
print("Unique stations:", typical_pm_profile["station_id"].nunique())
print("Unique time slots:", typical_pm_profile["time_of_day"].nunique())

display(typical_pm_profile.head(40))


# Critical coverage checks for CTM
mainline_profile_coverage = typical_pm_profile[
    typical_pm_profile["station_id"].isin(mainline_ids)
].copy()

mainline_time_coverage = (
    mainline_profile_coverage
    .groupby("time_of_day")["station_id"]
    .nunique()
    .reset_index(name="num_mainline_detectors")
)

bad_mainline_time_coverage = mainline_time_coverage[
    mainline_time_coverage["num_mainline_detectors"] != len(mainline_ids)
].copy()

if len(bad_mainline_time_coverage) > 0:
    print("Bad mainline detector coverage by time:")
    display(bad_mainline_time_coverage)

    raise ValueError(
        "Some benchmark time slots do not have all mainline detectors."
    )

expected_5min_rows = int(
    num_steps / steps_per_5min
)

if typical_pm_profile["time_of_day"].nunique() != expected_5min_rows:
    raise ValueError(
        f"Expected {expected_5min_rows} five-minute time slots, "
        f"found {typical_pm_profile['time_of_day'].nunique()}."
    )

print(
    "PASS: single-day 2-hour benchmark profile has full",
    expected_5min_rows,
    "slot mainline coverage."
)

Single-Day 2-Hour Benchmark Window Data
Benchmark date: 2026-05-27
Benchmark source file: d05_text_station_5min_2026_05_27.txt
Time window: 16:00–18:00
Rows: 360
Unique dates: 1
Unique timestamps: 24
Unique stations: 15


,timestamp,station_id,station_type,total_flow_5min,avg_occupancy,avg_speed,source_file,flow_vph,date,time_of_day,hour
113043,2026-05-27 16:00:00,501015153,ML,201.0,0.0803,66.5,d05_text_station_5min_2026_05_27.txt,2412.0,2026-05-27,16:00:00,16
113047,2026-05-27 16:00:00,501016013,ML,198.0,0.0768,66.9,d05_text_station_5min_2026_05_27.txt,2376.0,2026-05-27,16:00:00,16
113048,2026-05-27 16:00:00,501016014,OR,23.0,0.0190,NaN,d05_text_station_5min_2026_05_27.txt,276.0,2026-05-27,16:00:00,16
113051,2026-05-27 16:00:00,501016023,ML,203.0,0.0825,65.9,d05_text_station_5min_2026_05_27.txt,2436.0,2026-05-27,16:00:00,16
113052,2026-05-27 16:00:00,501016024,OR,50.0,0.0450,NaN,d05_text_station_5min_2026_05_27.txt,600.0,2026-05-27,16:00:00,16
113053,2026-05-27 16:00:00,501016031,ML,181.0,0.0688,68.9,d05_text_station_5min_2026_05_27.txt,2172.0,2026-05-27,16:00:00,16
113057,2026-05-27 16:00:00,501016043,ML,176.0,0.0677,67.2,d05_text_station_5min_2026_05_27.txt,2112.0,2026-05-27,16:00:00,16
113058,2026-05-27 16:00:00,501016044,FR,6.0,0.0075,NaN,d05_text_station_5min_2026_05_27.txt,72.0,2026-05-27,16:00:00,16
113061,2026-05-27 16:00:00,501016053,ML,188.0,0.0799,64.8,d05_text_station_5min_2026_05_27.txt,2256.0,2026-05-27,16:00:00,16
113063,2026-05-27 16:00:00,501016062,ML,129.0,0.0541,63.2,d05_text_station_5min_2026_05_27.txt,1548.0,2026-05-27,16:00:00,16


Single-day 2-hour benchmark profile
Benchmark date: 2026-05-27
Time window: 16:00–18:00
Rows: 360
Unique stations: 15
Unique time slots: 24


,station_id,station_type,time_of_day,actual_flow_5min,actual_flow_vph,actual_occupancy,actual_speed,source_file,date,median_flow_5min,median_flow_vph,mean_flow_vph,median_occupancy,median_speed
0,501015153,ML,16:00:00,201.0,2412.0,0.0803,66.5,d05_text_station_5min_2026_05_27.txt,2026-05-27,201.0,2412.0,2412.0,0.0803,66.5
1,501015153,ML,16:05:00,194.0,2328.0,0.0791,65.9,d05_text_station_5min_2026_05_27.txt,2026-05-27,194.0,2328.0,2328.0,0.0791,65.9
2,501015153,ML,16:10:00,201.0,2412.0,0.0805,65.8,d05_text_station_5min_2026_05_27.txt,2026-05-27,201.0,2412.0,2412.0,0.0805,65.8
3,501015153,ML,16:15:00,179.0,2148.0,0.0713,66.3,d05_text_station_5min_2026_05_27.txt,2026-05-27,179.0,2148.0,2148.0,0.0713,66.3
4,501015153,ML,16:20:00,190.0,2280.0,0.0766,65.9,d05_text_station_5min_2026_05_27.txt,2026-05-27,190.0,2280.0,2280.0,0.0766,65.9
5,501015153,ML,16:25:00,215.0,2580.0,0.0864,66.2,d05_text_station_5min_2026_05_27.txt,2026-05-27,215.0,2580.0,2580.0,0.0864,66.2
6,501015153,ML,16:30:00,231.0,2772.0,0.0963,64.9,d05_text_station_5min_2026_05_27.txt,2026-05-27,231.0,2772.0,2772.0,0.0963,64.9
7,501015153,ML,16:35:00,248.0,2976.0,0.1058,63.6,d05_text_station_5min_2026_05_27.txt,2026-05-27,248.0,2976.0,2976.0,0.1058,63.6
8,501015153,ML,16:40:00,230.0,2760.0,0.0958,63.5,d05_text_station_5min_2026_05_27.txt,2026-05-27,230.0,2760.0,2760.0,0.0958,63.5
9,501015153,ML,16:45:00,242.0,2904.0,0.1035,62.8,d05_text_station_5min_2026_05_27.txt,2026-05-27,242.0,2904.0,2904.0,0.1035,62.8


PASS: single-day 2-hour benchmark profile has full 24 slot mainline coverage.


In [54]:
# 6. Segment free-flow speed calculation
def segment_free_flow_speed(upstream_id, downstream_id, median_speed_df):
    v_upstream_series = median_speed_df.loc[
        median_speed_df["station_id"] == upstream_id,
        "median_speed"
    ]

    v_downstream_series = median_speed_df.loc[
        median_speed_df["station_id"] == downstream_id,
        "median_speed"
    ]

    if v_upstream_series.empty:
        raise ValueError(f"Missing median speed for upstream station {upstream_id}")

    if v_downstream_series.empty:
        raise ValueError(f"Missing median speed for downstream station {downstream_id}")

    v_upstream = v_upstream_series.iloc[0]
    v_downstream = v_downstream_series.iloc[0]

    return (2 * v_upstream * v_downstream) / (v_upstream + v_downstream)


mainline_segments = [
    {
        "segment": "S1",
        "from_id": 501015153,
        "to_id": 501016013,
        "from_station": "4TH ST",
        "to_station": "4TH ST ON AREA",
    },
    {
        "segment": "S2",
        "from_id": 501016013,
        "to_id": 501016023,
        "from_station": "4TH ST ON AREA",
        "to_station": "PRICE ST",
    },
    {
        "segment": "S3",
        "from_id": 501016023,
        "to_id": 501016031,
        "from_station": "PRICE ST",
        "to_station": "HINDS AVE",
    },
    {
        "segment": "S4",
        "from_id": 501016031,
        "to_id": 501016043,
        "from_station": "HINDS AVE",
        "to_station": "BELLO ST",
    },
    {
        "segment": "S5",
        "from_id": 501016043,
        "to_id": 501016053,
        "from_station": "BELLO ST",
        "to_station": "SHELL BEACH RD",
    },
    {
        "segment": "S6",
        "from_id": 501016053,
        "to_id": 501016062,
        "from_station": "SHELL BEACH RD",
        "to_station": "MATTIE RD",
    },
    {
        "segment": "S7",
        "from_id": 501016062,
        "to_id": 501016071,
        "from_station": "MATTIE RD",
        "to_station": "SPYGLASS DR",
    },
    {
        "segment": "S8",
        "from_id": 501016071,
        "to_id": 501016082,
        "from_station": "SPYGLASS DR",
        "to_station": "AVILA BEACH DR",
    },
    {
        "segment": "S9",
        "from_id": 501016082,
        "to_id": 501016091,
        "from_station": "AVILA BEACH DR",
        "to_station": "SAN LUIS BAY DR",
    },
]

segment_rows = []

for seg in mainline_segments:
    v_ff = segment_free_flow_speed(
        seg["from_id"],
        seg["to_id"],
        median_speed_by_station
    )

    segment_rows.append({
        "segment": seg["segment"],
        "from_station_id": seg["from_id"],
        "to_station_id": seg["to_id"],
        "from_station": seg["from_station"],
        "to_station": seg["to_station"],
        "v_ff_mph": v_ff
    })
print("Segment Free-Flow-speed")
segment_free_flow_df = pd.DataFrame(segment_rows)
segment_free_flow_df["v_ff_mph"] = segment_free_flow_df["v_ff_mph"].round(3)

display(segment_free_flow_df)

Segment Free-Flow-speed


,segment,from_station_id,to_station_id,from_station,to_station,v_ff_mph
0,S1,501015153,501016013,4TH ST,4TH ST ON AREA,66.775
1,S2,501016013,501016023,4TH ST ON AREA,PRICE ST,66.725
2,S3,501016023,501016031,PRICE ST,HINDS AVE,66.825
3,S4,501016031,501016043,HINDS AVE,BELLO ST,66.900
4,S5,501016043,501016053,BELLO ST,SHELL BEACH RD,67.000
5,S6,501016053,501016062,SHELL BEACH RD,MATTIE RD,67.025
6,S7,501016062,501016071,MATTIE RD,SPYGLASS DR,66.950
7,S8,501016071,501016082,SPYGLASS DR,AVILA BEACH DR,68.199
8,S9,501016082,501016091,AVILA BEACH DR,SAN LUIS BAY DR,68.147


In [55]:
# CTM TIME SETTINGS
# May 27, 16:00–18:00 => 2 hours * 240 steps/hour = 480 steps.
delta_t = 0.25  # minutes, 15 seconds

five_min_intervals_per_hour = 12
steps_per_5min = 20
steps_per_hour = int(60 / delta_t)

benchmark_duration_hours = (
    benchmark_end_hour
    - benchmark_start_hour
)

num_steps = int(
    benchmark_duration_hours
    * steps_per_hour
)

expected_5min_rows = int(
    num_steps
    / steps_per_5min
)

print("CTM time settings")
print("Benchmark window:", benchmark_window_label)
print("Benchmark duration hours:", benchmark_duration_hours)
print("five_min_intervals_per_hour:", five_min_intervals_per_hour)
print("delta_t:", delta_t)
print("steps_per_5min:", steps_per_5min)
print("steps_per_hour:", steps_per_hour)
print("num_steps:", num_steps)
print("expected_5min_rows:", expected_5min_rows)

assert five_min_intervals_per_hour == 12
assert steps_per_5min == 20
assert steps_per_hour == 240
assert num_steps == 480
assert expected_5min_rows == 24

CTM time settings
Benchmark window: 16:00–18:00
Benchmark duration hours: 2
five_min_intervals_per_hour: 12
delta_t: 0.25
steps_per_5min: 20
steps_per_hour: 240
num_steps: 480
expected_5min_rows: 24


In [56]:
#CTM DOORWAY CAPACITY
base_doorway_capacity_rows = []
for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"
    segment = seg["segment"]

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    from_station = seg["from_station"]
    to_station = seg["to_station"]

    lanes = station_lane_count[to_id]

    v_ff = float(
        segment_free_flow_df.loc[
            segment_free_flow_df["segment"] == segment,
            "v_ff_mph"
        ].iloc[0]
    )

    if v_ff <= 70:
        base_capacity_per_lane_vph = 1700 + 10 * v_ff
    else:
        base_capacity_per_lane_vph = 2400

    capacity_vph = base_capacity_per_lane_vph * lanes
    capacity_per_5min = capacity_vph / five_min_intervals_per_hour
    capacity_per_15sec = capacity_vph / steps_per_hour

    base_doorway_capacity_rows.append({
        "cell": cell,
        "segment": segment,
        "from_station_id": from_id,
        "to_station_id": to_id,
        "from_station": from_station,
        "to_station": to_station,
        "v_ff_mph": v_ff,
        "lanes": lanes,
        "base_capacity_per_lane_vph": base_capacity_per_lane_vph,
        "capacity_vph": capacity_vph,
        "capacity_per_5min": capacity_per_5min,
        "capacity_per_15sec": capacity_per_15sec,
    })

base_doorway_capacity_df = pd.DataFrame(base_doorway_capacity_rows)

ctm_doorway_capacity_base = {
    row["cell"]: float(row["capacity_per_15sec"])
    for _, row in base_doorway_capacity_df.iterrows()
}

expected_cells = [f"Cell {i}" for i in range(1, 10)]

assert list(ctm_doorway_capacity_base.keys()) == expected_cells
assert all(ctm_doorway_capacity_base[cell] > 0 for cell in expected_cells)

print("Base CTM doorway capacity built successfully.")
display(base_doorway_capacity_df.round(3))

for cell in expected_cells:
    print(cell, round(ctm_doorway_capacity_base[cell], 3))

Base CTM doorway capacity built successfully.


,cell,segment,from_station_id,to_station_id,from_station,to_station,v_ff_mph,lanes,base_capacity_per_lane_vph,capacity_vph,capacity_per_5min,capacity_per_15sec
0,Cell 1,S1,501015153,501016013,4TH ST,4TH ST ON AREA,66.775,2,2367.75,4735.50,394.625,19.731
1,Cell 2,S2,501016013,501016023,4TH ST ON AREA,PRICE ST,66.725,2,2367.25,4734.50,394.542,19.727
2,Cell 3,S3,501016023,501016031,PRICE ST,HINDS AVE,66.825,2,2368.25,4736.50,394.708,19.735
3,Cell 4,S4,501016031,501016043,HINDS AVE,BELLO ST,66.900,2,2369.00,4738.00,394.833,19.742
4,Cell 5,S5,501016043,501016053,BELLO ST,SHELL BEACH RD,67.000,2,2370.00,4740.00,395.000,19.750
5,Cell 6,S6,501016053,501016062,SHELL BEACH RD,MATTIE RD,67.025,2,2370.25,4740.50,395.042,19.752
6,Cell 7,S7,501016062,501016071,MATTIE RD,SPYGLASS DR,66.950,2,2369.50,4739.00,394.917,19.746
7,Cell 8,S8,501016071,501016082,SPYGLASS DR,AVILA BEACH DR,68.199,3,2381.99,7145.97,595.498,29.775
8,Cell 9,S9,501016082,501016091,AVILA BEACH DR,SAN LUIS BAY DR,68.147,2,2381.47,4762.94,396.912,19.846


Cell 1 19.731
Cell 2 19.727
Cell 3 19.735
Cell 4 19.742
Cell 5 19.75
Cell 6 19.752
Cell 7 19.746
Cell 8 29.775
Cell 9 19.846


In [57]:
# DATA-JUSTIFIED SPYGLASS / AVILA / SAN LUIS BAY BOTTLENECK CAPACITY
spyglass_ml_id = 501016071
avila_ml_id = 501016082
san_luis_bay_ml_id = 501016091

raw_pm_data = selected_data_afternoon.copy()

required_cols = [
    "timestamp",
    "station_id",
    "total_flow_5min",
    "avg_speed",
    "date",
    "time_of_day",
]

missing_cols = [
    col
    for col in required_cols
    if col not in raw_pm_data.columns
]

if missing_cols:
    raise KeyError(
        f"selected_data_afternoon is missing columns: {missing_cols}"
    )

raw_pm_data["station_id"] = pd.to_numeric(
    raw_pm_data["station_id"],
    errors="coerce"
).astype("Int64")

raw_pm_data["flow_vph"] = (
    pd.to_numeric(
        raw_pm_data["total_flow_5min"],
        errors="coerce"
    )
    * 12.0
)

raw_pm_data["speed_mph"] = pd.to_numeric(
    raw_pm_data["avg_speed"],
    errors="coerce"
)

bottleneck_station_ids = [
    spyglass_ml_id,
    avila_ml_id,
    san_luis_bay_ml_id,
]

bottleneck_raw = raw_pm_data[
    raw_pm_data["station_id"].isin(bottleneck_station_ids)
].copy()

bottleneck_raw = bottleneck_raw.dropna(
    subset=[
        "flow_vph",
        "speed_mph",
    ]
).copy()

if bottleneck_raw.empty:
    raise ValueError(
        "No raw bottleneck rows found."
    )

flow_pivot = bottleneck_raw.pivot_table(
    index=[
        "date",
        "time_of_day",
    ],
    columns="station_id",
    values="flow_vph",
    aggfunc="first"
)

speed_pivot = bottleneck_raw.pivot_table(
    index=[
        "date",
        "time_of_day",
    ],
    columns="station_id",
    values="speed_mph",
    aggfunc="first"
)

required_station_cols = [
    spyglass_ml_id,
    avila_ml_id,
    san_luis_bay_ml_id,
]

for station_id in required_station_cols:
    if station_id not in flow_pivot.columns:
        raise KeyError(
            f"Missing flow data for station {station_id}"
        )

    if station_id not in speed_pivot.columns:
        raise KeyError(
            f"Missing speed data for station {station_id}"
        )

bottleneck_discharge_raw_df = pd.DataFrame({
    "date": [
        idx[0]
        for idx in flow_pivot.index
    ],
    "time_of_day": [
        idx[1]
        for idx in flow_pivot.index
    ],

    "spyglass_flow_vph": flow_pivot[spyglass_ml_id].values,
    "avila_flow_vph": flow_pivot[avila_ml_id].values,
    "san_luis_bay_flow_vph": flow_pivot[san_luis_bay_ml_id].values,

    "spyglass_speed": speed_pivot[spyglass_ml_id].values,
    "avila_speed": speed_pivot[avila_ml_id].values,
    "san_luis_bay_speed": speed_pivot[san_luis_bay_ml_id].values,
})

bottleneck_discharge_raw_df = bottleneck_discharge_raw_df.dropna().copy()

bottleneck_discharge_raw_df = bottleneck_discharge_raw_df.sort_values(
    [
        "date",
        "time_of_day",
    ]
).reset_index(drop=True)

congestion_speed_threshold_mph = 45.0
downstream_discharge_speed_threshold_mph = 50.0

bottleneck_discharge_raw_df["bottleneck_active"] = (
    (
        bottleneck_discharge_raw_df["spyglass_speed"]
        < congestion_speed_threshold_mph
    )
    & (
        bottleneck_discharge_raw_df["san_luis_bay_speed"]
        >= downstream_discharge_speed_threshold_mph
    )
)

active_discharge_raw = bottleneck_discharge_raw_df[
    bottleneck_discharge_raw_df["bottleneck_active"]
].copy()

if len(active_discharge_raw) == 0:
    raise ValueError(
        "No active bottleneck periods found. "
        "Inspect Spyglass/San Luis Bay speeds or justify a different threshold."
    )

observed_avila_queue_flow_vph_mean = float(
    active_discharge_raw["avila_flow_vph"].mean()
)

observed_avila_queue_flow_vph_p15 = float(
    active_discharge_raw["avila_flow_vph"].quantile(0.15)
)

observed_avila_queue_flow_vph_p85 = float(
    active_discharge_raw["avila_flow_vph"].quantile(0.85)
)

observed_avila_queue_flow_vph_median = float(
    active_discharge_raw["avila_flow_vph"].median()
)

observed_san_luis_bay_discharge_vph_median = float(
    active_discharge_raw["san_luis_bay_flow_vph"].median()
)

observed_bottleneck_discharge_vph = (
    observed_san_luis_bay_discharge_vph_median
)

observed_bottleneck_capacity_per_15sec = (
    observed_bottleneck_discharge_vph
    / steps_per_hour
)


# Cell 9 receiving represents the Cell 8 -> Cell 9 bottleneck interface.
CAP9_VPH = 2400.0
ctm_inflow_capacity_official = ctm_doorway_capacity_base.copy()
ctm_inflow_capacity_official["Cell 9"] = (
    CAP9_VPH
    / steps_per_hour
)

# Cell 9 outflow represents the downstream exit past San Luis Bay,
ctm_outflow_capacity_official = ctm_doorway_capacity_base.copy()
ctm_outflow_capacity_official["Cell 9"] = (
    CAP9_VPH
    / steps_per_hour
)
# Legacy alias only for tables / old diagnostics.
ctm_doorway_capacity_official = ctm_inflow_capacity_official.copy()

print("Raw active bottleneck periods used for observed capacity")
display(
    active_discharge_raw[
        [
            "date",
            "time_of_day",
            "spyglass_flow_vph",
            "avila_flow_vph",
            "san_luis_bay_flow_vph",
            "spyglass_speed",
            "avila_speed",
            "san_luis_bay_speed",
        ]
    ].round(3)
)

capacity_justification_df = pd.DataFrame([
    {
        "metric": "active_bottleneck_periods",
        "value": len(active_discharge_raw),
        "unit": "5-min periods",
    },
    {
        "metric": "spyglass_congestion_speed_threshold",
        "value": congestion_speed_threshold_mph,
        "unit": "mph",
    },
    {
        "metric": "downstream_discharge_speed_threshold",
        "value": downstream_discharge_speed_threshold_mph,
        "unit": "mph",
    },
    {
        "metric": "observed_avila_queue_flow_median",
        "value": observed_avila_queue_flow_vph_median,
        "unit": "veh/hour",
    },
    {
        "metric": "san_luis_bay_discharge_median",
        "value": observed_san_luis_bay_discharge_vph_median,
        "unit": "veh/hour",
    },
    {
        "metric": "cell9_inflow_bottleneck_capacity",
        "value": ctm_inflow_capacity_official["Cell 9"],
        "unit": "veh/15-sec",
    },
    {
        "metric": "cell9_inflow_bottleneck_capacity_vph",
        "value": ctm_inflow_capacity_official["Cell 9"] * steps_per_hour,
        "unit": "veh/hour",
    },
    {
        "metric": "cell9_outflow_capacity",
        "value": ctm_outflow_capacity_official["Cell 9"],
        "unit": "veh/15-sec",
    },
    {
        "metric": "cell9_outflow_capacity_vph",
        "value": ctm_outflow_capacity_official["Cell 9"] * steps_per_hour,
        "unit": "veh/hour",
    },
])

print("Data-justified split inflow/outflow bottleneck capacity")
display(capacity_justification_df.round(3))

assert list(ctm_inflow_capacity_official.keys()) == cell_ids
assert list(ctm_outflow_capacity_official.keys()) == cell_ids

print("Official CTM doorway capacity built successfully.")

for cell in expected_cells:
    print(
        cell,
        "capacity per 15 sec =",
        round(ctm_doorway_capacity_official[cell], 3),
        "| vph =",
        round(
            ctm_doorway_capacity_official[cell]
            * steps_per_hour,
            1
        )
    )

Raw active bottleneck periods used for observed capacity


,date,time_of_day,spyglass_flow_vph,avila_flow_vph,san_luis_bay_flow_vph,spyglass_speed,avila_speed,san_luis_bay_speed
21,2026-05-27,17:45:00,2700.0,2256.0,3192.0,22.9,13.8,52.3
22,2026-05-27,17:50:00,2460.0,2412.0,2160.0,21.0,13.5,58.5
23,2026-05-27,17:55:00,2736.0,2100.0,2268.0,22.8,20.8,62.1


Data-justified split inflow/outflow bottleneck capacity


,metric,value,unit
0,active_bottleneck_periods,3.0,5-min periods
1,spyglass_congestion_speed_threshold,45.0,mph
2,downstream_discharge_speed_threshold,50.0,mph
3,observed_avila_queue_flow_median,2256.0,veh/hour
4,san_luis_bay_discharge_median,2268.0,veh/hour
5,cell9_inflow_bottleneck_capacity,10.0,veh/15-sec
6,cell9_inflow_bottleneck_capacity_vph,2400.0,veh/hour
7,cell9_outflow_capacity,10.0,veh/15-sec
8,cell9_outflow_capacity_vph,2400.0,veh/hour


Official CTM doorway capacity built successfully.
Cell 1 capacity per 15 sec = 19.731 | vph = 4735.5
Cell 2 capacity per 15 sec = 19.727 | vph = 4734.5
Cell 3 capacity per 15 sec = 19.735 | vph = 4736.5
Cell 4 capacity per 15 sec = 19.742 | vph = 4738.0
Cell 5 capacity per 15 sec = 19.75 | vph = 4740.0
Cell 6 capacity per 15 sec = 19.752 | vph = 4740.5
Cell 7 capacity per 15 sec = 19.746 | vph = 4739.0
Cell 8 capacity per 15 sec = 29.775 | vph = 7146.0
Cell 9 capacity per 15 sec = 10.0 | vph = 2400.0


In [58]:
# FREE-FLOW TRAVEL TIME CALCULATION
segment_free_flow_df["length_miles"] = (
    segment_free_flow_df["to_station_id"].map(station_pm)
    - segment_free_flow_df["from_station_id"].map(station_pm)
)

segment_free_flow_df["TT_ff_hr"] = (
    segment_free_flow_df["length_miles"] / segment_free_flow_df["v_ff_mph"]
)

segment_free_flow_df["TT_ff_min"] = (
    segment_free_flow_df["TT_ff_hr"] * 60
)

free_flow_tt_df = segment_free_flow_df.copy()

display(
    free_flow_tt_df[
        [
            "segment",
            "from_station",
            "to_station",
            "length_miles",
            "v_ff_mph",
            "TT_ff_hr",
            "TT_ff_min",
        ]
    ].round(5)
)

tt_ff_min = {
    f"Cell {i+1}": float(row["TT_ff_min"])
    for i, row in segment_free_flow_df.iterrows()
}

total_h = segment_free_flow_df["TT_ff_hr"].sum()
total_min = segment_free_flow_df["TT_ff_min"].sum()

print("Total free-flow travel time:")
print("Hours:", round(total_h, 5))
print("Minutes:", round(total_min, 5))
print("Total corridor length:", round(segment_free_flow_df["length_miles"].sum(), 5), "miles")

print("tt_ff_min:")
for cell in expected_cells:
    print(cell, round(tt_ff_min[cell], 5))

,segment,from_station,to_station,length_miles,v_ff_mph,TT_ff_hr,TT_ff_min
0,S1,4TH ST,4TH ST ON AREA,0.515,66.775,0.00771,0.46275
1,S2,4TH ST ON AREA,PRICE ST,0.449,66.725,0.00673,0.40375
2,S3,PRICE ST,HINDS AVE,0.502,66.825,0.00751,0.45073
3,S4,HINDS AVE,BELLO ST,0.510,66.900,0.00762,0.45740
4,S5,BELLO ST,SHELL BEACH RD,0.737,67.000,0.01100,0.66000
5,S6,SHELL BEACH RD,MATTIE RD,0.345,67.025,0.00515,0.30884
6,S7,MATTIE RD,SPYGLASS DR,1.526,66.950,0.02279,1.36759
7,S8,SPYGLASS DR,AVILA BEACH DR,1.141,68.199,0.01673,1.00383
8,S9,AVILA BEACH DR,SAN LUIS BAY DR,1.057,68.147,0.01551,0.93064


Total free-flow travel time:
Hours: 0.10076
Minutes: 6.04551
Total corridor length: 6.782 miles
tt_ff_min:
Cell 1 0.46275
Cell 2 0.40375
Cell 3 0.45073
Cell 4 0.4574
Cell 5 0.66
Cell 6 0.30884
Cell 7 1.36759
Cell 8 1.00383
Cell 9 0.93064


## State-Based $CTM$ Benchmark Setup and Simulation


In [59]:
def build_15sec_series_from_5min_flow(station_id, typical_profile):
    station_5min_data = typical_profile[
        typical_profile["station_id"] == station_id
    ].copy()

    station_5min_data = station_5min_data.sort_values(
        "time_of_day"
    ).reset_index(drop=True)

    expected_5min_rows = int(
        num_steps / steps_per_5min
    )

    if len(station_5min_data) != expected_5min_rows:
        raise ValueError(
            f"Station {station_id} has {len(station_5min_data)} 5-min rows, "
            f"but expected {expected_5min_rows} rows for {benchmark_window_label}."
        )

    station_5min_data["median_flow_5min"] = pd.to_numeric(
        station_5min_data["median_flow_5min"],
        errors="coerce"
    )

    if station_5min_data["median_flow_5min"].isna().any():
        raise ValueError(
            f"Station {station_id} has NaN median_flow_5min values."
        )

    flow_15sec_series = []

    for flow_5min in station_5min_data["median_flow_5min"]:
        flow_15sec = float(flow_5min) / steps_per_5min

        for _ in range(steps_per_5min):
            flow_15sec_series.append(flow_15sec)

    if len(flow_15sec_series) != num_steps:
        raise ValueError(
            f"Station {station_id} produced {len(flow_15sec_series)} steps, "
            f"but expected {num_steps}."
        )

    return flow_15sec_series

In [60]:
# BUILD 480-STEP CTM INPUT SERIES FROM SINGLE-DAY 2-HOUR BENCHMARK PROFILE
# 1. Boundary mainline inflow entering Cell 1
q_in_boundary_series = build_15sec_series_from_5min_flow(
    501015153,
    typical_pm_profile
)

arrival_multiplier=1.2

# 2. Observed controllable on-ramp releases
observed_release_series = {
    "u_4th": build_15sec_series_from_5min_flow(
        501016014,
        typical_pm_profile
    ),
    "u_price": build_15sec_series_from_5min_flow(
        501016024,
        typical_pm_profile
    ),
    "u_mattie": build_15sec_series_from_5min_flow(
        501016063,
        typical_pm_profile
    ),
    "u_avila": build_15sec_series_from_5min_flow(
        501016083,
        typical_pm_profile
    ),
}


# 3. Ramp arrivals
# For benchmark replay, use observed releases as the observed arrival baseline.
ramp_arrival_series = {
    ramp: [
        arrival_multiplier * float(observed_release_series[ramp][step])
        for step in range(num_steps)
    ]
    for ramp in ramp_ids
}


# 4. Mainline detector flow series for benchmark window
mainline_flow_series = [
    build_15sec_series_from_5min_flow(
        station_id,
        typical_pm_profile
    )
    for station_id in mainline_ids
]


# 5. Detected on-ramp flow by cell
detected_onramp_series_by_cell = {
    cell: [
        0.0
        for _ in range(num_steps)
    ]
    for cell in cell_ids
}

for ramp, cell in generic_ramp_cell_map.items():
    detected_onramp_series_by_cell[cell] = [
        detected_onramp_series_by_cell[cell][step]
        + observed_release_series[ramp][step]
        for step in range(num_steps)
    ]

assert detected_onramp_series_by_cell["Cell 9"] == [
    0.0
    for _ in range(num_steps)
], "Avila must not enter normal u_in_series."


# 6. Detected off-ramp flow by cell
# Old code placed Bello in Cell 4. That was geometrically wrong.
detected_offramp_cell_map = {
    "bello": "Cell 5"
}

observed_offramp_series = {
    "bello": build_15sec_series_from_5min_flow(
        501016044,
        typical_pm_profile
    )
}

detected_offramp_series_by_cell = {
    cell: [
        0.0
        for _ in range(num_steps)
    ]
    for cell in cell_ids
}

for off_name, cell in detected_offramp_cell_map.items():
    detected_offramp_series_by_cell[cell] = [
        detected_offramp_series_by_cell[cell][step]
        + observed_offramp_series[off_name][step]
        for step in range(num_steps)
    ]


# 7. Helper: total vehicles for one station in selected_data over any hour window
def total_selected_data_vehicles(
    station_id,
    start_hour,
    end_hour
):
    station_rows = selected_data[
        (
            selected_data["station_id"] == station_id
        )
        & (
            selected_data["date"] == benchmark_date
        )
        & (
            selected_data["timestamp"].dt.hour >= start_hour
        )
        & (
            selected_data["timestamp"].dt.hour < end_hour
        )
    ].copy()

    expected_rows = int(
        (
            end_hour
            - start_hour
        )
        * five_min_intervals_per_hour
    )

    if len(station_rows) != expected_rows:
        raise ValueError(
            f"Station {station_id} has {len(station_rows)} rows "
            f"from {start_hour}:00 to {end_hour}:00, "
            f"but expected {expected_rows}."
        )

    station_rows["total_flow_5min"] = pd.to_numeric(
        station_rows["total_flow_5min"],
        errors="coerce"
    )

    if station_rows["total_flow_5min"].isna().any():
        raise ValueError(
            f"Station {station_id} has NaN total_flow_5min values."
        )

    return float(
        station_rows["total_flow_5min"].sum()
    )


# 8. Compute exact detector-balance residuals
def build_segment_balance_audit(
    start_hour,
    end_hour,
    label
):
    rows = []

    duration_hours = (
        end_hour
        - start_hour
    )

    for i, cell in enumerate(cell_ids):
        upstream_id = mainline_ids[i]
        downstream_id = mainline_ids[i + 1]

        q_up = total_selected_data_vehicles(
            upstream_id,
            start_hour,
            end_hour
        )

        q_down = total_selected_data_vehicles(
            downstream_id,
            start_hour,
            end_hour
        )

        detected_on = 0.0

        for ramp in ramp_ids:
            if ramp_cell_map[ramp] == cell:
                detected_on += total_selected_data_vehicles(
                    {
                        "u_4th": 501016014,
                        "u_price": 501016024,
                        "u_mattie": 501016063,
                        "u_avila": 501016083,
                    }[ramp],
                    start_hour,
                    end_hour
                )

        detected_off = 0.0

        for off_name, off_cell in detected_offramp_cell_map.items():
            if off_cell == cell:
                detected_off += total_selected_data_vehicles(
                    {
                        "bello": 501016044,
                    }[off_name],
                    start_hour,
                    end_hour
                )

        net_residual = (
            q_down
            - q_up
            - detected_on
            + detected_off
        )

        inferred_missing_exit = max(
            -net_residual,
            0.0
        )

        inferred_missing_entry = max(
            net_residual,
            0.0
        )

        denominator_for_exit_split = (
            q_up
            + detected_on
        )

        if denominator_for_exit_split > 0:
            exit_split_fraction = (
                inferred_missing_exit
                / denominator_for_exit_split
            )
        else:
            exit_split_fraction = 0.0

        rows.append({
            "audit_window": label,
            "cell": cell,
            "from_detector": upstream_id,
            "to_detector": downstream_id,
            "q_up_veh": q_up,
            "q_down_veh": q_down,
            "detected_on_veh": detected_on,
            "detected_off_veh": detected_off,
            "net_residual_veh": net_residual,
            "inferred_missing_exit_veh": inferred_missing_exit,
            "inferred_missing_entry_veh": inferred_missing_entry,
            "missing_exit_vph": (
                inferred_missing_exit
                / duration_hours
            ),
            "missing_entry_vph": (
                inferred_missing_entry
                / duration_hours
            ),
            "exit_split_fraction": exit_split_fraction,
        })

    return pd.DataFrame(rows)

free_flow_calibration_start_hour = 13
free_flow_calibration_end_hour = 15

# Calibrated against observed 16:00–18:00 state trajectory.
BETA_SCALE = 1.5
ENTRY_SCALE = 0.75
MERGE_PRIORITY = 0.3


benchmark_balance_audit_df = build_segment_balance_audit(
    start_hour=benchmark_start_hour,
    end_hour=benchmark_end_hour,
    label=benchmark_window_label
)

free_flow_balance_audit_df = build_segment_balance_audit(
    start_hour=free_flow_calibration_start_hour,
    end_hour=free_flow_calibration_end_hour,
    label=(
        f"{free_flow_calibration_start_hour:02d}:00–"
        f"{free_flow_calibration_end_hour:02d}:00"
    )
)

print("Benchmark-window exact detector-balance residuals")
display(benchmark_balance_audit_df.round(6))

print("Free-flow detector-balance calibration residuals")
display(free_flow_balance_audit_df.round(6))


# 9. Build exit split fractions and undetected entry constants from free-flow balance

raw_exit_split_by_cell = {}
raw_undetected_entry_vph_by_cell = {}

exit_split_by_cell = {}
undetected_entry_vph_by_cell = {}

for _, row in free_flow_balance_audit_df.iterrows():
    cell = row["cell"]

    raw_beta = float(row["exit_split_fraction"])
    raw_entry_vph = float(row["missing_entry_vph"])

    raw_exit_split_by_cell[cell] = raw_beta
    raw_undetected_entry_vph_by_cell[cell] = raw_entry_vph

    exit_split_by_cell[cell] = min(
        0.6,
        max(0.0, raw_beta) * BETA_SCALE
    )

    undetected_entry_vph_by_cell[cell] = (
        max(0.0, raw_entry_vph)
        * ENTRY_SCALE
    )


undetected_entry_per_15sec_by_cell = {
    cell: (
        undetected_entry_vph_by_cell[cell]
        / steps_per_hour
    )
    for cell in cell_ids
}

undetected_entry_series_by_cell = {
    cell: [
        undetected_entry_per_15sec_by_cell[cell]
        for _ in range(num_steps)
    ]
    for cell in cell_ids
}

exit_split_df = pd.DataFrame([
    {
        "cell": cell,
        "raw_exit_split_fraction": raw_exit_split_by_cell[cell],
        "calibrated_exit_split_fraction": exit_split_by_cell[cell],
        "raw_undetected_entry_vph": raw_undetected_entry_vph_by_cell[cell],
        "calibrated_undetected_entry_vph": undetected_entry_vph_by_cell[cell],
        "undetected_entry_per_15sec": undetected_entry_per_15sec_by_cell[cell],
    }
    for cell in cell_ids
])

print("Free-flow-calibrated undetected ramp model")
display(exit_split_df.round(6))


# 10. Build final CTM lateral series
u_in_series = []
f_out_series = []

residual_inflow_series = []
residual_outflow_series = []

for step in range(num_steps):
    u_step = {
        cell: (
            float(detected_onramp_series_by_cell[cell][step])
            + float(undetected_entry_series_by_cell[cell][step])
        )
        for cell in cell_ids
    }

    f_step = {
        cell: float(
            detected_offramp_series_by_cell[cell][step]
        )
        for cell in cell_ids
    }

    residual_u_step = {
        cell: 0.0
        for cell in cell_ids
    }

    residual_f_step = {
        cell: 0.0
        for cell in cell_ids
    }

    for i, cell in enumerate(cell_ids):
        q_up = float(
            mainline_flow_series[i][step]
        )

        q_down = float(
            mainline_flow_series[i + 1][step]
        )

        detected_on = float(
            detected_onramp_series_by_cell[cell][step]
        )

        detected_off = float(
            detected_offramp_series_by_cell[cell][step]
        )

        detector_residual = (
            q_down
            - q_up
            - detected_on
            + detected_off
        )

        residual_u_step[cell] = max(
            detector_residual,
            0.0
        )

        residual_f_step[cell] = max(
            -detector_residual,
            0.0
        )

    u_in_series.append(u_step)
    f_out_series.append(f_step)

    residual_inflow_series.append(residual_u_step)
    residual_outflow_series.append(residual_f_step)


# 11. Sanity checks
assert len(q_in_boundary_series) == num_steps
assert len(u_in_series) == num_steps
assert len(f_out_series) == num_steps
assert len(residual_inflow_series) == num_steps
assert len(residual_outflow_series) == num_steps

for ramp in ramp_ids:
    assert len(observed_release_series[ramp]) == num_steps
    assert len(ramp_arrival_series[ramp]) == num_steps

for step in range(num_steps):
    assert list(u_in_series[step].keys()) == cell_ids
    assert list(f_out_series[step].keys()) == cell_ids
    assert list(residual_inflow_series[step].keys()) == cell_ids
    assert list(residual_outflow_series[step].keys()) == cell_ids

print("Created CTM input series with undetected exits modeled as split fractions.")
print("q_in_boundary_series:", len(q_in_boundary_series))
print("u_in_series:", len(u_in_series))
print("f_out_series:", len(f_out_series))
print("residual_inflow_series:", len(residual_inflow_series))
print("residual_outflow_series:", len(residual_outflow_series))

Benchmark-window exact detector-balance residuals


,audit_window,cell,from_detector,to_detector,q_up_veh,q_down_veh,detected_on_veh,detected_off_veh,net_residual_veh,inferred_missing_exit_veh,inferred_missing_entry_veh,missing_exit_vph,missing_entry_vph,exit_split_fraction
0,16:00–18:00,Cell 1,501015153,501016013,5199.0,5590.0,0.0,0.0,391.0,0.0,391.0,0.0,195.5,0.000000
1,16:00–18:00,Cell 2,501016013,501016023,5590.0,5766.0,2453.0,0.0,-2277.0,2277.0,0.0,1138.5,0.0,0.283103
2,16:00–18:00,Cell 3,501016023,501016031,5766.0,5338.0,0.0,0.0,-428.0,428.0,0.0,214.0,0.0,0.074228
3,16:00–18:00,Cell 4,501016031,501016043,5338.0,5425.0,0.0,0.0,87.0,0.0,87.0,0.0,43.5,0.000000
4,16:00–18:00,Cell 5,501016043,501016053,5425.0,5868.0,0.0,270.0,713.0,0.0,713.0,0.0,356.5,0.000000
5,16:00–18:00,Cell 6,501016053,501016062,5868.0,5247.0,578.0,0.0,-1199.0,1199.0,0.0,599.5,0.0,0.186007
6,16:00–18:00,Cell 7,501016062,501016071,5247.0,4471.0,0.0,0.0,-776.0,776.0,0.0,388.0,0.0,0.147894
7,16:00–18:00,Cell 8,501016071,501016082,4471.0,3201.0,0.0,0.0,-1270.0,1270.0,0.0,635.0,0.0,0.284053
8,16:00–18:00,Cell 9,501016082,501016091,3201.0,4760.0,625.0,0.0,934.0,0.0,934.0,0.0,467.0,0.000000


Free-flow detector-balance calibration residuals


,audit_window,cell,from_detector,to_detector,q_up_veh,q_down_veh,detected_on_veh,detected_off_veh,net_residual_veh,inferred_missing_exit_veh,inferred_missing_entry_veh,missing_exit_vph,missing_entry_vph,exit_split_fraction
0,13:00–15:00,Cell 1,501015153,501016013,4205.0,4694.0,0.0,0.0,489.0,0.0,489.0,0.0,244.5,0.000000
1,13:00–15:00,Cell 2,501016013,501016023,4694.0,4838.0,1933.0,0.0,-1789.0,1789.0,0.0,894.5,0.0,0.269956
2,13:00–15:00,Cell 3,501016023,501016031,4838.0,4742.0,0.0,0.0,-96.0,96.0,0.0,48.0,0.0,0.019843
3,13:00–15:00,Cell 4,501016031,501016043,4742.0,4862.0,0.0,0.0,120.0,0.0,120.0,0.0,60.0,0.000000
4,13:00–15:00,Cell 5,501016043,501016053,4862.0,5475.0,0.0,368.0,981.0,0.0,981.0,0.0,490.5,0.000000
5,13:00–15:00,Cell 6,501016053,501016062,5475.0,5170.0,305.0,0.0,-610.0,610.0,0.0,305.0,0.0,0.105536
6,13:00–15:00,Cell 7,501016062,501016071,5170.0,4759.0,0.0,0.0,-411.0,411.0,0.0,205.5,0.0,0.079497
7,13:00–15:00,Cell 8,501016071,501016082,4759.0,4704.0,0.0,0.0,-55.0,55.0,0.0,27.5,0.0,0.011557
8,13:00–15:00,Cell 9,501016082,501016091,4704.0,4856.0,443.0,0.0,-291.0,291.0,0.0,145.5,0.0,0.056538


Free-flow-calibrated undetected ramp model


,cell,raw_exit_split_fraction,calibrated_exit_split_fraction,raw_undetected_entry_vph,calibrated_undetected_entry_vph,undetected_entry_per_15sec
0,Cell 1,0.000000,0.000000,244.5,183.375,0.764062
1,Cell 2,0.269956,0.404934,0.0,0.000,0.000000
2,Cell 3,0.019843,0.029764,0.0,0.000,0.000000
3,Cell 4,0.000000,0.000000,60.0,45.000,0.187500
4,Cell 5,0.000000,0.000000,490.5,367.875,1.532812
5,Cell 6,0.105536,0.158304,0.0,0.000,0.000000
6,Cell 7,0.079497,0.119246,0.0,0.000,0.000000
7,Cell 8,0.011557,0.017336,0.0,0.000,0.000000
8,Cell 9,0.056538,0.084807,0.0,0.000,0.000000


Created CTM input series with undetected exits modeled as split fractions.
q_in_boundary_series: 480
u_in_series: 480
f_out_series: 480
residual_inflow_series: 480
residual_outflow_series: 480


In [61]:
# CTM INPUT SERIES CHECK
required_input_series_names = [
    "q_in_boundary_series",
    "u_in_series",
    "f_out_series",
    "observed_release_series",
    "ramp_arrival_series",
]

missing_input_series_names = [
    name
    for name in required_input_series_names
    if name not in globals()
]

if len(missing_input_series_names) > 0:
    raise NameError(
        "Missing CTM input series variables: "
        + ", ".join(missing_input_series_names)
        + ". Run the CTM input-series builder cell before this check cell."
    )

input_series_length_check_df = pd.DataFrame({
    "series_name": [
        "q_in_boundary_series",
        "u_in_series",
        "f_out_series",
        "u_4th observed_release",
        "u_price observed_release",
        "u_mattie observed_release",
        "u_avila observed_release",
        "u_4th ramp_arrival",
        "u_price ramp_arrival",
        "u_mattie ramp_arrival",
        "u_avila ramp_arrival",
    ],
    "length": [
        len(q_in_boundary_series),
        len(u_in_series),
        len(f_out_series),
        len(observed_release_series["u_4th"]),
        len(observed_release_series["u_price"]),
        len(observed_release_series["u_mattie"]),
        len(observed_release_series["u_avila"]),
        len(ramp_arrival_series["u_4th"]),
        len(ramp_arrival_series["u_price"]),
        len(ramp_arrival_series["u_mattie"]),
        len(ramp_arrival_series["u_avila"]),
    ],
})

display(input_series_length_check_df)

bad_lengths_df = input_series_length_check_df[
    input_series_length_check_df["length"] != num_steps
].copy()

if len(bad_lengths_df) > 0:
    print("BAD SERIES LENGTHS")
    display(bad_lengths_df)

    raise ValueError(
        f"Some CTM input series do not have num_steps = {num_steps}."
    )

first_step_input_df = pd.DataFrame({
    "input_type": [
        "boundary inflow",

        "observed release",
        "observed release",
        "observed release",
        "observed release",

        "ramp arrival",
        "ramp arrival",
        "ramp arrival",
        "ramp arrival",

        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
    ],
    "location": [
        "Cell 1 boundary",

        "4TH ST ON",
        "PRICE ST ON",
        "MATTIE RD ON",
        "AVILA BEACH ON",

        "4TH ST ON",
        "PRICE ST ON",
        "MATTIE RD ON",
        "AVILA BEACH ON",

        "Cell 1",
        "Cell 2",
        "Cell 3",
        "Cell 4",
        "Cell 5",
        "Cell 6",
        "Cell 7",
        "Cell 8",
        "Cell 9",
    ],
    "first_15sec_value": [
        q_in_boundary_series[0],

        observed_release_series["u_4th"][0],
        observed_release_series["u_price"][0],
        observed_release_series["u_mattie"][0],
        observed_release_series["u_avila"][0],

        ramp_arrival_series["u_4th"][0],
        ramp_arrival_series["u_price"][0],
        ramp_arrival_series["u_mattie"][0],
        ramp_arrival_series["u_avila"][0],

        f_out_series[0]["Cell 1"],
        f_out_series[0]["Cell 2"],
        f_out_series[0]["Cell 3"],
        f_out_series[0]["Cell 4"],
        f_out_series[0]["Cell 5"],
        f_out_series[0]["Cell 6"],
        f_out_series[0]["Cell 7"],
        f_out_series[0]["Cell 8"],
        f_out_series[0]["Cell 9"],
    ],
})

print(f"{num_steps}-Step CTM Input Series Length Check")
display(input_series_length_check_df)

print("First 15-Second CTM Input Values")
display(first_step_input_df.round(3))

print("PASS: all CTM input series have correct length:", num_steps)

,series_name,length
0,q_in_boundary_series,480
1,u_in_series,480
2,f_out_series,480
3,u_4th observed_release,480
4,u_price observed_release,480
5,u_mattie observed_release,480
6,u_avila observed_release,480
7,u_4th ramp_arrival,480
8,u_price ramp_arrival,480
9,u_mattie ramp_arrival,480


480-Step CTM Input Series Length Check


,series_name,length
0,q_in_boundary_series,480
1,u_in_series,480
2,f_out_series,480
3,u_4th observed_release,480
4,u_price observed_release,480
5,u_mattie observed_release,480
6,u_avila observed_release,480
7,u_4th ramp_arrival,480
8,u_price ramp_arrival,480
9,u_mattie ramp_arrival,480


First 15-Second CTM Input Values


,input_type,location,first_15sec_value
0,boundary inflow,Cell 1 boundary,10.05
1,observed release,4TH ST ON,1.15
2,observed release,PRICE ST ON,2.50
3,observed release,MATTIE RD ON,2.55
4,observed release,AVILA BEACH ON,0.65
5,ramp arrival,4TH ST ON,1.38
6,ramp arrival,PRICE ST ON,3.00
7,ramp arrival,MATTIE RD ON,3.06
8,ramp arrival,AVILA BEACH ON,0.78
9,off-ramp flow,Cell 1,0.00


PASS: all CTM input series have correct length: 480


In [62]:
# RAMP MAXIMUM QUEUE CAPACITY

ave_veh_length = 25  # feet per vehicle
meter_to_feet = 3.28084

ramp_length_m = {
    "4TH ST ON": 154.38,
    "PRICE ST ON": 334.97,
    "MATTIE RD ON": 186.19,
    "AVILA BEACH ON": 438.94,
}

ramp_lane_count = {
    "4TH ST ON": 1,
    "PRICE ST ON": 1,
    "MATTIE RD ON": 1,
    "AVILA BEACH ON": 1,
}

ramp_length_ft = {
    ramp_name: length_m * meter_to_feet
    for ramp_name, length_m in ramp_length_m.items()
}

ramp_max_queue_named = {
    ramp_name: (
        ramp_length_ft[ramp_name]
        * ramp_lane_count[ramp_name]
        / ave_veh_length
    )
    for ramp_name in ramp_length_ft
}

ramp_max_queue_by_u = {
    "u_4th": ramp_max_queue_named["4TH ST ON"],
    "u_price": ramp_max_queue_named["PRICE ST ON"],
    "u_mattie": ramp_max_queue_named["MATTIE RD ON"],
    "u_avila": ramp_max_queue_named["AVILA BEACH ON"],
}

ramp_capacity_df = pd.DataFrame([
    {
        "ramp": ramp_name,
        "ramp_length_m": ramp_length_m[ramp_name],
        "ramp_length_ft": ramp_length_ft[ramp_name],
        "lanes": ramp_lane_count[ramp_name],
        "max_queue_vehicles": ramp_max_queue_named[ramp_name],
    }
    for ramp_name in ramp_length_m
])

print("Ramp Maximum Queue Capacity")
display(ramp_capacity_df.round(3))

assert set(ramp_max_queue_by_u.keys()) == set(ramp_ids)
assert all(ramp_max_queue_by_u[ramp] > 0 for ramp in ramp_ids)

print("PASS: ramp maximum queue capacities built successfully.")

Ramp Maximum Queue Capacity


,ramp,ramp_length_m,ramp_length_ft,lanes,max_queue_vehicles
0,4TH ST ON,154.38,506.496,1,20.260
1,PRICE ST ON,334.97,1098.983,1,43.959
2,MATTIE RD ON,186.19,610.860,1,24.434
3,AVILA BEACH ON,438.94,1440.092,1,57.604


PASS: ramp maximum queue capacities built successfully.


In [63]:
# FAIRNESS PENALTY SETUP
gamma = 1.0

def fairness_penalty_one_step(
    R_next,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
):


    stress_dict = {}

    for u_name, R_t in R_next.items():

        if u_name not in ramp_name_map:
            raise KeyError(f"{u_name} missing from ramp_name_map")

        ramp_name = ramp_name_map[u_name]

        if ramp_name not in ramp_max_queue_named:
            raise KeyError(f"{ramp_name} missing from ramp_max_queue_named")

        R_max_i = float(ramp_max_queue_named[ramp_name])

        if R_max_i <= 0:
            raise ValueError(f"Invalid R_max for {ramp_name}: {R_max_i}")

        raw_stress = float(R_t) / R_max_i
        capped_stress = min(max(raw_stress, 0.0), 1.0)

        stress_dict[ramp_name] = capped_stress

    ramps = list(stress_dict.keys())
    fairness_sum = 0.0

    for i in range(len(ramps)):
        for j in range(i + 1, len(ramps)):
            phi_i = stress_dict[ramps[i]]
            phi_j = stress_dict[ramps[j]]

            fairness_sum += (phi_i - phi_j) ** 2

    L_fair = gamma * fairness_sum

    return stress_dict, fairness_sum, L_fair

In [64]:
# Fairness penalty test
R_next_test = {
    "u_4th": 5.0,
    "u_price": 10.0,
    "u_mattie": 3.0,
    "u_avila": 20.0,
}

stress_dict, fairness_sum, L_fair = fairness_penalty_one_step(
    R_next_test,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
)

print("Stress:")
print(stress_dict)

print("Fairness sum:", fairness_sum)
print("L_fair:", L_fair)

Stress:
{'4TH ST ON': 0.24679361821997695, 'PRICE ST ON': 0.22748305090485735, 'MATTIE RD ON': 0.12277780368698654, 'AVILA BEACH ON': 0.34720006179250046}
Fairness sum: 0.1014949755992006
L_fair: 0.1014949755992006


In [65]:
# RAMP QUEUE UPDATE WITH SPILLBACK
def ramp_next_queue_with_spillback(
    R_current,
    B_current,
    ramp_arrival_step,
    commanded_release_step,
    ramp_max_queue_by_u
):
    R_next = {}
    B_next = {}
    spillback_by_ramp = {}
    actual_release_step = {}

    for ramp in R_current:

        if ramp not in ramp_arrival_step:
            raise KeyError(f"{ramp} missing from ramp_arrival_step")

        if ramp not in commanded_release_step:
            raise KeyError(f"{ramp} missing from commanded_release_step")

        if ramp not in ramp_max_queue_by_u:
            raise KeyError(f"{ramp} missing from ramp_max_queue_by_u")

        R_now = float(R_current[ramp])
        B_now = float(B_current.get(ramp, 0.0))
        arrival = float(ramp_arrival_step[ramp])
        commanded_release = float(commanded_release_step[ramp])
        R_max = float(ramp_max_queue_by_u[ramp])

        if R_max <= 0:
            raise ValueError(f"Invalid R_max for {ramp}: {R_max}")

        available = (
            max(0.0, R_now)
            + max(0.0, B_now)
            + max(0.0, arrival)
        )

        actual_release = min(
            max(0.0, commanded_release),
            available
        )

        waiting_after_release = max(
            0.0,
            available - actual_release
        )

        R_next[ramp] = min(
            waiting_after_release,
            R_max
        )

        B_next[ramp] = max(
            0.0,
            waiting_after_release - R_max
        )

        spillback_by_ramp[ramp] = B_next[ramp]
        actual_release_step[ramp] = actual_release

    return R_next, B_next, spillback_by_ramp, actual_release_step

In [66]:
# RAMP QUEUE + FAIRNESS FIRST-STEP CHECK
fairness_test_step = 0

ramp_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

external_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

R_current_test = ramp_queue_0.copy()
B_current_test = external_queue_0.copy()

ramp_arrival_step_test = {
    ramp: ramp_arrival_series[ramp][fairness_test_step]
    for ramp in ramp_ids
}

commanded_release_step_test = {
    ramp: observed_release_series[ramp][fairness_test_step]
    for ramp in ramp_ids
}

(
    R_next_test,
    B_next_test,
    spillback_by_ramp_test,
    actual_release_step_test,
) = ramp_next_queue_with_spillback(
    R_current_test,
    B_current_test,
    ramp_arrival_step_test,
    commanded_release_step_test,
    ramp_max_queue_by_u
)

stress_dict_test, fairness_sum_test, L_fair_test = fairness_penalty_one_step(
    R_next_test,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
)

fairness_check_rows = []

for u_name in ramp_ids:
    ramp_name = ramp_name_map[u_name]
    R_max_i = ramp_max_queue_named[ramp_name]

    raw_stress = R_next_test[u_name] / R_max_i
    capped_stress = min(max(raw_stress, 0.0), 1.0)

    fairness_check_rows.append({
        "u_name": u_name,
        "ramp_name": ramp_name,
        "R_prev": R_current_test[u_name],
        "B_prev": B_current_test[u_name],
        "arrival": ramp_arrival_step_test[u_name],
        "commanded_release": commanded_release_step_test[u_name],
        "actual_release": actual_release_step_test[u_name],
        "R_next": R_next_test[u_name],
        "B_next": B_next_test[u_name],
        "total_waiting_next": R_next_test[u_name] + B_next_test[u_name],
        "R_max": R_max_i,
        "raw_stress": raw_stress,
        "capped_stress": capped_stress,
        "spillback": spillback_by_ramp_test[u_name],
    })

fairness_check_df = pd.DataFrame(fairness_check_rows)

print("Ramp Queue + Fairness First-Step Check")
display(fairness_check_df.round(3))

print("gamma:", gamma)
print("fairness_sum:", round(fairness_sum_test, 3))
print("L_fair:", round(L_fair_test, 3))

Ramp Queue + Fairness First-Step Check


,u_name,ramp_name,R_prev,B_prev,arrival,commanded_release,actual_release,R_next,B_next,total_waiting_next,R_max,raw_stress,capped_stress,spillback
0,u_4th,4TH ST ON,0.0,0.0,1.38,1.15,1.15,0.23,0.0,0.23,20.260,0.011,0.011,0.0
1,u_price,PRICE ST ON,0.0,0.0,3.00,2.50,2.50,0.50,0.0,0.50,43.959,0.011,0.011,0.0
2,u_mattie,MATTIE RD ON,0.0,0.0,3.06,2.55,2.55,0.51,0.0,0.51,24.434,0.021,0.021,0.0
3,u_avila,AVILA BEACH ON,0.0,0.0,0.78,0.65,0.65,0.13,0.0,0.13,57.604,0.002,0.002,0.0


gamma: 1.0
fairness_sum: 0.001
L_fair: 0.001


In [67]:

# PHYSICAL STORAGE CAPACITY SETUP FOR 9-CELL CTM
jam_density = 190  # veh / mile / lane

physical_capacity_rows = []

for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    from_station = seg["from_station"]
    to_station = seg["to_station"]

    length_miles = station_pm[to_id] - station_pm[from_id]

    # Use downstream detector lane count as cell lane count
    lanes = station_lane_count[to_id]

    max_vehicles = jam_density * length_miles * lanes

    physical_capacity_rows.append({
        "cell": cell,
        "segment": seg["segment"],
        "from_station_id": from_id,
        "to_station_id": to_id,
        "from_station": from_station,
        "to_station": to_station,
        "length_miles": length_miles,
        "lanes": lanes,
        "jam_density_veh_mi_lane": jam_density,
        "max_vehicles": max_vehicles,
    })

physical_capacity_df = pd.DataFrame(physical_capacity_rows)

physical_capacity = {
    row["cell"]: float(row["max_vehicles"])
    for _, row in physical_capacity_df.iterrows()
}

cell_length_by_cell = {
    row["cell"]: float(row["length_miles"])
    for _, row in physical_capacity_df.iterrows()
}

cell_lane_count_by_cell = {
    row["cell"]: int(row["lanes"])
    for _, row in physical_capacity_df.iterrows()
}

expected_cells = [f"Cell {i}" for i in range(1, 10)]

assert list(physical_capacity.keys()) == expected_cells
assert list(cell_length_by_cell.keys()) == expected_cells
assert list(cell_lane_count_by_cell.keys()) == expected_cells
assert all(physical_capacity[cell] > 0 for cell in expected_cells)

print("Physical Storage Capacity Setup")
display(physical_capacity_df.round(3))

print("physical_capacity:")
for cell in expected_cells:
    print(cell, round(physical_capacity[cell], 3))

Physical Storage Capacity Setup


,cell,segment,from_station_id,to_station_id,from_station,to_station,length_miles,lanes,jam_density_veh_mi_lane,max_vehicles
0,Cell 1,S1,501015153,501016013,4TH ST,4TH ST ON AREA,0.515,2,190,195.70
1,Cell 2,S2,501016013,501016023,4TH ST ON AREA,PRICE ST,0.449,2,190,170.62
2,Cell 3,S3,501016023,501016031,PRICE ST,HINDS AVE,0.502,2,190,190.76
3,Cell 4,S4,501016031,501016043,HINDS AVE,BELLO ST,0.510,2,190,193.80
4,Cell 5,S5,501016043,501016053,BELLO ST,SHELL BEACH RD,0.737,2,190,280.06
5,Cell 6,S6,501016053,501016062,SHELL BEACH RD,MATTIE RD,0.345,2,190,131.10
6,Cell 7,S7,501016062,501016071,MATTIE RD,SPYGLASS DR,1.526,2,190,579.88
7,Cell 8,S8,501016071,501016082,SPYGLASS DR,AVILA BEACH DR,1.141,3,190,650.37
8,Cell 9,S9,501016082,501016091,AVILA BEACH DR,SAN LUIS BAY DR,1.057,2,190,401.66


physical_capacity:
Cell 1 195.7
Cell 2 170.62
Cell 3 190.76
Cell 4 193.8
Cell 5 280.06
Cell 6 131.1
Cell 7 579.88
Cell 8 650.37
Cell 9 401.66


In [68]:
# SAFE THRESHOLD CAPACITY
# Define soft capacity threshold for doorway/storage penalty.
eta = 0.7

safe_threshold_capacity = {
    cell: eta * physical_capacity[cell]
    for cell in cell_ids
}

safe_threshold_capacity_df = pd.DataFrame([
    {
        "cell": cell,
        "physical_capacity": physical_capacity[cell],
        "safe_threshold_capacity": safe_threshold_capacity[cell],
        "eta": eta,
    }
    for cell in cell_ids
])

display(safe_threshold_capacity_df.round(3))

assert list(safe_threshold_capacity.keys()) == cell_ids

,cell,physical_capacity,safe_threshold_capacity,eta
0,Cell 1,195.70,136.990,0.7
1,Cell 2,170.62,119.434,0.7
2,Cell 3,190.76,133.532,0.7
3,Cell 4,193.80,135.660,0.7
4,Cell 5,280.06,196.042,0.7
5,Cell 6,131.10,91.770,0.7
6,Cell 7,579.88,405.916,0.7
7,Cell 8,650.37,455.259,0.7
8,Cell 9,401.66,281.162,0.7


In [69]:
# OFFICIAL PISMO 9-CELL / 4-RAMP MODEL DEFINITIONS
cell_ids = [f"Cell {i}" for i in range(1, 10)]
segment_ids = [f"S{i}" for i in range(1, 10)]


ramp_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

external_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

assert len(cell_ids) == 9
assert len(segment_ids) == 9
assert len(ramp_ids) == 4
assert set(ramp_cell_map.keys()) == set(ramp_ids)
assert set(ramp_name_map.keys()) == set(ramp_ids)
assert all(cell in cell_ids for cell in ramp_cell_map.values())

print("Official model IDs loaded.")
print("Cells:", cell_ids)
print("Ramps:", ramp_ids)
print("Ramp-cell map:", ramp_cell_map)

Official model IDs loaded.
Cells: ['Cell 1', 'Cell 2', 'Cell 3', 'Cell 4', 'Cell 5', 'Cell 6', 'Cell 7', 'Cell 8', 'Cell 9']
Ramps: ['u_4th', 'u_price', 'u_mattie', 'u_avila']
Ramp-cell map: {'u_4th': 'Cell 2', 'u_price': 'Cell 2', 'u_mattie': 'Cell 6', 'u_avila': 'Cell 9'}


In [70]:
# CAPACITY PENALTY FUNCTION FOR 9-CELL / 15-SEC CTM
lambda_1 = 1.0   # doorway demand pressure penalty weight
lambda_2 = 0.5   # safe threshold penalty weight
lambda_3 = 1.0   # physical capacity penalty weight
lambda_4 = 0.5   # spillback penalty weight

def capacity_penalty_one_step(
    q_in,
    u_in_step,
    controllable_onramp_in_step,
    x_next,
    inflow_capacity,
    safe_threshold_capacity,
    physical_capacity,
    spillback_by_ramp,
    lambda_1,
    lambda_2,
    lambda_3,
    lambda_4
):
    doorway_penalty_by_cell = {}
    safe_threshold_penalty_by_cell = {}
    physical_capacity_penalty_by_cell = {}
    spillback_penalty_by_ramp = {}

    total_doorway_penalty = 0.0
    total_safe_threshold_penalty = 0.0
    total_physical_capacity_penalty = 0.0
    total_spillback_penalty = 0.0

    for cell in cell_ids:
        doorway_demand_pressure = (
            float(q_in.get(cell, 0.0))
            + float(controllable_onramp_in_step.get(cell, 0.0))
        )

        doorway_excess = max(
            0.0,
            doorway_demand_pressure - float(inflow_capacity[cell])
        )

        doorway_penalty = lambda_1 * doorway_excess ** 2

        doorway_penalty_by_cell[cell] = doorway_penalty
        total_doorway_penalty += doorway_penalty

        safe_excess = max(
            0.0,
            float(x_next[cell]) - float(safe_threshold_capacity[cell])
        )

        safe_penalty = lambda_2 * safe_excess ** 2

        safe_threshold_penalty_by_cell[cell] = safe_penalty
        total_safe_threshold_penalty += safe_penalty

        physical_excess = max(
            0.0,
            float(x_next[cell]) - float(physical_capacity[cell])
        )

        physical_penalty = lambda_3 * physical_excess ** 2

        physical_capacity_penalty_by_cell[cell] = physical_penalty
        total_physical_capacity_penalty += physical_penalty

    for ramp in ramp_ids:
        spillback_excess = max(
            0.0,
            float(spillback_by_ramp.get(ramp, 0.0))
        )

        spillback_penalty = lambda_4 * spillback_excess ** 2

        spillback_penalty_by_ramp[ramp] = spillback_penalty
        total_spillback_penalty += spillback_penalty

    total_capacity_penalty = (
        total_doorway_penalty
        + total_safe_threshold_penalty
        + total_physical_capacity_penalty
        + total_spillback_penalty
    )

    return {
        "doorway_penalty_by_cell": doorway_penalty_by_cell,
        "safe_threshold_penalty_by_cell": safe_threshold_penalty_by_cell,
        "physical_capacity_penalty_by_cell": physical_capacity_penalty_by_cell,
        "spillback_penalty_by_ramp": spillback_penalty_by_ramp,
        "total_doorway_penalty": total_doorway_penalty,
        "total_safe_threshold_penalty": total_safe_threshold_penalty,
        "total_physical_capacity_penalty": total_physical_capacity_penalty,
        "total_spillback_penalty": total_spillback_penalty,
        "total_capacity_penalty": total_capacity_penalty,
    }

In [71]:
# 15-SECOND CTM STEP WITH:
def ctm_15sec_step(
    x_current,
    q_in_boundary_step,
    u_in_step,
    f_out_step,
    avila_release_demand,
    inflow_capacity,
    outflow_capacity,
    physical_capacity,
    movement_factor_by_cell,
    wave_speed_ratio_by_cell,
    upstream_boundary_queue=0.0,
    exit_split_by_cell=None,
    merge_priority=None
):
    cells = cell_ids

    if exit_split_by_cell is None:
        exit_split_by_cell = {
            cell: 0.0
            for cell in cells
        }

    if merge_priority is None:
        merge_priority = MERGE_PRIORITY

    assert list(exit_split_by_cell.keys()) == cells

    sending_total = {}
    receiving = {}

    q_out = {}
    q_in = {}

    split_exit_flow = {}
    actual_fixed_f_out = {}
    actual_f_out = {}

    x_next = {}

    # 1. Sending and receiving
    for cell in cells:
        beta = float(exit_split_by_cell[cell])

        if beta < 0.0 or beta >= 1.0:
            raise ValueError(
                f"exit_split_by_cell[{cell}] must be in [0, 1). Got {beta}."
            )

        sending_total[cell] = min(
            float(movement_factor_by_cell[cell]) * float(x_current[cell]),
            float(outflow_capacity[cell])
        )

        available_storage = max(
            0.0,
            float(physical_capacity[cell]) - float(x_current[cell])
        )

        receiving[cell] = max(
            0.0,
            min(
                float(inflow_capacity[cell]),
                float(wave_speed_ratio_by_cell[cell]) * available_storage
            )
        )

    # 2. Mainline links except Cell 8 -> Cell 9 merge
    for i, cell in enumerate(cells):
        beta = float(exit_split_by_cell[cell])

        if cell == "Cell 8":
            continue

        if i < len(cells) - 1:
            downstream_cell = cells[i + 1]

            if 1.0 - beta <= 1e-12:
                total_leave = 0.0
            else:
                total_leave = min(
                    sending_total[cell],
                    receiving[downstream_cell] / (1.0 - beta)
                )

            q_out[cell] = (
                1.0
                - beta
            ) * total_leave

            split_exit_flow[cell] = (
                beta
                * total_leave
            )

        else:
            # Last modeled cell exits the corridor.
            total_leave = sending_total[cell]

            q_out[cell] = (
                1.0
                - beta
            ) * total_leave

            split_exit_flow[cell] = (
                beta
                * total_leave
            )

    # 3. Avila merge at Cell 9 entrance
    beta8 = float(exit_split_by_cell["Cell 8"])

    if beta8 < 0.0 or beta8 >= 1.0:
        raise ValueError(
            f"exit_split_by_cell['Cell 8'] must be in [0, 1). Got {beta8}."
        )

    cell8_total_sending = sending_total["Cell 8"]

    cell8_mainline_demand = (
        1.0
        - beta8
    ) * cell8_total_sending

    R9 = receiving["Cell 9"]

    avila_demand = max(
        0.0,
        float(avila_release_demand)
    )

    avila_capacity_share = max(
        float(merge_priority) * R9,
        R9 - cell8_mainline_demand
    )

    avila_capacity_share = max(
        0.0,
        min(
            R9,
            avila_capacity_share
        )
    )

    avila_accepted = min(
        avila_demand,
        avila_capacity_share
    )

    cell8_mainline_accepted = min(
        cell8_mainline_demand,
        max(
            0.0,
            R9 - avila_accepted
        )
    )

    q_out["Cell 8"] = cell8_mainline_accepted

    if 1.0 - beta8 <= 1e-12:
        cell8_total_leave = 0.0
    else:
        cell8_total_leave = (
            cell8_mainline_accepted
            / (1.0 - beta8)
        )

    cell8_total_leave = min(
        cell8_total_leave,
        cell8_total_sending
    )

    split_exit_flow["Cell 8"] = (
        beta8
        * cell8_total_leave
    )

    # 4. Boundary inflow with upstream boundary queue
    boundary_demand = (
        float(upstream_boundary_queue)
        + float(q_in_boundary_step)
    )

    q_in["Cell 1"] = min(
        boundary_demand,
        receiving["Cell 1"]
    )

    upstream_boundary_queue_next = (
        boundary_demand
        - q_in["Cell 1"]
    )

    upstream_boundary_delay = (
        (
            float(upstream_boundary_queue)
            + float(upstream_boundary_queue_next)
        )
        / 2.0
    ) * delta_t

    # 5. Internal mainline inflows
    for i in range(1, len(cells)):
        current_cell = cells[i]
        upstream_cell = cells[i - 1]

        q_in[current_cell] = q_out[upstream_cell]

    # 6. Cell state update
    for cell in cells:
        external_inflow = float(u_in_step[cell])

        if cell == merge_ramp_cell:
            external_inflow += avila_accepted

        available_before_fixed_off = max(
            0.0,
            float(x_current[cell])
            + float(q_in[cell])
            + external_inflow
            - float(q_out[cell])
            - float(split_exit_flow[cell])
        )

        actual_fixed_f_out[cell] = min(
            max(
                0.0,
                float(f_out_step[cell])
            ),
            available_before_fixed_off
        )

        actual_f_out[cell] = (
            split_exit_flow[cell]
            + actual_fixed_f_out[cell]
        )

        x_next[cell] = (
            float(x_current[cell])
            + float(q_in[cell])
            + external_inflow
            - float(q_out[cell])
            - float(split_exit_flow[cell])
            - float(actual_fixed_f_out[cell])
        )

        x_next[cell] = max(
            0.0,
            x_next[cell]
        )

    return (
        x_next,
        q_out,
        q_in,
        sending_total,
        receiving,
        actual_f_out,
        avila_accepted,
        upstream_boundary_queue_next,
        upstream_boundary_delay
    )

In [72]:
# OFFICIAL PHYSICAL MOVEMENT FACTORS
movement_factor_by_cell_official = {
    cell: min(1.0, delta_t / tt_ff_min[cell])
    for cell in cell_ids
}

backward_wave_speed_mph = 15.0

wave_speed_ratio_by_cell_official = {
    cell: min(
        1.0,
        backward_wave_speed_mph
        * (delta_t / 60.0)
        / cell_length_by_cell[cell]
    )
    for cell in cell_ids
}

movement_factor_check_df = pd.DataFrame([
    {
        "cell": cell,
        "tt_ff_min": tt_ff_min[cell],
        "delta_t_min": delta_t,
        "movement_factor": movement_factor_by_cell_official[cell],
        "wave_speed_ratio": wave_speed_ratio_by_cell_official[cell],
    }
    for cell in cell_ids
])

print("Official physical movement factors")
display(movement_factor_check_df.round(4))

assert list(movement_factor_by_cell_official.keys()) == cell_ids
assert list(wave_speed_ratio_by_cell_official.keys()) == cell_ids

assert all(
    0.0 < movement_factor_by_cell_official[cell] <= 1.0
    for cell in cell_ids
)

assert all(
    0.0 < wave_speed_ratio_by_cell_official[cell] <= 1.0
    for cell in cell_ids
)

print("PASS: official movement factors built successfully.")

Official physical movement factors


,cell,tt_ff_min,delta_t_min,movement_factor,wave_speed_ratio
0,Cell 1,0.4627,0.25,0.5403,0.1214
1,Cell 2,0.4037,0.25,0.6192,0.1392
2,Cell 3,0.4507,0.25,0.5547,0.1245
3,Cell 4,0.4574,0.25,0.5466,0.1225
4,Cell 5,0.6600,0.25,0.3788,0.0848
5,Cell 6,0.3088,0.25,0.8095,0.1812
6,Cell 7,1.3676,0.25,0.1828,0.0410
7,Cell 8,1.0038,0.25,0.2490,0.0548
8,Cell 9,0.9306,0.25,0.2686,0.0591


PASS: official movement factors built successfully.


In [73]:
# CTM FIRST-STEP STATE UPDATE CHECK

ctm_test_step = 0

# Start from empty mainline state for this basic unit test
ctm_test_x_current = {
    cell: 0.0
    for cell in cell_ids
}

# Official regular lateral inflow.
# This excludes Avila; Avila enters through the merge.
ctm_test_u_in_step = u_in_series[ctm_test_step]

ctm_test_f_out_step = f_out_series[ctm_test_step]

ctm_test_upstream_boundary_queue = 0.0

# Avila demand waiting to enter the Cell 9 merge
ctm_test_avila_release_demand = (
    float(ramp_queue_0[merge_ramp_id])
    + float(external_queue_0[merge_ramp_id])
    + float(ramp_arrival_series[merge_ramp_id][ctm_test_step])
)

# Run one CTM step using official dictionaries
(
    ctm_test_x_next,
    ctm_test_q_out,
    ctm_test_q_in,
    ctm_test_sending,
    ctm_test_receiving,
    ctm_test_actual_f_out,
    ctm_test_avila_accepted,
    ctm_test_upstream_boundary_queue_next,
    ctm_test_upstream_boundary_delay,
) = ctm_15sec_step(
    x_current=ctm_test_x_current,
    q_in_boundary_step=q_in_boundary_series[ctm_test_step],
    u_in_step=ctm_test_u_in_step,
    f_out_step=ctm_test_f_out_step,
    avila_release_demand=ctm_test_avila_release_demand,
    inflow_capacity=ctm_inflow_capacity_official,
    outflow_capacity=ctm_outflow_capacity_official,
    physical_capacity=physical_capacity,
    movement_factor_by_cell=movement_factor_by_cell_official,
    wave_speed_ratio_by_cell=wave_speed_ratio_by_cell_official,
    upstream_boundary_queue=ctm_test_upstream_boundary_queue,
    exit_split_by_cell=exit_split_by_cell,
    merge_priority=MERGE_PRIORITY
)

ctm_first_step_check_df = pd.DataFrame([
    {
        "cell": cell,
        "x_current": ctm_test_x_current[cell],
        "q_in": ctm_test_q_in[cell],
        "u_in_regular": ctm_test_u_in_step[cell],
        "avila_accepted_if_cell9": (
            ctm_test_avila_accepted
            if cell == merge_ramp_cell
            else 0.0
        ),
        "sending": ctm_test_sending[cell],
        "receiving": ctm_test_receiving[cell],
        "q_out": ctm_test_q_out[cell],
        "requested_f_out": ctm_test_f_out_step[cell],
        "actual_f_out": ctm_test_actual_f_out[cell],
        "f_out_capped_gap": (
            ctm_test_f_out_step[cell]
            - ctm_test_actual_f_out[cell]
        ),
        "x_next": ctm_test_x_next[cell],
    }
    for cell in cell_ids
])

print("CTM First-Step State Update Check")
display(ctm_first_step_check_df.round(3))

print(
    "Avila release demand:",
    round(ctm_test_avila_release_demand, 6),
    "veh"
)

print(
    "Avila accepted by merge:",
    round(ctm_test_avila_accepted, 6),
    "veh"
)

print(
    "Upstream boundary queue next:",
    round(ctm_test_upstream_boundary_queue_next, 6),
    "veh"
)

print(
    "Upstream boundary delay:",
    round(ctm_test_upstream_boundary_delay, 6),
    "veh-min"
)

# Mass-balance check over:
mass_before = (
    sum(ctm_test_x_current.values())
    + ctm_test_upstream_boundary_queue
)

mass_entering = (
    float(q_in_boundary_series[ctm_test_step])
    + sum(ctm_test_u_in_step.values())
    + float(ctm_test_avila_accepted)
)

mass_leaving = (
    ctm_test_q_out["Cell 9"]
    + sum(ctm_test_actual_f_out.values())
)

mass_after = (
    sum(ctm_test_x_next.values())
    + ctm_test_upstream_boundary_queue_next
)

mass_balance_error = (
    mass_before
    + mass_entering
    - mass_leaving
    - mass_after
)

print("Mass balance error:", round(mass_balance_error, 10))

assert abs(mass_balance_error) < 1e-8

print("PASS: CTM first-step official model mass balance holds.")

CTM First-Step State Update Check


,cell,x_current,q_in,u_in_regular,avila_accepted_if_cell9,sending,receiving,q_out,requested_f_out,actual_f_out,f_out_capped_gap,x_next
0,Cell 1,0.0,10.05,0.764,0.00,0.0,19.731,0.0,0.0,0.0,0.0,10.814
1,Cell 2,0.0,0.00,3.650,0.00,0.0,19.727,0.0,0.0,0.0,0.0,3.650
2,Cell 3,0.0,0.00,0.000,0.00,0.0,19.735,0.0,0.0,0.0,0.0,0.000
3,Cell 4,0.0,0.00,0.188,0.00,0.0,19.742,0.0,0.0,0.0,0.0,0.188
4,Cell 5,0.0,0.00,1.533,0.00,0.0,19.750,0.0,0.3,0.3,0.0,1.233
5,Cell 6,0.0,0.00,2.550,0.00,0.0,19.752,0.0,0.0,0.0,0.0,2.550
6,Cell 7,0.0,0.00,0.000,0.00,0.0,19.746,0.0,0.0,0.0,0.0,0.000
7,Cell 8,0.0,0.00,0.000,0.00,0.0,29.775,0.0,0.0,0.0,0.0,0.000
8,Cell 9,0.0,0.00,0.000,0.78,0.0,10.000,0.0,0.0,0.0,0.0,0.780


Avila release demand: 0.78 veh
Avila accepted by merge: 0.78 veh
Upstream boundary queue next: 0.0 veh
Upstream boundary delay: 0.0 veh-min
Mass balance error: 0.0
PASS: CTM first-step official model mass balance holds.


In [74]:
# BUILD STATION METADATA DICTIONARIES
from pathlib import Path
import pandas as pd

metadata_file_candidates = [
    Path("District_5_5min_station_data") / "d05_text_meta_2026_04_28.txt",
    Path("District_5_5min_station_data") / "d05_text_meta_2026_04_28(1).txt",
    Path("d05_text_meta_2026_04_28.txt"),
    Path("d05_text_meta_2026_04_28(1).txt"),
]

metadata_file_path = None

for candidate_path in metadata_file_candidates:
    if candidate_path.exists():
        metadata_file_path = candidate_path
        break

if metadata_file_path is None:
    raise FileNotFoundError(
        "Could not find PeMS metadata file. "
        "Expected d05_text_meta_2026_04_28.txt or d05_text_meta_2026_04_28(1).txt."
    )

print("Loading metadata file:", metadata_file_path)

meta_raw = pd.read_csv(
    metadata_file_path,
    sep="\t",
    header=None,
    dtype=str
)

meta_raw = meta_raw.iloc[:, :14].copy()

meta_raw.columns = [
    "station_id",
    "freeway",
    "direction",
    "district",
    "county",
    "city",
    "state_pm",
    "absolute_pm",
    "latitude",
    "longitude",
    "length",
    "station_type",
    "lanes",
    "station_name",
]

meta_raw["station_id"] = pd.to_numeric(
    meta_raw["station_id"],
    errors="coerce"
).astype("Int64")

meta_raw["absolute_pm"] = pd.to_numeric(
    meta_raw["absolute_pm"],
    errors="coerce"
)

meta_raw["lanes"] = pd.to_numeric(
    meta_raw["lanes"],
    errors="coerce"
)

meta_selected = meta_raw[
    meta_raw["station_id"].isin(ids_to_keep)
].copy()

if meta_selected.empty:
    raise ValueError(
        "No metadata rows found for ids_to_keep. "
        "Check ids_to_keep and metadata file."
    )

station_name_by_id = {
    int(row["station_id"]): str(row["station_name"])
    for _, row in meta_selected.dropna(
        subset=["station_id"]
    ).iterrows()
}

station_pm = {
    int(row["station_id"]): float(row["absolute_pm"])
    for _, row in meta_selected.dropna(
        subset=[
            "station_id",
            "absolute_pm",
        ]
    ).iterrows()
}

station_lane_count = {
    int(row["station_id"]): int(float(row["lanes"]))
    for _, row in meta_selected.dropna(
        subset=[
            "station_id",
            "lanes",
        ]
    ).iterrows()
}

station_type_by_id = {
    int(row["station_id"]): str(row["station_type"]).strip()
    for _, row in meta_selected.dropna(
        subset=[
            "station_id",
            "station_type",
        ]
    ).iterrows()
}

required_metadata_ids = set(ids_to_keep)

missing_name_ids = sorted(
    required_metadata_ids - set(station_name_by_id.keys())
)

missing_pm_ids = sorted(
    required_metadata_ids - set(station_pm.keys())
)

missing_lane_ids = sorted(
    required_metadata_ids - set(station_lane_count.keys())
)

if len(missing_name_ids) > 0:
    print("WARNING: missing station names:", missing_name_ids)

if len(missing_pm_ids) > 0:
    print("WARNING: missing absolute postmiles:", missing_pm_ids)

if len(missing_lane_ids) > 0:
    print("WARNING: missing lane counts:", missing_lane_ids)

print("Metadata dictionaries created.")
print("station_name_by_id:", len(station_name_by_id))
print("station_pm:", len(station_pm))
print("station_lane_count:", len(station_lane_count))
print("station_type_by_id:", len(station_type_by_id))

display(
    meta_selected[
        [
            "station_id",
            "station_name",
            "absolute_pm",
            "station_type",
            "lanes",
        ]
    ].sort_values("absolute_pm")
)

Loading metadata file: District_5_5min_station_data\d05_text_meta_2026_04_28.txt
Metadata dictionaries created.
station_name_by_id: 15
station_pm: 15
station_lane_count: 15
station_type_by_id: 15


,station_id,station_name,absolute_pm,station_type,lanes
340,501015153,4TH ST 101 NB EXIT VDS MLSB SB,188.738,ML,2.0
344,501016013,4TH ST 101 NB ON RAMP VDS MLSB S,189.253,ML,2.0
345,501016014,4TH ST 101 NB ON RAMP VDS ONSB S,189.254,OR,1.0
348,501016023,PRICE ST EXIT SIGN 101 NB VDS ML,189.702,ML,2.0
349,501016024,PRICE ST EXIT SIGN 101 NB VDS ON,189.703,OR,1.0
350,501016031,HINDS AVE 101 SB VDS MLSB SB,190.204,ML,2.0
354,501016043,BELLO ST 101 NB VDS MLSB SB,190.714,ML,2.0
355,501016044,BELLO ST 101 NB VDS OFSB SB,190.715,FR,1.0
358,501016053,SHELL BEACH RD 101 NB VDS MLSB S,191.451,ML,2.0
360,501016062,MATTIE RD 101 NB VDS MLSB SB,191.796,ML,2.0


In [75]:
# PEMS-DERIVED MAINLINE INITIAL STATE
def normalize_time_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, dt.time):
        return value

    if isinstance(value, pd.Timestamp):
        return value.time()

    parsed_value = pd.to_datetime(
        str(value),
        errors="coerce"
    )

    if pd.isna(parsed_value):
        raise ValueError(
            f"Could not parse time_of_day value: {value}"
        )

    return parsed_value.time()


typical_pm_profile = typical_pm_profile.copy()

typical_pm_profile["station_id"] = pd.to_numeric(
    typical_pm_profile["station_id"],
    errors="coerce"
).astype("Int64")

typical_pm_profile["time_of_day_normalized"] = typical_pm_profile[
    "time_of_day"
].apply(normalize_time_value)

if "station_type" in typical_pm_profile.columns:
    typical_pm_profile["station_type"] = (
        typical_pm_profile["station_type"]
        .astype(str)
        .str.strip()
    )


# 1. Find available mainline profile times
mainline_profile_rows = typical_pm_profile[
    typical_pm_profile["station_id"].isin(mainline_ids)
].copy()

if mainline_profile_rows.empty:
    raise ValueError(
        "No rows found in typical_pm_profile for mainline_ids. "
        "Check station_id type or mainline_ids."
    )

available_start_times = sorted(
    [
        t
        for t in mainline_profile_rows["time_of_day_normalized"].dropna().unique()
    ]
)

if len(available_start_times) == 0:
    raise ValueError(
        "No usable time_of_day values found for mainline rows."
    )

if benchmark_start_time not in available_start_times:
    print("Available mainline profile times:")
    print(available_start_times)

    raise ValueError(
        f"Official benchmark_start_time {benchmark_start_time} "
        f"is not available in typical_pm_profile."
    )

print(
    "Using official benchmark initial-state time:",
    benchmark_start_time
)

print("Available mainline profile times:")
print(available_start_times)


# 2. Extract mainline detector data at selected start time
mainline_16pm_state_data = typical_pm_profile[
    (
        typical_pm_profile["station_id"].isin(mainline_ids)
    )
    & (
        typical_pm_profile["time_of_day_normalized"] == benchmark_start_time
    )
].copy()

# Do not require station_type == ML here because mainline_ids already define ML detectors.

for col in [
    "median_flow_5min",
    "median_flow_vph",
    "median_speed",
]:
    mainline_16pm_state_data[col] = pd.to_numeric(
        mainline_16pm_state_data[col],
        errors="coerce"
    )


# 3. Add station metadata
mainline_16pm_state_data["station_name"] = (
    mainline_16pm_state_data["station_id"].map(station_name_by_id)
)

mainline_16pm_state_data["absolute_postmile"] = (
    mainline_16pm_state_data["station_id"].map(station_pm)
)

mainline_16pm_state_data["lanes"] = (
    mainline_16pm_state_data["station_id"].map(station_lane_count)
)

mainline_16pm_state_data = mainline_16pm_state_data.sort_values(
    "absolute_postmile"
).reset_index(drop=True)


# 4. Safety checks
if len(mainline_16pm_state_data) != len(mainline_ids):
    found_ids = set(
        mainline_16pm_state_data["station_id"].dropna().astype(int)
    )

    expected_ids = set(mainline_ids)

    missing_ids = sorted(
        expected_ids - found_ids
    )

    extra_ids = sorted(
        found_ids - expected_ids
    )

    print("Missing mainline IDs:", missing_ids)
    print("Extra mainline IDs:", extra_ids)

    debug_rows = typical_pm_profile[
        typical_pm_profile["station_id"].isin(mainline_ids)
    ][
        [
            "station_id",
            "time_of_day",
            "time_of_day_normalized",
        ]
    ].drop_duplicates().sort_values(
        [
            "station_id",
            "time_of_day_normalized",
        ]
    )

    print("Available rows for expected mainline IDs:")
    display(debug_rows.head(120))

    raise ValueError(
        f"Expected {len(mainline_ids)} mainline detectors at {benchmark_start_time}, "
        f"but found {len(mainline_16pm_state_data)}."
    )


required_cols = [
    "median_flow_5min",
    "median_flow_vph",
    "median_speed",
    "absolute_postmile",
    "lanes",
]

for col in required_cols:
    if mainline_16pm_state_data[col].isna().any():
        raise ValueError(
            f"Missing values in {col} for initial mainline data."
        )

if (
    mainline_16pm_state_data["median_speed"] <= 0
).any():
    raise ValueError(
        "Nonpositive median_speed found in initial mainline detector data."
    )


# 5. Convert flow/speed to detector density
mainline_16pm_state_data["density_veh_per_mile"] = (
    mainline_16pm_state_data["median_flow_vph"]
    / mainline_16pm_state_data["median_speed"]
)


# 6. Build 9 CTM cell geometry
cell_geometry_rows = []

for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    cell_start = float(station_pm[from_id])
    cell_end = float(station_pm[to_id])

    cell_midpoint = 0.5 * (
        cell_start
        + cell_end
    )

    cell_length_miles = (
        cell_end
        - cell_start
    )

    if cell_length_miles <= 0:
        raise ValueError(
            f"{cell} has nonpositive length: {cell_length_miles}"
        )

    cell_geometry_rows.append({
        "cell": cell,
        "segment": seg["segment"],
        "from_station_id": from_id,
        "to_station_id": to_id,
        "from_station": seg["from_station"],
        "to_station": seg["to_station"],
        "cell_start_postmile": cell_start,
        "cell_end_postmile": cell_end,
        "cell_midpoint_postmile": cell_midpoint,
        "cell_length_miles": cell_length_miles,
    })

mainline_cell_geometry_df = pd.DataFrame(
    cell_geometry_rows
)


# 7. Interpolate detector density to CTM cell midpoints
detector_postmiles = mainline_16pm_state_data[
    "absolute_postmile"
].to_numpy(dtype=float)

detector_densities = mainline_16pm_state_data[
    "density_veh_per_mile"
].to_numpy(dtype=float)

cell_midpoints = mainline_cell_geometry_df[
    "cell_midpoint_postmile"
].to_numpy(dtype=float)

mainline_cell_geometry_df["interpolated_density_veh_per_mile"] = np.interp(
    cell_midpoints,
    detector_postmiles,
    detector_densities
)

mainline_cell_geometry_df["initial_state_x0"] = (
    mainline_cell_geometry_df["interpolated_density_veh_per_mile"]
    * mainline_cell_geometry_df["cell_length_miles"]
)


# 8. Final benchmark initial state
mainline_initial_state = {
    row["cell"]: float(row["initial_state_x0"])
    for _, row in mainline_cell_geometry_df.iterrows()
}

ramp_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}


# 9. Display derivation tables
mainline_initial_detector_derivation_df = mainline_16pm_state_data[
    [
        "station_id",
        "station_name",
        "absolute_postmile",
        "lanes",
        "median_flow_5min",
        "median_flow_vph",
        "median_speed",
        "density_veh_per_mile",
    ]
].copy()

mainline_initial_state_derivation_df = mainline_cell_geometry_df[
    [
        "cell",
        "segment",
        "from_station",
        "to_station",
        "cell_start_postmile",
        "cell_end_postmile",
        "cell_midpoint_postmile",
        "cell_length_miles",
        "interpolated_density_veh_per_mile",
        "initial_state_x0",
    ]
].copy()

print("PeMS detector-level density derivation")
display(mainline_initial_detector_derivation_df.round(3))

print("PeMS-derived 9-cell initial mainline state")
display(mainline_initial_state_derivation_df.round(3))

print("mainline_initial_state:")
for cell in cell_ids:
    print(
        cell,
        "=",
        round(mainline_initial_state[cell], 3)
    )


# 10. Clean check tables
assert list(mainline_initial_state.keys()) == cell_ids
assert list(physical_capacity.keys()) == cell_ids
assert list(safe_threshold_capacity.keys()) == cell_ids

mainline_initial_state_df = pd.DataFrame([
    {
        "cell": cell,
        "initial_state_x0": mainline_initial_state[cell],
        "safe_threshold_capacity": safe_threshold_capacity[cell],
        "physical_capacity": physical_capacity[cell],
        "x0_over_safe_threshold": (
            mainline_initial_state[cell]
            / safe_threshold_capacity[cell]
        ),
        "x0_over_physical_capacity": (
            mainline_initial_state[cell]
            / physical_capacity[cell]
        ),
    }
    for cell in cell_ids
])

ramp_queue_0_df = pd.DataFrame([
    {
        "ramp": ramp,
        "initial_queue": ramp_queue_0[ramp],
        "max_queue": ramp_max_queue_by_u[ramp],
    }
    for ramp in ramp_ids
])

print("9-Cell Mainline Initial State")
display(mainline_initial_state_df.round(3))

print("Initial Ramp Queue")
display(ramp_queue_0_df.round(3))

Using official benchmark initial-state time: 16:00:00
Available mainline profile times:
[datetime.time(16, 0), datetime.time(16, 5), datetime.time(16, 10), datetime.time(16, 15), datetime.time(16, 20), datetime.time(16, 25), datetime.time(16, 30), datetime.time(16, 35), datetime.time(16, 40), datetime.time(16, 45), datetime.time(16, 50), datetime.time(16, 55), datetime.time(17, 0), datetime.time(17, 5), datetime.time(17, 10), datetime.time(17, 15), datetime.time(17, 20), datetime.time(17, 25), datetime.time(17, 30), datetime.time(17, 35), datetime.time(17, 40), datetime.time(17, 45), datetime.time(17, 50), datetime.time(17, 55)]
PeMS detector-level density derivation


,station_id,station_name,absolute_postmile,lanes,median_flow_5min,median_flow_vph,median_speed,density_veh_per_mile
0,501015153,4TH ST 101 NB EXIT VDS MLSB SB,188.738,2,201.0,2412.0,66.5,36.271
1,501016013,4TH ST 101 NB ON RAMP VDS MLSB S,189.253,2,198.0,2376.0,66.9,35.516
2,501016023,PRICE ST EXIT SIGN 101 NB VDS ML,189.702,2,203.0,2436.0,65.9,36.965
3,501016031,HINDS AVE 101 SB VDS MLSB SB,190.204,2,181.0,2172.0,68.9,31.524
4,501016043,BELLO ST 101 NB VDS MLSB SB,190.714,2,176.0,2112.0,67.2,31.429
5,501016053,SHELL BEACH RD 101 NB VDS MLSB S,191.451,2,188.0,2256.0,64.8,34.815
6,501016062,MATTIE RD 101 NB VDS MLSB SB,191.796,2,129.0,1548.0,63.2,24.494
7,501016071,SPYGLASS DR 101 SB VDS MLSB SB,193.322,2,98.0,1176.0,8.4,140.000
8,501016082,AVILA BEACH DR 101 NB VDS MLSB S,194.463,3,53.0,636.0,12.8,49.688
9,501016091,SAN LUIS BAY DR 101 SB VDS MLSB,195.520,2,222.0,2664.0,33.1,80.483


PeMS-derived 9-cell initial mainline state


,cell,segment,from_station,to_station,cell_start_postmile,cell_end_postmile,cell_midpoint_postmile,cell_length_miles,interpolated_density_veh_per_mile,initial_state_x0
0,Cell 1,S1,4TH ST,4TH ST ON AREA,188.738,189.253,188.996,0.515,35.893,18.485
1,Cell 2,S2,4TH ST ON AREA,PRICE ST,189.253,189.702,189.478,0.449,36.240,16.272
2,Cell 3,S3,PRICE ST,HINDS AVE,189.702,190.204,189.953,0.502,34.245,17.191
3,Cell 4,S4,HINDS AVE,BELLO ST,190.204,190.714,190.459,0.510,31.476,16.053
4,Cell 5,S5,BELLO ST,SHELL BEACH RD,190.714,191.451,191.082,0.737,33.122,24.411
5,Cell 6,S6,SHELL BEACH RD,MATTIE RD,191.451,191.796,191.623,0.345,29.654,10.231
6,Cell 7,S7,MATTIE RD,SPYGLASS DR,191.796,193.322,192.559,1.526,82.247,125.509
7,Cell 8,S8,SPYGLASS DR,AVILA BEACH DR,193.322,194.463,193.892,1.141,94.844,108.217
8,Cell 9,S9,AVILA BEACH DR,SAN LUIS BAY DR,194.463,195.520,194.992,1.057,65.085,68.795


mainline_initial_state:
Cell 1 = 18.485
Cell 2 = 16.272
Cell 3 = 17.191
Cell 4 = 16.053
Cell 5 = 24.411
Cell 6 = 10.231
Cell 7 = 125.509
Cell 8 = 108.217
Cell 9 = 68.795
9-Cell Mainline Initial State


,cell,initial_state_x0,safe_threshold_capacity,physical_capacity,x0_over_safe_threshold,x0_over_physical_capacity
0,Cell 1,18.485,136.990,195.70,0.135,0.094
1,Cell 2,16.272,119.434,170.62,0.136,0.095
2,Cell 3,17.191,133.532,190.76,0.129,0.090
3,Cell 4,16.053,135.660,193.80,0.118,0.083
4,Cell 5,24.411,196.042,280.06,0.125,0.087
5,Cell 6,10.231,91.770,131.10,0.111,0.078
6,Cell 7,125.509,405.916,579.88,0.309,0.216
7,Cell 8,108.217,455.259,650.37,0.238,0.166
8,Cell 9,68.795,281.162,401.66,0.245,0.171


Initial Ramp Queue


,ramp,initial_queue,max_queue
0,u_4th,0.0,20.260
1,u_price,0.0,43.959
2,u_mattie,0.0,24.434
3,u_avila,0.0,57.604


In [76]:
# CALCULATE 9-CELL FREE-FLOW TRAVEL TIME
tt_ff_rows = []
tt_ff_min = {}

for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"
    segment = seg["segment"]

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    from_station = seg["from_station"]
    to_station = seg["to_station"]

    start_postmile = station_pm[from_id]
    end_postmile = station_pm[to_id]

    cell_length_miles = end_postmile - start_postmile

    v_ff_series = segment_free_flow_df.loc[
        segment_free_flow_df["segment"] == segment,
        "v_ff_mph"
    ]

    if v_ff_series.empty:
        raise ValueError(f"Missing free-flow speed for segment {segment}")

    v_ff_mph = float(v_ff_series.iloc[0])

    if v_ff_mph <= 0:
        raise ValueError(f"Invalid free-flow speed for {segment}: {v_ff_mph}")

    TT_ff_hr = cell_length_miles / v_ff_mph
    TT_ff_min_value = TT_ff_hr * 60.0

    tt_ff_min[cell] = TT_ff_min_value

    tt_ff_rows.append({
        "cell": cell,
        "segment": segment,
        "from_station": from_station,
        "to_station": to_station,
        "start_postmile": start_postmile,
        "end_postmile": end_postmile,
        "cell_length_miles": cell_length_miles,
        "v_ff_mph": v_ff_mph,
        "TT_ff_hr": TT_ff_hr,
        "TT_ff_min": TT_ff_min_value,
    })

tt_ff_min_df = pd.DataFrame(tt_ff_rows)

assert list(tt_ff_min.keys()) == cell_ids
assert all(tt_ff_min[cell] > 0 for cell in cell_ids)

print("Calculated 9-Cell Free-Flow Travel Time")
display(tt_ff_min_df.round(5))

print("Calculated tt_ff_min dictionary:")
for cell in cell_ids:
    print(cell, ":", round(tt_ff_min[cell], 5))

print(
    "Total corridor free-flow travel time, min:",
    round(sum(tt_ff_min.values()), 5)
)

Calculated 9-Cell Free-Flow Travel Time


,cell,segment,from_station,to_station,start_postmile,end_postmile,cell_length_miles,v_ff_mph,TT_ff_hr,TT_ff_min
0,Cell 1,S1,4TH ST,4TH ST ON AREA,188.738,189.253,0.515,66.775,0.00771,0.46275
1,Cell 2,S2,4TH ST ON AREA,PRICE ST,189.253,189.702,0.449,66.725,0.00673,0.40375
2,Cell 3,S3,PRICE ST,HINDS AVE,189.702,190.204,0.502,66.825,0.00751,0.45073
3,Cell 4,S4,HINDS AVE,BELLO ST,190.204,190.714,0.510,66.900,0.00762,0.45740
4,Cell 5,S5,BELLO ST,SHELL BEACH RD,190.714,191.451,0.737,67.000,0.01100,0.66000
5,Cell 6,S6,SHELL BEACH RD,MATTIE RD,191.451,191.796,0.345,67.025,0.00515,0.30884
6,Cell 7,S7,MATTIE RD,SPYGLASS DR,191.796,193.322,1.526,66.950,0.02279,1.36759
7,Cell 8,S8,SPYGLASS DR,AVILA BEACH DR,193.322,194.463,1.141,68.199,0.01673,1.00383
8,Cell 9,S9,AVILA BEACH DR,SAN LUIS BAY DR,194.463,195.520,1.057,68.147,0.01551,0.93064


Calculated tt_ff_min dictionary:
Cell 1 : 0.46275
Cell 2 : 0.40375
Cell 3 : 0.45073
Cell 4 : 0.4574
Cell 5 : 0.66
Cell 6 : 0.30884
Cell 7 : 1.36759
Cell 8 : 1.00383
Cell 9 : 0.93064
Total corridor free-flow travel time, min: 6.04551


In [77]:
def build_controllable_onramp_in_from_actual_release(actual_release_step):

    controllable_onramp_in_step = {
        cell: 0.0
        for cell in cell_ids
    }

    for ramp in ramp_ids:
        if ramp not in actual_release_step:
            raise KeyError(f"{ramp} missing from actual_release_step")

        cell = ramp_cell_map[ramp]

        controllable_onramp_in_step[cell] += float(
            actual_release_step[ramp]
        )

    return controllable_onramp_in_step


def mainline_delay_one_step(
    x_now,
    x_next,
    q_out,
    actual_f_out,
    tt_ff_min,
    delta_t
):
    rows = []

    total_mainline_delay_raw = 0.0
    total_mainline_delay_clipped = 0.0

    for cell in cell_ids:
        if cell not in x_now:
            raise KeyError(f"{cell} missing from x_now")

        if cell not in x_next:
            raise KeyError(f"{cell} missing from x_next")

        if cell not in q_out:
            raise KeyError(f"{cell} missing from q_out")

        if cell not in actual_f_out:
            raise KeyError(f"{cell} missing from actual_f_out")

        if cell not in tt_ff_min:
            raise KeyError(f"{cell} missing from tt_ff_min")

        ttt = (
            (
                float(x_now[cell])
                + float(x_next[cell])
            )
            / 2.0
        ) * float(delta_t)

        ff_term = (
            float(q_out[cell])
            + float(actual_f_out[cell])
        ) * float(tt_ff_min[cell])

        delay_raw = (
            ttt
            - ff_term
        )

        delay_clipped = max(
            delay_raw,
            0.0
        )

        total_mainline_delay_raw += delay_raw

        total_mainline_delay_clipped += delay_clipped

        rows.append({
            "cell": cell,
            "x_now": x_now[cell],
            "x_next": x_next[cell],
            "q_out": q_out[cell],
            "actual_f_out": actual_f_out[cell],
            "TTT_veh_min": ttt,
            "free_flow_component": ff_term,
            "delay_raw": delay_raw,
            "delay_clipped": delay_clipped,
            "mainline_delay_veh_min": delay_clipped,
        })

    mainline_delay_df = pd.DataFrame(rows)

    return (
        mainline_delay_df,
        total_mainline_delay_raw,
        total_mainline_delay_clipped
    )

In [78]:
# FIRST-STEP CTM MAINLINE DELAY CHECK

test_step = 0

u_in_step_0 = u_in_series[test_step]
f_out_step_0 = f_out_series[test_step]

upstream_boundary_queue_0 = 0.0

avila_release_demand_0 = (
    float(ramp_queue_0[merge_ramp_id])
    + float(external_queue_0[merge_ramp_id])
    + float(ramp_arrival_series[merge_ramp_id][test_step])
)

(
    x_next_0,
    q_out_0,
    q_in_0,
    sending_0,
    receiving_0,
    actual_f_out_0,
    avila_accepted_0,
    upstream_boundary_queue_next_0,
    upstream_boundary_delay_0,
) = ctm_15sec_step(
    x_current=mainline_initial_state,
    q_in_boundary_step=q_in_boundary_series[test_step],
    u_in_step=u_in_step_0,
    f_out_step=f_out_step_0,
    avila_release_demand=avila_release_demand_0,
    inflow_capacity=ctm_inflow_capacity_official,
    outflow_capacity=ctm_outflow_capacity_official,
    physical_capacity=physical_capacity,
    movement_factor_by_cell=movement_factor_by_cell_official,
    wave_speed_ratio_by_cell=wave_speed_ratio_by_cell_official,
    upstream_boundary_queue=upstream_boundary_queue_0,
    exit_split_by_cell=exit_split_by_cell,
    merge_priority=MERGE_PRIORITY
)

(
    mainline_delay_df_0,
    total_mainline_delay_raw_0,
    total_mainline_delay_clipped_0,
) = mainline_delay_one_step(
    x_now=mainline_initial_state,
    x_next=x_next_0,
    q_out=q_out_0,
    actual_f_out=actual_f_out_0,
    tt_ff_min=tt_ff_min,
    delta_t=delta_t
)

use_clipped_mainline_delay = True
arrival_multiplier = 1.2

if use_clipped_mainline_delay:
    total_mainline_delay_0 = (
        total_mainline_delay_clipped_0
        + upstream_boundary_delay_0
    )
else:
    total_mainline_delay_0 = (
        total_mainline_delay_raw_0
        + upstream_boundary_delay_0
    )

print("First-Step CTM Mainline Delay Check")
display(mainline_delay_df_0.round(3))

print(
    "Avila release demand:",
    round(avila_release_demand_0, 3),
    "veh"
)

print(
    "Avila accepted by merge:",
    round(avila_accepted_0, 3),
    "veh"
)

print(
    "Raw first-step mainline delay:",
    round(total_mainline_delay_raw_0, 3),
    "veh-min"
)

print(
    "Clipped first-step mainline delay:",
    round(total_mainline_delay_clipped_0, 3),
    "veh-min"
)

print(
    "Upstream boundary delay:",
    round(upstream_boundary_delay_0, 3),
    "veh-min"
)

print(
    "Total first-step mainline delay used:",
    round(total_mainline_delay_0, 3),
    "veh-min"
)

print(
    "Upstream boundary queue next:",
    round(upstream_boundary_queue_next_0, 3),
    "veh"
)

print("PASS: first-step delay diagnostic used official Avila-merge CTM.")

First-Step CTM Mainline Delay Check


,cell,x_now,x_next,q_out,actual_f_out,TTT_veh_min,free_flow_component,delay_raw,delay_clipped,mainline_delay_veh_min
0,Cell 1,18.485,19.313,9.987,0.000,4.725,4.621,0.103,0.103,0.103
1,Cell 2,16.272,19.833,5.996,4.080,4.513,4.068,0.445,0.445,0.445
2,Cell 3,17.191,13.651,9.251,0.284,3.855,4.298,-0.442,0.000,0.000
3,Cell 4,16.053,16.718,8.774,0.000,4.096,4.013,0.083,0.083,0.083
4,Cell 5,24.411,25.171,9.246,0.300,6.198,6.301,-0.103,0.000,0.000
5,Cell 6,10.231,13.746,6.971,1.311,2.997,2.558,0.439,0.439,0.439
6,Cell 7,125.509,112.733,17.391,2.355,29.780,27.004,2.776,2.776,2.776
7,Cell 8,108.217,116.225,9.220,0.163,28.055,9.419,18.637,18.637,18.637
8,Cell 9,68.795,68.795,9.152,0.848,17.199,9.306,7.892,7.892,7.892


Avila release demand: 0.78 veh
Avila accepted by merge: 0.78 veh
Raw first-step mainline delay: 29.831 veh-min
Clipped first-step mainline delay: 30.376 veh-min
Upstream boundary delay: 0.0 veh-min
Total first-step mainline delay used: 30.376 veh-min
Upstream boundary queue next: 0.0 veh
PASS: first-step delay diagnostic used official Avila-merge CTM.


In [79]:
# LOCAL RAMP DELAY CALCULATION FOR ONE CTM STEP
def ramp_delay_with_cap(
    R_current,
    R_next,
    B_current,
    B_next,
    ramp_arrival_step,
    commanded_release_step,
    actual_release_step,
    delta_t
):
    rows = []
    total_local_delay = 0.0

    for ramp in ramp_ids:

        required_dicts = {
            "R_current": R_current,
            "R_next": R_next,
            "ramp_arrival_step": ramp_arrival_step,
            "commanded_release_step": commanded_release_step,
            "actual_release_step": actual_release_step,
        }

        for name, dictionary in required_dicts.items():
            if ramp not in dictionary:
                raise KeyError(f"{ramp} missing from {name}")

        waiting_current = (
            float(R_current[ramp])
            + float(B_current.get(ramp, 0.0))
        )

        waiting_next = (
            float(R_next[ramp])
            + float(B_next.get(ramp, 0.0))
        )

        waiting_avg = (waiting_current + waiting_next) / 2.0

        local_delay = waiting_avg * delta_t

        total_local_delay += local_delay

        rows.append({
            "ramp": ramp,
            "R_current": R_current[ramp],
            "B_current": B_current.get(ramp, 0.0),
            "arrival": ramp_arrival_step[ramp],
            "commanded_release": commanded_release_step[ramp],
            "actual_release": actual_release_step[ramp],
            "R_next": R_next[ramp],
            "B_next": B_next.get(ramp, 0.0),
            "waiting_current": waiting_current,
            "waiting_next": waiting_next,
            "waiting_avg": waiting_avg,
            "local_delay_veh_min": local_delay,
        })

    ramp_delay_df = pd.DataFrame(rows)

    return ramp_delay_df, total_local_delay

In [80]:
# FIRST-STEP LOCAL RAMP DELAY CHECK

test_step = 0

# Initial external spillback queue
if external_queue_0 is None:
    B_step_0 = {
        ramp: 0.0
        for ramp in ramp_ids
    }
else:
    B_step_0 = external_queue_0.copy()

# First-step observed ramp releases
observed_release_step_0 = {
    ramp: float(observed_release_series[ramp][test_step])
    for ramp in ramp_ids
}

# First-step ramp arrivals
ramp_arrival_step_0 = {
    ramp: float(ramp_arrival_series[ramp][test_step])
    for ramp in ramp_ids
}

# Compute first-step ramp queue transition
(
    R_next_step_0,
    B_next_step_0,
    spillback_by_ramp_step_0,
    actual_release_step_0,
) = ramp_next_queue_with_spillback(
    R_current=ramp_queue_0,
    B_current=B_step_0,
    ramp_arrival_step=ramp_arrival_step_0,
    commanded_release_step=observed_release_step_0,
    ramp_max_queue_by_u=ramp_max_queue_by_u
)

# Compute first-step local ramp delay
ramp_delay_df_0, total_local_delay_0 = ramp_delay_with_cap(
    R_current=ramp_queue_0,
    R_next=R_next_step_0,
    B_current=B_step_0,
    B_next=B_next_step_0,
    ramp_arrival_step=ramp_arrival_step_0,
    commanded_release_step=observed_release_step_0,
    actual_release_step=actual_release_step_0,
    delta_t=delta_t
)

print("First-Step Local Ramp Delay Check")
display(ramp_delay_df_0.round(3))

print(
    "Total first-step local ramp delay:",
    round(total_local_delay_0, 3),
    "veh-min"
)

print("First-step ramp queue transition")
display(pd.DataFrame([
    {
        "ramp": ramp,
        "R_current": ramp_queue_0[ramp],
        "arrival": ramp_arrival_step_0[ramp],
        "commanded_release": observed_release_step_0[ramp],
        "actual_release": actual_release_step_0[ramp],
        "R_next": R_next_step_0[ramp],
        "B_current": B_step_0[ramp],
        "B_next": B_next_step_0[ramp],
        "spillback": spillback_by_ramp_step_0[ramp],
    }
    for ramp in ramp_ids
]).round(3))

First-Step Local Ramp Delay Check


,ramp,R_current,B_current,arrival,commanded_release,actual_release,R_next,B_next,waiting_current,waiting_next,waiting_avg,local_delay_veh_min
0,u_4th,0.0,0.0,1.38,1.15,1.15,0.23,0.0,0.0,0.23,0.115,0.029
1,u_price,0.0,0.0,3.00,2.50,2.50,0.50,0.0,0.0,0.50,0.250,0.062
2,u_mattie,0.0,0.0,3.06,2.55,2.55,0.51,0.0,0.0,0.51,0.255,0.064
3,u_avila,0.0,0.0,0.78,0.65,0.65,0.13,0.0,0.0,0.13,0.065,0.016


Total first-step local ramp delay: 0.171 veh-min
First-step ramp queue transition


,ramp,R_current,arrival,commanded_release,actual_release,R_next,B_current,B_next,spillback
0,u_4th,0.0,1.38,1.15,1.15,0.23,0.0,0.0,0.0
1,u_price,0.0,3.00,2.50,2.50,0.50,0.0,0.0,0.0
2,u_mattie,0.0,3.06,2.55,2.55,0.51,0.0,0.0,0.0
3,u_avila,0.0,0.78,0.65,0.65,0.13,0.0,0.0,0.0


In [81]:
# OFFICIAL 480-STEP STATE-BASED CTM BENCHMARK SIMULATION
def simulate_state_based_benchmark_480_steps(
    num_steps,
    mainline_initial_state,
    ramp_queue_0,
    q_in_boundary_series,
    observed_release_series,
    ramp_arrival_series,
    u_in_series,
    f_out_series,
    residual_inflow_series,
    residual_outflow_series,
    inflow_capacity,
    outflow_capacity,
    physical_capacity,
    safe_threshold_capacity,
    ramp_name_map,
    ramp_max_queue_named,
    ramp_max_queue_by_u,
    tt_ff_min,
    delta_t,
    gamma,
    lambda_1,
    lambda_2,
    lambda_3,
    lambda_4,
    movement_factor_by_cell,
    wave_speed_ratio_by_cell,
    exit_split_by_cell,
    external_queue_0=None,
    use_clipped_mainline_delay=True
):

    # 0. Sanity checks
    assert num_steps == 480, (
        f"Official two-hour benchmark must have 480 steps, got {num_steps}."
    )

    assert list(mainline_initial_state.keys()) == cell_ids
    assert list(inflow_capacity.keys()) == cell_ids
    assert list(outflow_capacity.keys()) == cell_ids
    assert list(physical_capacity.keys()) == cell_ids
    assert list(safe_threshold_capacity.keys()) == cell_ids
    assert list(tt_ff_min.keys()) == cell_ids
    assert list(movement_factor_by_cell.keys()) == cell_ids
    assert list(wave_speed_ratio_by_cell.keys()) == cell_ids
    assert list(exit_split_by_cell.keys()) == cell_ids

    assert set(ramp_queue_0.keys()) == set(ramp_ids)
    assert set(ramp_max_queue_by_u.keys()) == set(ramp_ids)
    assert set(ramp_name_map.keys()) == set(ramp_ids)

    assert merge_ramp_id == "u_avila"
    assert ramp_cell_map[merge_ramp_id] == merge_ramp_cell
    assert merge_ramp_id not in generic_ramp_cell_map

    assert len(q_in_boundary_series) == num_steps
    assert len(u_in_series) == num_steps
    assert len(f_out_series) == num_steps
    assert len(residual_inflow_series) == num_steps
    assert len(residual_outflow_series) == num_steps

    for ramp in ramp_ids:
        assert len(observed_release_series[ramp]) == num_steps
        assert len(ramp_arrival_series[ramp]) == num_steps

    for step in range(num_steps):
        assert list(u_in_series[step].keys()) == cell_ids
        assert list(f_out_series[step].keys()) == cell_ids
        assert list(residual_inflow_series[step].keys()) == cell_ids
        assert list(residual_outflow_series[step].keys()) == cell_ids

        # Avila must not be embedded in ordinary u_in_series.
        assert abs(
            float(u_in_series[step][merge_ramp_cell])
            - float(undetected_entry_per_15sec_by_cell[merge_ramp_cell])
        ) < 1e-9

    # 1. Initial states
    x_current = mainline_initial_state.copy()
    R_current = ramp_queue_0.copy()
    upstream_boundary_queue_current = 0.0

    if external_queue_0 is None:
        B_current = {
            ramp: 0.0
            for ramp in ramp_ids
        }
    else:
        assert set(external_queue_0.keys()) == set(ramp_ids)
        B_current = external_queue_0.copy()

    # 2. History container
    history = {
        "step": [],
        "x": [],
        "R": [],
        "B": [],
        "q_in": [],
        "q_out": [],
        "u_in": [],
        "f_out": [],
        "residual_inflow": [],
        "residual_outflow": [],
        "actual_f_out": [],
        "observed_release": [],
        "actual_release": [],
        "ramp_arrival": [],
        "spillback": [],
        "controllable_onramp_in": [],

        # Avila diagnostics
        "avila_available_demand": [],
        "avila_commanded_release": [],
        "avila_release_request": [],
        "avila_release_demand": [],   # legacy alias; stores request to merge
        "avila_accepted": [],

        "upstream_boundary_queue": [],
        "upstream_boundary_delay": [],
        "mainline_delay": [],
        "mainline_delay_raw": [],
        "mainline_delay_clipped": [],
        "local_delay": [],
        "fairness_penalty": [],
        "doorway_penalty": [],
        "safe_penalty": [],
        "physical_penalty": [],
        "spillback_penalty": [],
        "capacity_penalty": [],
        "total_objective": [],
    }

    # 3. Simulation loop
    for step in range(num_steps):

        q_in_boundary_step = float(
            q_in_boundary_series[step]
        )

        observed_release_step = {
            ramp: float(observed_release_series[ramp][step])
            for ramp in ramp_ids
        }

        ramp_arrival_step = {
            ramp: float(ramp_arrival_series[ramp][step])
            for ramp in ramp_ids
        }

        u_in_step = u_in_series[step]
        f_out_step = f_out_series[step]
        residual_inflow_step = residual_inflow_series[step]
        residual_outflow_step = residual_outflow_series[step]

        # Ordinary ramp queue update.
        # Avila is excluded here because its actual release is determined
        # by the Cell 9 merge acceptance inside ctm_15sec_step.
        generic_ramp_arrival_step = ramp_arrival_step.copy()
        generic_commanded_release_step = observed_release_step.copy()

        generic_ramp_arrival_step[merge_ramp_id] = 0.0
        generic_commanded_release_step[merge_ramp_id] = 0.0

        (
            R_next,
            B_next,
            spillback_by_ramp,
            actual_release_step,
        ) = ramp_next_queue_with_spillback(
            R_current=R_current,
            B_current=B_current,
            ramp_arrival_step=generic_ramp_arrival_step,
            commanded_release_step=generic_commanded_release_step,
            ramp_max_queue_by_u=ramp_max_queue_by_u
        )

        # Avila available demand waiting at the ramp.
        avila_available_demand = (
            float(R_current[merge_ramp_id])
            + float(B_current[merge_ramp_id])
            + float(ramp_arrival_step[merge_ramp_id])
        )

        # Historical-release stress baseline:
        # even if arrival demand is scaled up, Avila's commanded release
        # stays equal to the historical observed release.
        avila_commanded_release = float(
            observed_release_step[merge_ramp_id]
        )

        # This is the amount Avila is allowed to request from the Cell 9 merge.
        # It cannot exceed available demand, and it cannot exceed the
        # historical release command.
        avila_release_request = min(
            avila_available_demand,
            avila_commanded_release
        )

        # CTM mainline state update with Avila merge.
        # The 30% merge priority still applies inside ctm_15sec_step.
        (
            x_next,
            q_out,
            q_in,
            sending,
            receiving,
            actual_f_out,
            avila_accepted,
            upstream_boundary_queue_next,
            upstream_boundary_delay,
        ) = ctm_15sec_step(
            x_current=x_current,
            q_in_boundary_step=q_in_boundary_step,
            u_in_step=u_in_step,
            f_out_step=f_out_step,
            avila_release_demand=avila_release_request,
            inflow_capacity=inflow_capacity,
            outflow_capacity=outflow_capacity,
            physical_capacity=physical_capacity,
            movement_factor_by_cell=movement_factor_by_cell,
            wave_speed_ratio_by_cell=wave_speed_ratio_by_cell,
            upstream_boundary_queue=upstream_boundary_queue_current,
            exit_split_by_cell=exit_split_by_cell,
            merge_priority=MERGE_PRIORITY
        )

        # Update Avila queue using the actual merge-accepted release.
        # Leftover demand carries over to R/B.
        avila_remaining = max(
            0.0,
            avila_available_demand
            - float(avila_accepted)
        )

        avila_max_queue = float(
            ramp_max_queue_by_u[merge_ramp_id]
        )

        R_next[merge_ramp_id] = min(
            avila_remaining,
            avila_max_queue
        )

        B_next[merge_ramp_id] = max(
            0.0,
            avila_remaining
            - avila_max_queue
        )

        spillback_by_ramp[merge_ramp_id] = B_next[merge_ramp_id]

        actual_release_step[merge_ramp_id] = float(
            avila_accepted
        )

        # Controllable ramp inflow only for doorway/control penalty.
        # Includes Avila accepted by the merge.
        controllable_onramp_in_step = build_controllable_onramp_in_from_actual_release(
            actual_release_step
        )

        # Local ramp delay after final Avila queue/release correction.
        _, total_local_delay = ramp_delay_with_cap(
            R_current=R_current,
            R_next=R_next,
            B_current=B_current,
            B_next=B_next,
            ramp_arrival_step=ramp_arrival_step,
            commanded_release_step=observed_release_step,
            actual_release_step=actual_release_step,
            delta_t=delta_t
        )

        # Fairness penalty after final Avila queue correction.
        _, _, L_fair = fairness_penalty_one_step(
            R_next=R_next,
            ramp_name_map=ramp_name_map,
            ramp_max_queue_named=ramp_max_queue_named,
            gamma=gamma
        )

        # State-based mainline delay.
        (
            _,
            total_mainline_delay_raw,
            total_mainline_delay_clipped,
        ) = mainline_delay_one_step(
            x_now=x_current,
            x_next=x_next,
            q_out=q_out,
            actual_f_out=actual_f_out,
            tt_ff_min=tt_ff_min,
            delta_t=delta_t
        )

        if use_clipped_mainline_delay:
            total_mainline_delay = (
                total_mainline_delay_clipped
                + upstream_boundary_delay
            )
        else:
            total_mainline_delay = (
                total_mainline_delay_raw
                + upstream_boundary_delay
            )

        # Capacity penalties.
        capacity_info = capacity_penalty_one_step(
            q_in=q_in,
            u_in_step=u_in_step,
            controllable_onramp_in_step=controllable_onramp_in_step,
            x_next=x_next,
            inflow_capacity=inflow_capacity,
            safe_threshold_capacity=safe_threshold_capacity,
            physical_capacity=physical_capacity,
            spillback_by_ramp=spillback_by_ramp,
            lambda_1=lambda_1,
            lambda_2=lambda_2,
            lambda_3=lambda_3,
            lambda_4=lambda_4
        )

        # Total objective.
        total_objective = (
            total_mainline_delay
            + total_local_delay
            + L_fair
            + capacity_info["total_capacity_penalty"]
        )

        # Store results.
        history["step"].append(step)
        history["x"].append(x_next.copy())
        history["R"].append(R_next.copy())
        history["B"].append(B_next.copy())
        history["q_in"].append(q_in.copy())
        history["q_out"].append(q_out.copy())
        history["u_in"].append(u_in_step.copy())
        history["f_out"].append(f_out_step.copy())
        history["residual_inflow"].append(residual_inflow_step.copy())
        history["residual_outflow"].append(residual_outflow_step.copy())
        history["actual_f_out"].append(actual_f_out.copy())
        history["observed_release"].append(observed_release_step.copy())
        history["actual_release"].append(actual_release_step.copy())
        history["ramp_arrival"].append(ramp_arrival_step.copy())
        history["spillback"].append(spillback_by_ramp.copy())

        history["controllable_onramp_in"].append(
            controllable_onramp_in_step.copy()
        )

        history["avila_available_demand"].append(
            float(avila_available_demand)
        )

        history["avila_commanded_release"].append(
            float(avila_commanded_release)
        )

        history["avila_release_request"].append(
            float(avila_release_request)
        )

        # Legacy key for old diagnostics.
        # This now stores the amount Avila requested from the merge,
        # not the full available demand.
        history["avila_release_demand"].append(
            float(avila_release_request)
        )

        history["avila_accepted"].append(
            float(avila_accepted)
        )

        history["upstream_boundary_queue"].append(
            upstream_boundary_queue_next
        )

        history["upstream_boundary_delay"].append(
            upstream_boundary_delay
        )

        history["mainline_delay"].append(
            total_mainline_delay
        )

        history["mainline_delay_raw"].append(
            total_mainline_delay_raw
        )

        history["mainline_delay_clipped"].append(
            total_mainline_delay_clipped
        )

        history["local_delay"].append(
            total_local_delay
        )

        history["fairness_penalty"].append(
            L_fair
        )

        history["doorway_penalty"].append(
            capacity_info["total_doorway_penalty"]
        )

        history["safe_penalty"].append(
            capacity_info["total_safe_threshold_penalty"]
        )

        history["physical_penalty"].append(
            capacity_info["total_physical_capacity_penalty"]
        )

        history["spillback_penalty"].append(
            capacity_info["total_spillback_penalty"]
        )

        history["capacity_penalty"].append(
            capacity_info["total_capacity_penalty"]
        )

        history["total_objective"].append(
            total_objective
        )

        # Move to next step.
        x_current = x_next.copy()
        R_current = R_next.copy()
        B_current = B_next.copy()
        upstream_boundary_queue_current = upstream_boundary_queue_next

    # 4. Final states.
    history["R_final"] = R_current.copy()
    history["B_final"] = B_current.copy()
    history["x_final"] = x_current.copy()
    history["upstream_boundary_queue_final"] = upstream_boundary_queue_current

    assert len(history["step"]) == num_steps

    return history

In [82]:
# RUN OFFICIAL STATE-BASED CTM BENCHMARK SIMULATION
use_clipped_mainline_delay = True

if external_queue_0 is None:
    external_queue_0_run = {
        ramp: 0.0
        for ramp in ramp_ids
    }
else:
    external_queue_0_run = external_queue_0.copy()

state_based_benchmark_history = simulate_state_based_benchmark_480_steps(
    num_steps=num_steps,
    mainline_initial_state=mainline_initial_state,
    ramp_queue_0=ramp_queue_0,

    q_in_boundary_series=q_in_boundary_series,
    observed_release_series=observed_release_series,
    ramp_arrival_series=ramp_arrival_series,
    u_in_series=u_in_series,
    f_out_series=f_out_series,
    residual_inflow_series=residual_inflow_series,
    residual_outflow_series=residual_outflow_series,

    inflow_capacity=ctm_inflow_capacity_official,
    outflow_capacity=ctm_outflow_capacity_official,
    physical_capacity=physical_capacity,
    safe_threshold_capacity=safe_threshold_capacity,

    ramp_name_map=ramp_name_map,
    ramp_max_queue_named=ramp_max_queue_named,
    ramp_max_queue_by_u=ramp_max_queue_by_u,

    tt_ff_min=tt_ff_min,
    delta_t=delta_t,

    gamma=gamma,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    lambda_4=lambda_4,
    movement_factor_by_cell=movement_factor_by_cell_official,
    wave_speed_ratio_by_cell=wave_speed_ratio_by_cell_official,
    exit_split_by_cell=exit_split_by_cell,
    external_queue_0=external_queue_0_run,
    use_clipped_mainline_delay=use_clipped_mainline_delay
)

print("Benchmark simulation finished.")
print("Number of steps:", len(state_based_benchmark_history["step"]))

assert len(state_based_benchmark_history["step"]) == num_steps
print(f"PASS: official {num_steps}-step benchmark simulation completed.")

Benchmark simulation finished.
Number of steps: 480
PASS: official 480-step benchmark simulation completed.


In [83]:
# RAMP MASS-CONSERVATION DIAGNOSTIC
num_history_steps = len(state_based_benchmark_history["actual_release"])

# Use existing external_queue_0 if already defined.
# Do not redefine it unless missing.
if "external_queue_0" not in globals():
    external_queue_0 = {
        ramp: 0.0
        for ramp in ramp_ids
    }

total_initial_R = sum(
    float(ramp_queue_0[ramp])
    for ramp in ramp_ids
)

total_initial_B = sum(
    float(external_queue_0[ramp])
    for ramp in ramp_ids
)

total_arrivals = sum(
    float(ramp_arrival_series[ramp][step])
    for ramp in ramp_ids
    for step in range(num_history_steps)
)

total_actual_release = sum(
    float(state_based_benchmark_history["actual_release"][step][ramp])
    for ramp in ramp_ids
    for step in range(num_history_steps)
)

total_final_R = sum(
    float(state_based_benchmark_history["R_final"][ramp])
    for ramp in ramp_ids
)

total_final_B = sum(
    float(state_based_benchmark_history["B_final"][ramp])
    for ramp in ramp_ids
)

ramp_mass_residual = (
    total_initial_R
    + total_initial_B
    + total_arrivals
    - total_actual_release
    - total_final_R
    - total_final_B
)

ramp_mass_check_df = pd.DataFrame({
    "metric": [
        "initial_physical_ramp_queue",
        "initial_external_spillback_queue",
        "total_arrivals",
        "total_actual_release",
        "final_physical_ramp_queue",
        "final_external_spillback_queue",
        "mass_residual",
    ],
    "value": [
        total_initial_R,
        total_initial_B,
        total_arrivals,
        total_actual_release,
        total_final_R,
        total_final_B,
        ramp_mass_residual,
    ],
})

print("Ramp Mass-Conservation Check")
display(ramp_mass_check_df.round(8))

if abs(ramp_mass_residual) < 1e-8:
    print("PASS: ramp demand is conserved.")
else:
    print("FAIL: ramp demand is not conserved.")

Ramp Mass-Conservation Check


,metric,value
0,initial_physical_ramp_queue,0.000000
1,initial_external_spillback_queue,0.000000
2,total_arrivals,4387.200000
3,total_actual_release,3656.000000
4,final_physical_ramp_queue,146.257223
5,final_external_spillback_queue,584.942777
6,mass_residual,-0.000000


PASS: ramp demand is conserved.


In [84]:
# STATE-BASED CTM BENCHMARK SUMMARY
state_based_benchmark_summary_df = pd.DataFrame({
    "metric": [
        "Mainline Delay",
        "Local Ramp Delay",
        "Fairness Penalty",
        "Doorway Penalty",
        "Safe Threshold Penalty",
        "Physical Capacity Penalty",
        "Spillback Penalty",
        "Total Capacity Penalty",
        "Raw Total Objective",
    ],
    "value": [
        sum(state_based_benchmark_history["mainline_delay"]),
        sum(state_based_benchmark_history["local_delay"]),
        sum(state_based_benchmark_history["fairness_penalty"]),
        sum(state_based_benchmark_history["doorway_penalty"]),
        sum(state_based_benchmark_history["safe_penalty"]),
        sum(state_based_benchmark_history["physical_penalty"]),
        sum(state_based_benchmark_history["spillback_penalty"]),
        sum(state_based_benchmark_history["capacity_penalty"]),
        sum(state_based_benchmark_history["total_objective"]),
    ],
    "unit": [
        "veh-min",
        "veh-min",
        "penalty",
        "penalty",
        "penalty",
        "penalty",
        "penalty",
        "penalty",
        "raw objective",
    ],
})

print(
    f"State-Based CTM Benchmark: "
    f"9-cell / 15-sec / {benchmark_window_label}"
)
display(state_based_benchmark_summary_df.round(3))


# FINAL STATES
final_x = state_based_benchmark_history["x_final"]
final_R = state_based_benchmark_history["R_final"]
final_B = state_based_benchmark_history["B_final"]

final_x_df = pd.DataFrame([
    {
        "cell": cell,
        "final_x": final_x[cell],
        "safe_threshold_capacity": safe_threshold_capacity[cell],
        "physical_capacity": physical_capacity[cell],
        "x_over_safe_threshold": final_x[cell] / safe_threshold_capacity[cell],
        "x_over_physical_capacity": final_x[cell] / physical_capacity[cell],
    }
    for cell in cell_ids
])

final_R_df = pd.DataFrame([
    {
        "ramp": ramp,
        "final_physical_queue_R": final_R[ramp],
        "final_external_spillback_B": final_B[ramp],
        "final_total_waiting_R_plus_B": final_R[ramp] + final_B[ramp],
        "max_physical_queue": ramp_max_queue_by_u[ramp],
        "R_over_max_queue": final_R[ramp] / ramp_max_queue_by_u[ramp],
    }
    for ramp in ramp_ids
])

print(f"Final Mainline State After {num_steps} Steps")
display(final_x_df.round(3))

print(f"Final Ramp Queue and External Spillback After {num_steps} Steps")
display(final_R_df.round(3))


# HISTORY LENGTH CHECK
history_series_names = [
    "mainline_delay",
    "local_delay",
    "fairness_penalty",
    "doorway_penalty",
    "safe_penalty",
    "physical_penalty",
    "spillback_penalty",
    "capacity_penalty",
    "total_objective",
    "x",
    "R",
    "B",
    "actual_release",
]

history_length_check_df = pd.DataFrame([
    {
        "history_series": name,
        "length": len(state_based_benchmark_history[name]),
        "expected_length": num_steps,
        "pass": len(state_based_benchmark_history[name]) == num_steps,
    }
    for name in history_series_names
])

print("History Length Check")
display(history_length_check_df)

assert history_length_check_df["pass"].all(), (
    "Some history series do not have num_steps entries."
)

State-Based CTM Benchmark: 9-cell / 15-sec / 16:00–18:00


,metric,value,unit
0,Mainline Delay,25799.028,veh-min
1,Local Ramp Delay,44657.000,veh-min
2,Fairness Penalty,143.624,penalty
3,Doorway Penalty,0.104,penalty
4,Safe Threshold Penalty,0.000,penalty
5,Physical Capacity Penalty,0.000,penalty
6,Spillback Penalty,7849384.087,penalty
7,Total Capacity Penalty,7849384.191,penalty
8,Raw Total Objective,7919983.843,raw objective


Final Mainline State After 480 Steps


,cell,final_x,safe_threshold_capacity,physical_capacity,x_over_safe_threshold,x_over_physical_capacity
0,Cell 1,18.443,136.990,195.70,0.135,0.094
1,Cell 2,22.956,119.434,170.62,0.192,0.135
2,Cell 3,15.250,133.532,190.76,0.114,0.080
3,Cell 4,15.358,135.660,193.80,0.113,0.079
4,Cell 5,24.637,196.042,280.06,0.126,0.088
5,Cell 6,12.337,91.770,131.10,0.134,0.094
6,Cell 7,46.917,405.916,579.88,0.116,0.081
7,Cell 8,198.692,455.259,650.37,0.436,0.306
8,Cell 9,68.795,281.162,401.66,0.245,0.171


Final Ramp Queue and External Spillback After 480 Steps


,ramp,final_physical_queue_R,final_external_spillback_B,final_total_waiting_R_plus_B,max_physical_queue,R_over_max_queue
0,u_4th,20.260,142.940,163.2,20.260,1.0
1,u_price,43.959,283.441,327.4,43.959,1.0
2,u_mattie,24.434,91.166,115.6,24.434,1.0
3,u_avila,57.604,67.396,125.0,57.604,1.0


History Length Check


,history_series,length,expected_length,pass
0,mainline_delay,480,480,True
1,local_delay,480,480,True
2,fairness_penalty,480,480,True
3,doorway_penalty,480,480,True
4,safe_penalty,480,480,True
5,physical_penalty,480,480,True
6,spillback_penalty,480,480,True
7,capacity_penalty,480,480,True
8,total_objective,480,480,True
9,x,480,480,True


In [85]:
# OBSERVED TARGET-TIME CTM STATE DIAGNOSTIC
import datetime

def normalize_time_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, datetime.time):
        return value

    if isinstance(value, pd.Timestamp):
        return value.time()

    if isinstance(value, pd.Timedelta):
        total_seconds = int(value.total_seconds()) % (24 * 3600)
        return datetime.time(
            hour=total_seconds // 3600,
            minute=(total_seconds % 3600) // 60,
            second=total_seconds % 60
        )

    parsed_value = pd.to_datetime(
        str(value),
        errors="coerce"
    )

    if pd.isna(parsed_value):
        raise ValueError(f"Could not parse time_of_day value: {value}")

    return parsed_value.time()


def time_to_seconds(t):
    return (
        t.hour * 3600
        + t.minute * 60
        + t.second
    )


target_time_requested = None
typical_pm_profile = typical_pm_profile.copy()

typical_pm_profile["station_id"] = pd.to_numeric(
    typical_pm_profile["station_id"],
    errors="coerce"
).astype("Int64")

typical_pm_profile["time_of_day_normalized"] = typical_pm_profile[
    "time_of_day"
].apply(normalize_time_value)

if "station_type" in typical_pm_profile.columns:
    typical_pm_profile["station_type"] = (
        typical_pm_profile["station_type"]
        .astype(str)
        .str.strip()
    )


# 1. Find mainline rows
mainline_target_pool = typical_pm_profile[
    typical_pm_profile["station_id"].isin(mainline_ids)
].copy()

if mainline_target_pool.empty:
    raise ValueError(
        "No rows found for mainline_ids in typical_pm_profile."
    )


# 2. Find available times with all expected mainline detectors
coverage_by_time = (
    mainline_target_pool
    .dropna(subset=["time_of_day_normalized"])
    .groupby("time_of_day_normalized")["station_id"]
    .nunique()
    .reset_index(name="num_mainline_detectors")
)

full_coverage_times = coverage_by_time[
    coverage_by_time["num_mainline_detectors"] == len(mainline_ids)
]["time_of_day_normalized"].tolist()

if len(full_coverage_times) == 0:
    print("Available time coverage:")
    display(coverage_by_time.sort_values("time_of_day_normalized"))

    raise ValueError(
        "No time has all expected mainline detectors. "
        "Check mainline_ids or typical_pm_profile construction."
    )


# 3. Use the last full-coverage time in the benchmark profile
# For a 16:00–18:00 benchmark, this should usually be 17:55.

if target_time_requested is None:
    target_time_used = max(full_coverage_times)
else:
    if target_time_requested in full_coverage_times:
        target_time_used = target_time_requested
    else:
        target_seconds = time_to_seconds(target_time_requested)

        target_time_used = min(
            full_coverage_times,
            key=lambda t: abs(time_to_seconds(t) - target_seconds)
        )

        print(
            "WARNING: requested target time was not found. "
            "Using nearest full-coverage time:",
            target_time_used
        )

print("Observed CTM diagnostic target time used:", target_time_used)


# 4. Extract detector data at selected target time
mainline_target_state_data = mainline_target_pool[
    mainline_target_pool["time_of_day_normalized"] == target_time_used
].copy()

for col in [
    "median_flow_vph",
    "median_speed",
]:
    mainline_target_state_data[col] = pd.to_numeric(
        mainline_target_state_data[col],
        errors="coerce"
    )

mainline_target_state_data["absolute_postmile"] = (
    mainline_target_state_data["station_id"].map(station_pm)
)

mainline_target_state_data = mainline_target_state_data.sort_values(
    "absolute_postmile"
).reset_index(drop=True)


# 5. Safety checks
if len(mainline_target_state_data) != len(mainline_ids):
    found_ids = set(
        mainline_target_state_data["station_id"].dropna().astype(int)
    )

    missing_ids = sorted(
        set(mainline_ids) - found_ids
    )

    print("Missing mainline IDs:", missing_ids)

    display(
        mainline_target_state_data[
            [
                "station_id",
                "time_of_day",
                "time_of_day_normalized",
                "median_flow_vph",
                "median_speed",
                "absolute_postmile",
            ]
        ]
    )

    raise ValueError(
        f"Expected {len(mainline_ids)} mainline detectors at {target_time_used}, "
        f"but found {len(mainline_target_state_data)}."
    )

if mainline_target_state_data["median_flow_vph"].isna().any():
    raise ValueError("Missing median_flow_vph at target time.")

if mainline_target_state_data["median_speed"].isna().any():
    raise ValueError("Missing median_speed at target time.")

if mainline_target_state_data["absolute_postmile"].isna().any():
    raise ValueError("Missing absolute_postmile at target time.")

if (
    mainline_target_state_data["median_speed"] <= 0
).any():
    raise ValueError("Nonpositive median_speed at target time.")


# 6. Convert flow/speed to density
mainline_target_state_data["density_veh_per_mile"] = (
    mainline_target_state_data["median_flow_vph"]
    / mainline_target_state_data["median_speed"]
)


# 7. Prepare interpolation arrays
detector_postmiles = mainline_target_state_data[
    "absolute_postmile"
].to_numpy(dtype=float)

detector_densities = mainline_target_state_data[
    "density_veh_per_mile"
].to_numpy(dtype=float)

if len(detector_postmiles) == 0:
    raise ValueError(
        "No detector postmiles available for interpolation."
    )


# 8. Interpolate detector density to CTM cell midpoints
target_rows = []

for i, seg in enumerate(mainline_segments, start=1):
    cell = f"Cell {i}"

    from_id = seg["from_id"]
    to_id = seg["to_id"]

    cell_start = float(station_pm[from_id])
    cell_end = float(station_pm[to_id])

    cell_midpoint = 0.5 * (
        cell_start
        + cell_end
    )

    cell_length = (
        cell_end
        - cell_start
    )

    if cell_length <= 0:
        raise ValueError(
            f"{cell} has nonpositive length: {cell_length}"
        )

    density = float(
        np.interp(
            cell_midpoint,
            detector_postmiles,
            detector_densities
        )
    )

    x_observed = (
        density
        * cell_length
    )

    target_rows.append({
        "cell": cell,
        "segment": seg["segment"],
        "from_station": seg["from_station"],
        "to_station": seg["to_station"],
        "cell_midpoint_postmile": cell_midpoint,
        "cell_length_miles": cell_length,
        "observed_density_veh_per_mile": density,
        "observed_x_target": x_observed,
    })

observed_target_state_df = pd.DataFrame(target_rows)

print("Observed CTM State Diagnostic")
display(observed_target_state_df.round(3))

print("Detector-level target-time data")
display(
    mainline_target_state_data[
        [
            "station_id",
            "time_of_day",
            "time_of_day_normalized",
            "absolute_postmile",
            "median_flow_vph",
            "median_speed",
            "density_veh_per_mile",
        ]
    ].round(3)
)

Observed CTM diagnostic target time used: 17:55:00
Observed CTM State Diagnostic


,cell,segment,from_station,to_station,cell_midpoint_postmile,cell_length_miles,observed_density_veh_per_mile,observed_x_target
0,Cell 1,S1,4TH ST,4TH ST ON AREA,188.996,0.515,52.094,26.828
1,Cell 2,S2,4TH ST ON AREA,PRICE ST,189.478,0.449,49.620,22.279
2,Cell 3,S3,PRICE ST,HINDS AVE,189.953,0.502,48.677,24.436
3,Cell 4,S4,HINDS AVE,BELLO ST,190.459,0.510,47.899,24.429
4,Cell 5,S5,BELLO ST,SHELL BEACH RD,191.082,0.737,54.242,39.976
5,Cell 6,S6,SHELL BEACH RD,MATTIE RD,191.623,0.345,57.728,19.916
6,Cell 7,S7,MATTIE RD,SPYGLASS DR,192.559,1.526,87.834,134.035
7,Cell 8,S8,SPYGLASS DR,AVILA BEACH DR,193.892,1.141,110.481,126.059
8,Cell 9,S9,AVILA BEACH DR,SAN LUIS BAY DR,194.992,1.057,68.742,72.660


Detector-level target-time data


,station_id,time_of_day,time_of_day_normalized,absolute_postmile,median_flow_vph,median_speed,density_veh_per_mile
0,501015153,17:55:00,17:55:00,188.738,2208.0,40.0,55.200
1,501016013,17:55:00,17:55:00,189.253,2856.0,58.3,48.988
2,501016023,17:55:00,17:55:00,189.702,3000.0,59.7,50.251
3,501016031,17:55:00,17:55:00,190.204,3024.0,64.2,47.103
4,501016043,17:55:00,17:55:00,190.714,3024.0,62.1,48.696
5,501016053,17:55:00,17:55:00,191.451,3396.0,56.8,59.789
6,501016062,17:55:00,17:55:00,191.796,3084.0,55.4,55.668
7,501016071,17:55:00,17:55:00,193.322,2736.0,22.8,120.000
8,501016082,17:55:00,17:55:00,194.463,2100.0,20.8,100.962
9,501016091,17:55:00,17:55:00,195.520,2268.0,62.1,36.522


The CTM validation at 17:55 shows a total absolute state error of 227.995 vehicles. Most of this error comes from Cells 7 and 8. Cell 7 is under-estimated by 81.538 vehicles, while Cell 8 is over-estimated by 100.975 vehicles. This indicates that the model places too much congestion in Cell 8 instead of spreading the queue upstream into Cell 7.

However, the total vehicle mass across Cells 7–9 is close to the observed total. The model simulates 348.325 vehicles across Cells 7–9, while the observed value is 332.754 vehicles, a difference of only 15.571 vehicles. Therefore, the main issue is spatial queue placement, not total mass conservation.

Cell 9 is well matched after applying the 2400 vph inflow/outflow bottleneck cap. The simulated Cell 9 state is 68.795 vehicles, compared with the observed value of 72.660 vehicles, giving an error of only -3.865 vehicles.

Overall, the validation confirms that the Cell 9 bottleneck correction improved the downstream state, while the remaining error is mainly due to queue distribution between Cells 7 and 8.

In [86]:
# COMPARE OBSERVED TARGET STATE TO MATCHING SIMULATED CTM STATE

target_elapsed_minutes = (
    time_to_seconds(target_time_used)
    - time_to_seconds(benchmark_start_time)
) / 60.0

target_step_count = int(
    round(target_elapsed_minutes / delta_t)
)

target_step_index = target_step_count - 1

if target_step_index < 0 or target_step_index >= num_steps:
    raise ValueError(
        f"Target step index {target_step_index} is outside simulation range."
    )

simulated_target_state = state_based_benchmark_history["x"][
    target_step_index
]

observed_target_state_by_cell = {
    row["cell"]: float(row["observed_x_target"])
    for _, row in observed_target_state_df.iterrows()
}

state_validation_df = pd.DataFrame([
    {
        "cell": cell,
        "simulated_x": simulated_target_state[cell],
        "observed_x": observed_target_state_by_cell[cell],
        "sim_minus_observed": (
            simulated_target_state[cell]
            - observed_target_state_by_cell[cell]
        ),
        "abs_error": abs(
            simulated_target_state[cell]
            - observed_target_state_by_cell[cell]
        ),
    }
    for cell in cell_ids
])

print("CTM State Validation at Observed Target Time")
print("target_time_used:", target_time_used)
print("target_step_index:", target_step_index)

display(state_validation_df.round(3))

print(
    "Total absolute state error:",
    round(state_validation_df["abs_error"].sum(), 3),
    "veh"
)

CTM State Validation at Observed Target Time
target_time_used: 17:55:00
target_step_index: 459


,cell,simulated_x,observed_x,sim_minus_observed,abs_error
0,Cell 1,21.035,26.828,-5.794,5.794
1,Cell 2,26.024,22.279,3.745,3.745
2,Cell 3,17.288,24.436,-7.148,7.148
3,Cell 4,17.365,24.429,-7.064,7.064
4,Cell 5,27.914,39.976,-12.063,12.063
5,Cell 6,14.111,19.916,-5.805,5.805
6,Cell 7,52.497,134.035,-81.538,81.538
7,Cell 8,227.033,126.059,100.975,100.975
8,Cell 9,68.795,72.660,-3.865,3.865


Total absolute state error: 227.995 veh


In [87]:
# FULL BENCHMARK HISTORY SUMMARY
benchmark_step_summary_df = pd.DataFrame({
    "step": state_based_benchmark_history["step"],
    "mainline_delay": state_based_benchmark_history["mainline_delay"],
    "local_delay": state_based_benchmark_history["local_delay"],
    "fairness_penalty": state_based_benchmark_history["fairness_penalty"],
    "doorway_penalty": state_based_benchmark_history["doorway_penalty"],
    "safe_penalty": state_based_benchmark_history["safe_penalty"],
    "physical_penalty": state_based_benchmark_history["physical_penalty"],
    "spillback_penalty": state_based_benchmark_history["spillback_penalty"],
    "capacity_penalty": state_based_benchmark_history["capacity_penalty"],
    "total_objective": state_based_benchmark_history["total_objective"],

    "total_physical_ramp_queue_R": [
        sum(float(value) for value in step_R.values())
        for step_R in state_based_benchmark_history["R"]
    ],

    "total_external_spillback_B": [
        sum(float(value) for value in step_B.values())
        for step_B in state_based_benchmark_history["B"]
    ],

    "total_actual_release": [
        sum(float(value) for value in step_release.values())
        for step_release in state_based_benchmark_history["actual_release"]
    ],
})

print(f"Full {num_steps}-Step Benchmark Summary")
pd.set_option("display.max_rows", num_steps)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(benchmark_step_summary_df.round(3))

Full 480-Step Benchmark Summary


,step,mainline_delay,local_delay,fairness_penalty,doorway_penalty,safe_penalty,physical_penalty,spillback_penalty,capacity_penalty,total_objective,total_physical_ramp_queue_R,total_external_spillback_B,total_actual_release
0,0,30.227,0.171,0.001,0.000,0.0,0.0,0.000,0.000,30.399,1.370,0.000,6.85
1,1,28.742,0.514,0.003,0.000,0.0,0.0,0.000,0.000,29.258,2.740,0.000,6.85
2,2,30.468,0.856,0.006,0.000,0.0,0.0,0.000,0.000,31.331,4.110,0.000,6.85
3,3,31.906,1.199,0.011,0.000,0.0,0.0,0.000,0.000,33.116,5.480,0.000,6.85
4,4,33.141,1.541,0.017,0.000,0.0,0.0,0.000,0.000,34.699,6.850,0.000,6.85
5,5,34.122,1.884,0.025,0.000,0.0,0.0,0.000,0.000,36.031,8.220,0.000,6.85
6,6,34.896,2.226,0.034,0.000,0.0,0.0,0.000,0.000,37.156,9.590,0.000,6.85
7,7,35.500,2.569,0.044,0.000,0.0,0.0,0.000,0.000,38.113,10.960,0.000,6.85
8,8,35.969,2.911,0.056,0.000,0.0,0.0,0.000,0.000,38.936,12.330,0.000,6.85
9,9,36.315,3.254,0.069,0.000,0.0,0.0,0.000,0.000,39.638,13.700,0.000,6.85


In [88]:
# SAVE OFFICIAL BENCHMARK TOTALS
benchmark_mainline_delay = sum(state_based_benchmark_history["mainline_delay"])
benchmark_local_delay = sum(state_based_benchmark_history["local_delay"])
benchmark_fairness_penalty = sum(state_based_benchmark_history["fairness_penalty"])

benchmark_doorway_penalty = sum(state_based_benchmark_history["doorway_penalty"])
benchmark_safe_penalty = sum(state_based_benchmark_history["safe_penalty"])
benchmark_physical_penalty = sum(state_based_benchmark_history["physical_penalty"])
benchmark_spillback_penalty = sum(state_based_benchmark_history["spillback_penalty"])
benchmark_capacity_penalty = sum(state_based_benchmark_history["capacity_penalty"])

benchmark_raw_objective = sum(state_based_benchmark_history["total_objective"])

official_benchmark_totals = {
    "mainline_delay": benchmark_mainline_delay,
    "local_delay": benchmark_local_delay,
    "fairness_penalty": benchmark_fairness_penalty,
    "doorway_penalty": benchmark_doorway_penalty,
    "safe_penalty": benchmark_safe_penalty,
    "physical_penalty": benchmark_physical_penalty,
    "spillback_penalty": benchmark_spillback_penalty,
    "capacity_penalty": benchmark_capacity_penalty,
    "raw_objective": benchmark_raw_objective,
}

official_benchmark_totals_df = pd.DataFrame([
    {
        "metric": metric,
        "value": value,
    }
    for metric, value in official_benchmark_totals.items()
])

print("Official State-Based Benchmark Totals")
display(official_benchmark_totals_df.round(3))

Official State-Based Benchmark Totals


,metric,value
0,mainline_delay,25799.028
1,local_delay,44657.000
2,fairness_penalty,143.624
3,doorway_penalty,0.104
4,safe_penalty,0.000
5,physical_penalty,0.000
6,spillback_penalty,7849384.087
7,capacity_penalty,7849384.191
8,raw_objective,7919983.843


In [89]:
# BENCHMARK RAMP SERVICE / CONSERVATION METRICS
required_mass_vars = [
    "total_initial_R",
    "total_initial_B",
    "total_arrivals",
    "total_actual_release",
    "total_final_R",
    "total_final_B",
    "ramp_mass_residual",
]

missing_mass_vars = [
    name for name in required_mass_vars
    if name not in globals()
]

if missing_mass_vars:
    raise NameError(
        "Run ramp mass-conservation diagnostic before this block. "
        f"Missing: {missing_mass_vars}"
    )

benchmark_service_metrics = {
    "initial_physical_ramp_queue_R": total_initial_R,
    "initial_external_spillback_queue_B": total_initial_B,
    "total_ramp_arrivals": total_arrivals,
    "total_actual_release": total_actual_release,
    "final_physical_ramp_queue_R": total_final_R,
    "final_external_spillback_queue_B": total_final_B,
    "ramp_mass_residual": ramp_mass_residual,
}

total_ramp_demand_to_account = (
    total_initial_R
    + total_initial_B
    + total_arrivals
)

benchmark_service_metrics["total_ramp_demand_to_account"] = (
    total_ramp_demand_to_account
)

if total_ramp_demand_to_account > 0:
    benchmark_service_metrics["served_fraction"] = (
        benchmark_service_metrics["total_actual_release"]
        / total_ramp_demand_to_account
    )
else:
    benchmark_service_metrics["served_fraction"] = 1.0

benchmark_service_metrics_df = pd.DataFrame([
    {
        "metric": metric,
        "value": value,
    }
    for metric, value in benchmark_service_metrics.items()
])

print("Benchmark Ramp Service / Conservation Metrics")
display(benchmark_service_metrics_df.round(6))

Benchmark Ramp Service / Conservation Metrics


,metric,value
0,initial_physical_ramp_queue_R,0.000000
1,initial_external_spillback_queue_B,0.000000
2,total_ramp_arrivals,4387.200000
3,total_actual_release,3656.000000
4,final_physical_ramp_queue_R,146.257223
5,final_external_spillback_queue_B,584.942777
6,ramp_mass_residual,-0.000000
7,total_ramp_demand_to_account,4387.200000
8,served_fraction,0.833333


In [90]:
# EXPORT SOURCE-OF-TRUTH BENCHMARK INPUTS FOR ADMM-MPC

import pickle
import pandas as pd

# 0. Explicit settings used by both benchmark and ADMM
use_clipped_mainline_delay = True

if external_queue_0 is None:
    external_queue_0_export = {
        ramp: 0.0
        for ramp in ramp_ids
    }
else:
    external_queue_0_export = external_queue_0.copy()


# 1. Official benchmark re-run
official_benchmark_history = simulate_state_based_benchmark_480_steps(
    num_steps=num_steps,
    mainline_initial_state=mainline_initial_state,
    ramp_queue_0=ramp_queue_0,
    q_in_boundary_series=q_in_boundary_series,
    observed_release_series=observed_release_series,
    ramp_arrival_series=ramp_arrival_series,
    u_in_series=u_in_series,
    f_out_series=f_out_series,
    residual_inflow_series=residual_inflow_series,
    residual_outflow_series=residual_outflow_series,

    inflow_capacity=ctm_inflow_capacity_official,
    outflow_capacity=ctm_outflow_capacity_official,
    physical_capacity=physical_capacity,
    safe_threshold_capacity=safe_threshold_capacity,

    ramp_name_map=ramp_name_map,
    ramp_max_queue_named=ramp_max_queue_named,
    ramp_max_queue_by_u=ramp_max_queue_by_u,

    tt_ff_min=tt_ff_min,
    delta_t=delta_t,

    gamma=gamma,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    lambda_4=lambda_4,

    movement_factor_by_cell=movement_factor_by_cell_official,
    wave_speed_ratio_by_cell=wave_speed_ratio_by_cell_official,
    exit_split_by_cell=exit_split_by_cell,
    external_queue_0=external_queue_0_export,
    use_clipped_mainline_delay=use_clipped_mainline_delay
)


# 2. Official benchmark totals
official_totals = {
    "mainline_delay": sum(official_benchmark_history["mainline_delay"]),
    "mainline_delay_raw": sum(official_benchmark_history["mainline_delay_raw"]),
    "mainline_delay_clipped": sum(official_benchmark_history["mainline_delay_clipped"]),
    "upstream_boundary_delay": sum(official_benchmark_history["upstream_boundary_delay"]),

    "local_delay": sum(official_benchmark_history["local_delay"]),
    "fairness_penalty": sum(official_benchmark_history["fairness_penalty"]),
    "doorway_penalty": sum(official_benchmark_history["doorway_penalty"]),
    "safe_penalty": sum(official_benchmark_history["safe_penalty"]),
    "physical_penalty": sum(official_benchmark_history["physical_penalty"]),
    "spillback_penalty": sum(official_benchmark_history["spillback_penalty"]),
}

official_totals["capacity_penalty"] = (
    official_totals["doorway_penalty"]
    + official_totals["safe_penalty"]
    + official_totals["physical_penalty"]
    + official_totals["spillback_penalty"]
)

official_totals["raw_objective"] = (
    official_totals["mainline_delay"]
    + official_totals["local_delay"]
    + official_totals["fairness_penalty"]
    + official_totals["capacity_penalty"]
)

official_totals_df = pd.DataFrame([
    {
        "term": term,
        "value": value
    }
    for term, value in official_totals.items()
])

print("Official benchmark totals")
display(official_totals_df.round(6))


# 3. Ramp service / conservation metrics
R_final = official_benchmark_history["R_final"]
B_final = official_benchmark_history["B_final"]

official_service_metrics = {
    "initial_physical_ramp_queue_R": sum(
        float(ramp_queue_0[ramp])
        for ramp in ramp_ids
    ),

    "initial_external_spillback_queue_B": sum(
        float(external_queue_0_export[ramp])
        for ramp in ramp_ids
    ),

    "total_ramp_arrivals": sum(
        float(ramp_arrival_series[ramp][step])
        for ramp in ramp_ids
        for step in range(num_steps)
    ),

    "total_actual_release": sum(
        float(official_benchmark_history["actual_release"][step][ramp])
        for ramp in ramp_ids
        for step in range(num_steps)
    ),

    "final_physical_ramp_queue_R": sum(
        float(R_final[ramp])
        for ramp in ramp_ids
    ),

    "final_external_spillback_queue_B": sum(
        float(B_final[ramp])
        for ramp in ramp_ids
    ),

    "final_upstream_boundary_queue": float(
        official_benchmark_history["upstream_boundary_queue_final"]
    ),
}

official_service_metrics["ramp_mass_residual"] = (
    official_service_metrics["initial_physical_ramp_queue_R"]
    + official_service_metrics["initial_external_spillback_queue_B"]
    + official_service_metrics["total_ramp_arrivals"]
    - official_service_metrics["total_actual_release"]
    - official_service_metrics["final_physical_ramp_queue_R"]
    - official_service_metrics["final_external_spillback_queue_B"]
)

official_total_ramp_demand_to_account = (
    official_service_metrics["initial_physical_ramp_queue_R"]
    + official_service_metrics["initial_external_spillback_queue_B"]
    + official_service_metrics["total_ramp_arrivals"]
)

official_service_metrics["total_ramp_demand_to_account"] = (
    official_total_ramp_demand_to_account
)

if official_total_ramp_demand_to_account > 0:
    official_service_metrics["served_fraction"] = (
        official_service_metrics["total_actual_release"]
        / official_total_ramp_demand_to_account
    )
else:
    official_service_metrics["served_fraction"] = 1.0

official_service_metrics_df = pd.DataFrame([
    {
        "metric": metric,
        "value": value
    }
    for metric, value in official_service_metrics.items()
])

print("Official benchmark ramp service / conservation metrics")
display(official_service_metrics_df.round(6))


# 4. Normalization denominators
normalization_scales = {
    "mainline_delay_veh_min": 10000.0,
    "local_delay_veh_min": 1000.0,
    "fairness_penalty_units": 100.0,
    "doorway_penalty_units": 1000.0,
    "safe_penalty_units": 1000.0,
    "physical_penalty_units": 1000.0,
    "spillback_penalty_units": 1000.0,
}

benchmark_denominators = {
    "D_main_base": normalization_scales["mainline_delay_veh_min"],
    "D_local_base": normalization_scales["local_delay_veh_min"],
    "L_fair_base": normalization_scales["fairness_penalty_units"],
    "P_door_base": normalization_scales["doorway_penalty_units"],
    "P_safe_base": normalization_scales["safe_penalty_units"],
    "P_phys_base": normalization_scales["physical_penalty_units"],
    "P_spill_base": normalization_scales["spillback_penalty_units"],
}

for key, value in benchmark_denominators.items():
    if value <= 0:
        raise ValueError(f"{key} must be positive.")

benchmark_denominators_df = pd.DataFrame([
    {
        "denominator": denominator,
        "value": value
    }
    for denominator, value in benchmark_denominators.items()
])

print("Benchmark normalization denominators")
display(benchmark_denominators_df.round(6))


# 5. Build shared source-of-truth dictionary
shared_benchmark_inputs = {
    # core simulation settings
    "num_steps": num_steps,
    "delta_t": delta_t,
    "arrival_multiplier": arrival_multiplier,
    "use_clipped_mainline_delay": use_clipped_mainline_delay,

    "benchmark_profile_type": benchmark_profile_type,
    "benchmark_date": benchmark_date,
    "benchmark_source_file": benchmark_source_file,
    "benchmark_start_hour": benchmark_start_hour,
    "benchmark_end_hour": benchmark_end_hour,
    "benchmark_start_time": benchmark_start_time,
    "benchmark_end_time": benchmark_end_time,
    "benchmark_window_label": benchmark_window_label,

    # objective weights / penalty coefficients
    "gamma": gamma,
    "lambda_1": lambda_1,
    "lambda_2": lambda_2,
    "lambda_3": lambda_3,
    "lambda_4": lambda_4,

    # CTM initial states
    "mainline_initial_state": mainline_initial_state,
    "ramp_queue_0": ramp_queue_0,
    "external_queue_0": external_queue_0_export,

    # benchmark time series
    "q_in_boundary_series": q_in_boundary_series,
    "observed_release_series": observed_release_series,
    "ramp_arrival_series": ramp_arrival_series,
    "observed_offramp_series": observed_offramp_series,
    "single_day_benchmark_profile": single_day_benchmark_profile,

    "u_in_series": u_in_series,
    "f_out_series": f_out_series,
    "residual_inflow_series": residual_inflow_series,
    "residual_outflow_series": residual_outflow_series,

    "benchmark_balance_audit_df": benchmark_balance_audit_df,
    "free_flow_balance_audit_df": free_flow_balance_audit_df,
    "exit_split_by_cell": exit_split_by_cell,
    "exit_split_df": exit_split_df,
    "undetected_entry_vph_by_cell": undetected_entry_vph_by_cell,
    "undetected_entry_per_15sec_by_cell": undetected_entry_per_15sec_by_cell,
    "detected_offramp_cell_map": detected_offramp_cell_map,

    "generic_ramp_cell_map": generic_ramp_cell_map,
    "merge_ramp_id": merge_ramp_id,
    "merge_ramp_cell": merge_ramp_cell,

    "CAP9_VPH": CAP9_VPH,
    "BETA_SCALE": BETA_SCALE,
    "ENTRY_SCALE": ENTRY_SCALE,
    "MERGE_PRIORITY": MERGE_PRIORITY,

    "raw_exit_split_by_cell": raw_exit_split_by_cell,
    "raw_undetected_entry_vph_by_cell": raw_undetected_entry_vph_by_cell,


    # normalization
    "normalization_scales": normalization_scales,
    "benchmark_denominators": benchmark_denominators,

    # CTM parameters
    "backward_wave_speed_mph": backward_wave_speed_mph,
    "inflow_capacity": ctm_inflow_capacity_official,
    "outflow_capacity": ctm_outflow_capacity_official,
    "doorway_capacity_legacy": ctm_doorway_capacity_official,
    "physical_capacity": physical_capacity,
    "safe_threshold_capacity": safe_threshold_capacity,
    "movement_factor_by_cell": movement_factor_by_cell_official,
    "wave_speed_ratio_by_cell": wave_speed_ratio_by_cell_official,

    # ramp metadata
    "ramp_ids": ramp_ids,
    "ramp_cell_map": ramp_cell_map,
    "ramp_name_map": ramp_name_map,
    "ramp_max_queue_by_u": ramp_max_queue_by_u,
    "ramp_max_queue_named": ramp_max_queue_named,

    # free-flow travel time
    "tt_ff_min": tt_ff_min,

    # official benchmark results
    "official_benchmark_history": official_benchmark_history,
    "official_totals": official_totals,
    "official_service_metrics": official_service_metrics,
}


# 6. Save pickle file
with open("shared_benchmark_inputs.pkl", "wb") as fh:
    pickle.dump(shared_benchmark_inputs, fh)

print("Saved shared_benchmark_inputs.pkl")


# 7. Reload and verify export consistency
with open("shared_benchmark_inputs.pkl", "rb") as fh:
    shared_check = pickle.load(fh)

max_total_error = max(
    abs(
        float(shared_check["official_totals"][key])
        - float(official_totals[key])
    )
    for key in official_totals
)

max_denominator_error = max(
    abs(
        float(shared_check["benchmark_denominators"][key])
        - float(benchmark_denominators[key])
    )
    for key in benchmark_denominators
)

max_service_error = max(
    abs(
        float(shared_check["official_service_metrics"][key])
        - float(official_service_metrics[key])
    )
    for key in official_service_metrics
)

print("Max official total export mismatch:", max_total_error)
print("Max denominator export mismatch:", max_denominator_error)
print("Max service metric export mismatch:", max_service_error)

if (
    max_total_error < 1e-9
    and max_denominator_error < 1e-9
    and max_service_error < 1e-9
):
    print("PASS: shared_benchmark_inputs.pkl matches current official benchmark.")
else:
    print("FAIL: shared_benchmark_inputs.pkl does not match current benchmark values.")

Official benchmark totals


,term,value
0,mainline_delay,2.579903e+04
1,mainline_delay_raw,2.559841e+04
2,mainline_delay_clipped,2.579903e+04
3,upstream_boundary_delay,0.000000e+00
4,local_delay,4.465700e+04
5,fairness_penalty,1.436237e+02
6,doorway_penalty,1.039050e-01
7,safe_penalty,0.000000e+00
8,physical_penalty,0.000000e+00
9,spillback_penalty,7.849384e+06


Official benchmark ramp service / conservation metrics


,metric,value
0,initial_physical_ramp_queue_R,0.000000
1,initial_external_spillback_queue_B,0.000000
2,total_ramp_arrivals,4387.200000
3,total_actual_release,3656.000000
4,final_physical_ramp_queue_R,146.257223
5,final_external_spillback_queue_B,584.942777
6,final_upstream_boundary_queue,0.000000
7,ramp_mass_residual,-0.000000
8,total_ramp_demand_to_account,4387.200000
9,served_fraction,0.833333


Benchmark normalization denominators


,denominator,value
0,D_main_base,10000.0
1,D_local_base,1000.0
2,L_fair_base,100.0
3,P_door_base,1000.0
4,P_safe_base,1000.0
5,P_phys_base,1000.0
6,P_spill_base,1000.0


Saved shared_benchmark_inputs.pkl
Max official total export mismatch: 0.0
Max denominator export mismatch: 0.0
Max service metric export mismatch: 0.0
PASS: shared_benchmark_inputs.pkl matches current official benchmark.


In [91]:
# Stored total mainline delay vs manual total mainline delay
manual_total_mainline_delay = 0.0
step_delay_rows = []

for t in range(num_steps):

    # x_now is the state before step t
    if t == 0:
        x_now_t = mainline_initial_state
    else:
        x_now_t = state_based_benchmark_history["x"][t - 1]

    # x_next is the state after step t
    x_next_t = state_based_benchmark_history["x"][t]

    q_out_t = state_based_benchmark_history["q_out"][t]
    actual_f_out_t = state_based_benchmark_history["actual_f_out"][t]

    step_delay = 0.0

    for cell in cell_ids:
        inventory_vehicle_min = (
            (float(x_now_t[cell]) + float(x_next_t[cell])) / 2.0
        ) * delta_t

        free_flow_vehicle_min = (
            float(q_out_t[cell]) + float(actual_f_out_t[cell])
        ) * float(tt_ff_min[cell])

        raw_delay = inventory_vehicle_min - free_flow_vehicle_min
        clipped_delay = max(0.0, raw_delay)

        step_delay += clipped_delay

    manual_total_mainline_delay += step_delay

    step_delay_rows.append({
        "step": t + 1,
        "manual_mainline_delay_vehicle_min": step_delay,
        "stored_mainline_delay_vehicle_min": state_based_benchmark_history["mainline_delay"][t],
        "difference": step_delay - state_based_benchmark_history["mainline_delay"][t],
    })

manual_delay_df = pd.DataFrame(step_delay_rows)

print("Manual total mainline delay:", round(manual_total_mainline_delay, 3))
print("Stored total mainline delay:", round(sum(state_based_benchmark_history["mainline_delay"]), 3))

display(manual_delay_df.head())
display(manual_delay_df.tail())

print("Max absolute step difference:",
      manual_delay_df["difference"].abs().max())

Manual total mainline delay: 25799.028
Stored total mainline delay: 25799.028


,step,manual_mainline_delay_vehicle_min,stored_mainline_delay_vehicle_min,difference
0,1,30.226932,30.226932,0.0
1,2,28.741551,28.741551,0.0
2,3,30.468083,30.468083,0.0
3,4,31.905998,31.905998,0.0
4,5,33.140637,33.140637,0.0


,step,manual_mainline_delay_vehicle_min,stored_mainline_delay_vehicle_min,difference
475,476,50.093997,50.093997,0.0
476,477,49.653499,49.653499,0.0
477,478,49.199014,49.199014,0.0
478,479,48.732312,48.732312,0.0
479,480,48.255096,48.255096,0.0


Max absolute step difference: 0.0


In [92]:
print("arrival_multiplier =", arrival_multiplier)

print(
    "total observed release =",
    sum(
        observed_release_series[ramp][step]
        for ramp in ramp_ids
        for step in range(num_steps)
    )
)

print(
    "total ramp arrivals =",
    sum(
        ramp_arrival_series[ramp][step]
        for ramp in ramp_ids
        for step in range(num_steps)
    )
)

arrival_multiplier = 1.2
total observed release = 3656.0
total ramp arrivals = 4387.2


In [93]:
# RAMP MASS-CONSERVATION CHECK FOR CURRENT SIMULATION

history = state_based_benchmark_history
num_history_steps = len(history["actual_release"])

# Initial physical ramp queue
total_initial_R = sum(
    float(ramp_queue_0[ramp])
    for ramp in ramp_ids
)

# Initial external spillback queue
if "external_queue_0_run" in globals():
    initial_B_source = external_queue_0_run
elif "external_queue_0" in globals() and external_queue_0 is not None:
    initial_B_source = external_queue_0
else:
    initial_B_source = {
        ramp: 0.0
        for ramp in ramp_ids
    }

total_initial_B = sum(
    float(initial_B_source[ramp])
    for ramp in ramp_ids
)

# Use the arrivals that were actually stored during simulation
total_arrival = sum(
    float(history["ramp_arrival"][step][ramp])
    for step in range(num_history_steps)
    for ramp in ramp_ids
)

# Use actual releases stored during simulation
total_release = sum(
    float(history["actual_release"][step][ramp])
    for step in range(num_history_steps)
    for ramp in ramp_ids
)

# Final queues
final_physical_queue = sum(
    float(history["R_final"][ramp])
    for ramp in ramp_ids
)

final_external_queue = sum(
    float(history["B_final"][ramp])
    for ramp in ramp_ids
)

mass_residual = (
    total_initial_R
    + total_initial_B
    + total_arrival
    - total_release
    - final_physical_queue
    - final_external_queue
)

print("total initial physical queue =", total_initial_R)
print("total initial external queue =", total_initial_B)
print("total arrival =", total_arrival)
print("total release =", total_release)
print("final physical queue =", final_physical_queue)
print("final external queue =", final_external_queue)
print("mass residual =", mass_residual)

if abs(mass_residual) < 1e-6:
    print("PASS: ramp mass is conserved.")
else:
    print("FAIL: ramp mass is not conserved.")

total initial physical queue = 0.0
total initial external queue = 0.0
total arrival = 4387.2
total release = 3656.0
final physical queue = 146.257222528
final external queue = 584.9427774720012
mass residual = -1.4779288903810084e-12
PASS: ramp mass is conserved.


In [94]:
ramp_mass_rows = []

for ramp in ramp_ids:
    initial_R = float(ramp_queue_0[ramp])
    initial_B = float(initial_B_source[ramp])

    arrival = sum(
        float(history["ramp_arrival"][step][ramp])
        for step in range(num_history_steps)
    )

    release = sum(
        float(history["actual_release"][step][ramp])
        for step in range(num_history_steps)
    )

    final_R = float(history["R_final"][ramp])
    final_B = float(history["B_final"][ramp])

    residual = (
        initial_R
        + initial_B
        + arrival
        - release
        - final_R
        - final_B
    )

    ramp_mass_rows.append({
        "ramp": ramp,
        "initial_R": initial_R,
        "initial_B": initial_B,
        "arrival": arrival,
        "actual_release": release,
        "final_R": final_R,
        "final_B": final_B,
        "mass_residual": residual,
    })

ramp_mass_df = pd.DataFrame(ramp_mass_rows)

display(ramp_mass_df.round(6))

assert ramp_mass_df["mass_residual"].abs().max() < 1e-6
print("PASS: per-ramp mass is conserved.")

,ramp,initial_R,initial_B,arrival,actual_release,final_R,final_B,mass_residual
0,u_4th,0.0,0.0,979.2,816.0,20.259843,142.940157,-0.0
1,u_price,0.0,0.0,1964.4,1637.0,43.959319,283.440681,-0.0
2,u_mattie,0.0,0.0,693.6,578.0,24.434384,91.165616,0.0
3,u_avila,0.0,0.0,750.0,625.0,57.603676,67.396324,0.0


PASS: per-ramp mass is conserved.
